# Modified Spades — Strong Play V3 · Runtime Safety V2

Runtime changes preserve the original game rules,13 opponents, network, teacher/PPO
objective, RUN_NAME,three-hour segments and90-minute checkpoints. This notebook
uses validated current allocator pool capacity, minimum1.45×estimate+256MiB
admissions, exact-configuration/runtime identity checks, and host validation of
native checkpoints before device transfer. Compiler errors stop immediately.

Every accepted PPO rollout records integer planned/attempted/accepted/skipped/
rejected optimizer work in checkpoint history and metrics.jsonl. Planned work is
derived from the saved PPO cadence; skipped slots are never reported as executed.
An isolated valid KL stop remains allowed. Three consecutive same-cadence KL stops
with aggregate acceptance below25% commit the valid state, force a unique retained
health checkpoint and stop before evaluation. Resume preserves this journal and
stop reason; the trainer never automatically retries or changes learning settings.

## Resume and recovery

For direct native recovery, set RESUME_FILE in Setup to the exact ZIP path, run
Setup and follow any explicit restart instruction, then run definitions and
configuration. Version checks and the saved config are authoritative. An exact
Strong Play V3 checkpoint keeps every parameter, Adam state, RNG, game/league state,
teacher progress and lifetime counter; it is never silently resized or reset.
New saves continue in `strong_play_v3_runtime_v1` for checkpoint continuity.
Original `strong_play_v3`, `score_feedback_v2` and original run folders are read-only
recovery inputs. Explicit older-trainer migration retains its documented resets.

After an optimizer health stop, use the printed `optimizer_health_stop_*.zip` path;
these unique saves use no retention cleanup. A handled failure prints the last
accepted native checkpoint. If Drive saving fails, the local recovery path is
`/tmp/modified_spades_emergency/emergency_*.zip` in Colab's temporary directory;
download/export it before runtime disposal. Native checkpoint history contains the
per-update journal and failure reason even if metrics.jsonl could not be written.
Inspect and fix the recorded cause offline before retrying; do not rerun an
unchanged failed cell. A checkpoint recording a health stop remains stopped on
exact resume until an explicit matching recovery is recorded. After reviewing and
fixing the cause, select its latest accepted native ZIP, then set this runtime-only
acknowledgement before the training cell (replace the reason with the actual fix):

```python
stop = next(row for row in reversed(training_history) if row.get('kind') == 'optimizer_health_stop')
RUN_CFG['optimizer_health_acknowledgement'] = {
    'stop_update': stop['update'], 'stop_id': stop['stop_id'],
    'config_signature': _optimizer_config_signature(TRAIN_CFG),
    'reason': 'Describe the reviewed cause and fix here',
}
```

The acknowledgement resets only the rolling health window; it preserves every
model/optimizer/RNG/game/counter and saved learning setting. A stale acknowledgement
cannot approve another stop. Healthy completed segments can be continued normally.

## Evidence and limits

The original recovered38-cell source SHA-256 is
`3236d2c95d4d1a95bf31a5cb7299853be6871783691829ee775fbb38366d1389`.
Physical T4 diagnostic V3 passed the corrected pool accounting at3584×128 with
the same calibrated guard, two finite legal PPO updates and zero rejections.
It executed26 accepted optimizer minibatches out of48 planned:2/24 with KL stop,
then24/24 without KL stop. This demonstrates physical pool admission for that
diagnostic, not the final production loader/health integration or playing strength.
Its raw JSON SHA-256 is
`e6762ff49a727805b76f31eaa0f63fda2cf6fa3d4ae33f8ecc99ca620d9efb1c`.

The sustained37-rollout/4096-game run is not promoted by this notebook; results
were pending at preparation. Full production T4 integration remains unproven.
Eager initialization/completion, transfer transients and fragmentation are not
separate measured peak bounds. CPU checks and source-cell hashes accompany this
artifact; CPU success does not establish GPU safety or improved strength.


## 1 · Fast setup, Drive, and revision selection

Use a fresh T4 Colab session. Keep `RUN_NAME` unchanged to locate your existing progress; `RESUME_FILE` optionally selects an exact ZIP. The default search order is `strong_play_v3_runtime_v1`, `strong_play_v3`, `score_feedback_v2`, then the original run folder. Saves use creation chronology, including a continuation from a lower update.

Healthy installed GPU packages are reused without pip. Saved checkpoints select their saved JAX/Flax/Optax versions. The earlier constrained Flax repair remains: it does not incidentally replace a working JAX/CUDA stack.

The allocator ceiling becomes 96% before importing JAX, while preallocation remains disabled. This permits larger useful workloads; it does not reserve 96% upfront. An already-imported JAX runtime with different allocator settings requires a session restart. Installation progress and restart instructions remain explicit.

In [ ]:
# Fresh Colab runtime: Runtime > Change runtime type > T4 GPU.
# Healthy Colab packages are reused. Broken Flax/Optax is repaired without replacing GPU libraries.
# Set SETUP_MODE = 'pinned' only to repair a broken native JAX installation.
import os, sys, json, re, time, zipfile, tempfile, subprocess, importlib, hashlib
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone

RUN_NAME = 'modified_spades_run_001'  # Keep this name to resume your original run.
LEGACY_CHECKPOINT_DIR = f'/content/drive/MyDrive/ModifiedSpadesJAX/{RUN_NAME}'
RECOVERY_CHECKPOINT_DIR = str(Path(LEGACY_CHECKPOINT_DIR) / 'score_feedback_v2')
PRIOR_V3_CHECKPOINT_DIR = str(Path(LEGACY_CHECKPOINT_DIR) / 'strong_play_v3')
CHECKPOINT_DIR = str(Path(LEGACY_CHECKPOINT_DIR) / 'strong_play_v3_runtime_v1')
# A legacy policy is recovered once into this separate folder; later sessions resume it.
RESUME = True
RESUME_FILE = ''  # Optional exact .zip path; otherwise inspect the newest regular save.
REQUIRE_GPU = True  # False is for small CPU smoke tests only.
SETUP_MODE = 'auto'  # 'auto': reuse Colab; 'pinned': repair using the original stack.
PINNED_STACK = {'jax': '0.4.35', 'flax': '0.10.2', 'optax': '0.2.4'}
_SETUP_PACKAGES = ('jax', 'jaxlib', 'flax', 'optax', 'numpy', 'pandas', 'matplotlib')
_loaded_versions = {name: getattr(sys.modules[name], '__version__', None)
                    for name in _SETUP_PACKAGES if name in sys.modules}
if 'jax' in sys.modules and (os.environ.get('XLA_PYTHON_CLIENT_MEM_FRACTION') != '0.96'
        or os.environ.get('XLA_PYTHON_CLIENT_PREALLOCATE', '').lower() != 'false'):
    raise RuntimeError('This notebook raises the GPU allocator ceiling for the 14 GiB workload. '
        'Restart the Colab session and run this notebook from Setup; save the old run first.')
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.96'
print('MODIFIED SPADES — STRONG PLAY V3 | measured 14 GiB workload target', flush=True)
print('New checkpoints:', CHECKPOINT_DIR, flush=True)


def _mount_drive():
    from google.colab import drive
    drive.mount('/content/drive')


def _installed_versions():
    versions = {}
    for name in _SETUP_PACKAGES:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            pass
    return versions


def _accelerator_versions():
    """Installed accelerator distributions, including CUDA plugin families."""
    versions = {}
    for distribution in importlib.metadata.distributions():
        name = re.sub(r'[-_.]+', '-', distribution.metadata.get('Name', '')).lower()
        if (name.startswith(('nvidia-', 'cuda-')) or
                (name.startswith('jax') and name.endswith(('-plugin', '-pjrt')))):
            versions[name] = distribution.version
    return versions


def _preserve_working_stack(installed):
    constraints = _accelerator_versions()
    constraints.update({name: installed[name] for name in ('jax', 'jaxlib', 'numpy')
                        if name in installed})
    return constraints


def _training_repair_requirements(installed):
    """Known tested fixes where available; otherwise resolve within frozen JAX."""
    match = re.match(r'^(\d+)\.(\d+)', installed.get('jax', ''))
    series = tuple(map(int, match.groups())) if match else None
    release = re.match(r'^0\.11\.(\d+)', installed.get('jax', ''))
    if series == (0, 10) or (release and int(release.group(1)) == 0):
        # 0.12.8 fixes the removed API and supports JAX 0.10/0.11.0.
        # Flax 0.12.9+ requires JAX >= 0.11.1, so cannot repair 0.11.0 in place.
        return ['flax==0.12.8', 'optax==0.2.8']
    if series == (0, 11):
        # Older Flax calls jax.core.get_opaque_trace_state, removed by JAX 0.11.
        return ['flax==0.12.10', 'optax==0.2.8']
    if installed.get('jax') == PINNED_STACK['jax']:
        return [f'{name}=={PINNED_STACK[name]}' for name in ('flax', 'optax')]
    return ['flax', 'optax']


def _require_physical_gpu():
    if not REQUIRE_GPU:
        return
    try:
        result = subprocess.run(['nvidia-smi', '-L'], text=True, capture_output=True,
                                timeout=15, check=False)
    except (OSError, subprocess.TimeoutExpired) as exc:
        raise RuntimeError('No NVIDIA GPU runtime is available. Select Runtime > Change '
                           'runtime type > T4 GPU, then rerun setup. No CUDA packages '
                           'have been downloaded.') from exc
    if result.returncode or 'GPU ' not in result.stdout:
        raise RuntimeError('No NVIDIA GPU runtime is available. Select Runtime > Change '
                           'runtime type > T4 GPU, then rerun setup. No CUDA packages '
                           'have been downloaded.\n' + result.stderr[-1500:])
    print(result.stdout.strip(), flush=True)


def _setup_checkpoint_creation_key(path):
    """Order saves by creation time, including explicit lower-update resumes."""
    path = Path(path)
    try:
        stamp = datetime.strptime(path.stem.rsplit('_', 1)[-1], '%Y%m%dT%H%M%S%f')
        created = stamp.replace(tzinfo=timezone.utc).timestamp()
    except ValueError:
        # Imported/older names may lack our UTC suffix; use their file time.
        created = path.stat().st_mtime_ns / 1e9
    return created, path.name


def _validate_setup_archive(archive, manifest):
    """Check recoverable bytes before choosing or installing a saved stack."""
    checksum = manifest.get('sha256')
    if not isinstance(checksum, str) or not re.fullmatch(r'[0-9a-f]{64}', checksum):
        raise ValueError('Checkpoint checksum must be a SHA256 hexadecimal string.')
    state_info = archive.getinfo('state.npz')
    history_info = archive.getinfo('history.json')
    if state_info.file_size == 0:
        raise ValueError('Checkpoint state payload is empty.')
    history = json.loads(archive.read(history_info))
    if not isinstance(history, list) or any(not isinstance(row, dict) for row in history):
        raise ValueError('Checkpoint history must be a list of JSON objects.')
    try:
        json.dumps(history, allow_nan=False)
    except (TypeError, ValueError) as exc:
        raise ValueError('Checkpoint history must contain finite JSON values.') from exc
    # This extra sequential Drive read avoids installing a different stack for
    # a damaged save. Streaming bounds memory; full state validation follows load.
    digest = hashlib.sha256()
    with archive.open(state_info) as payload:
        for chunk in iter(lambda: payload.read(1024 * 1024), b''):
            digest.update(chunk)
    if digest.hexdigest() != checksum:
        raise ValueError('Checkpoint is corrupt: array checksum mismatch.')


def _read_setup_manifest():
    """Select a saved stack only after archive checksum/history validation."""
    if RESUME_FILE:
        paths = [Path(RESUME_FILE)]
    elif RESUME:
        paths = sorted(Path(CHECKPOINT_DIR).glob('checkpoint_*.zip'),
                       key=_setup_checkpoint_creation_key, reverse=True)
        for previous_folder in (PRIOR_V3_CHECKPOINT_DIR, RECOVERY_CHECKPOINT_DIR, LEGACY_CHECKPOINT_DIR):
            if not paths:
                # Select the latest previous revision's library stack before import.
                paths = sorted(Path(previous_folder).glob('checkpoint_*.zip'),
                               key=_setup_checkpoint_creation_key, reverse=True)
    else:
        paths = []
    for path in paths:
        try:
            with zipfile.ZipFile(path) as archive:
                info = archive.getinfo('manifest.json')
                if info.file_size > 1_000_000:
                    raise ValueError('Checkpoint manifest is unexpectedly large.')
                manifest = json.loads(archive.read(info))
                if not isinstance(manifest, dict):
                    raise ValueError('Checkpoint manifest must be a JSON object.')
                print(f'Validating saved checkpoint before stack selection: {path.name}', flush=True)
                _validate_setup_archive(archive, manifest)
        except (OSError, ValueError, KeyError, zipfile.BadZipFile) as exc:
            if RESUME_FILE:
                raise RuntimeError(f'Cannot validate requested checkpoint {path}: {exc}') from exc
            print(f'Skipping unreadable checkpoint {path.name}: {exc}', flush=True)
            continue
        if (manifest.get('format') != 1 or
                manifest.get('ruleset') != 'modified-spades-500-blind-swap-team-failure-v1'):
            raise ValueError(f'{path.name}: checkpoint format or game rules do not match. '
                             'Choose the correct run folder; do not overwrite it.')
        versions = manifest.get('versions', {})
        if not isinstance(versions, dict) or any(
                not isinstance(versions.get(name), str) or
                not re.fullmatch(r'[0-9]+(?:\.[0-9]+)+(?:[a-zA-Z0-9.+_-]*)', versions[name])
                for name in PINNED_STACK):
            message = (f'{path.name}: valid saved JAX/Flax/Optax versions are required '
                       'for an exact resume.')
            if RESUME_FILE:
                raise ValueError(message)
            print('Skipping unreadable checkpoint manifest: ' + message, flush=True)
            continue
        return str(path), {name: versions[name] for name in PINNED_STACK}
    if paths:
        raise RuntimeError('No validated checkpoint was found. Restore a backup '
                           'or choose another run folder.')
    return None, None


def _probe_stack(include_training=True, require_gpu=True):
    """Probe in a child process: pip never replaces JAX imported by this notebook."""
    probe = r'''
import json, sys
import numpy as np
import jax
import jax.numpy as jnp
jax.config.update('jax_enable_x64', False)
devices = jax.devices()
if REQUIRE_GPU and not any(d.platform == 'gpu' for d in devices):
    raise RuntimeError('JAX cannot use the allocated GPU: ' + repr(devices))
x = jax.jit(lambda a: jnp.sin(a) + 1.0)(jnp.ones((8,), dtype=jnp.float32))
x.block_until_ready()
if not np.isfinite(np.asarray(x)).all():
    raise RuntimeError('JAX GPU arithmetic returned non-finite results.')
versions = {'jax': jax.__version__}
if INCLUDE_TRAINING:
    import flax
    from flax import linen as nn
    import optax
    model = nn.Dense(4)
    inputs = jnp.ones((2, 8), dtype=jnp.float32)
    params = model.init(jax.random.PRNGKey(0), inputs)['params']
    optimizer = optax.adam(1e-3)
    state = optimizer.init(params)
    @jax.jit
    def update(p, s):
        loss, grads = jax.value_and_grad(
            lambda q: jnp.mean(model.apply({'params': q}, inputs) ** 2))(p)
        updates, s = optimizer.update(grads, s, p)
        return optax.apply_updates(p, updates), s, loss
    params, state, loss = update(params, state)
    loss.block_until_ready()
    if not all(np.isfinite(np.asarray(a)).all()
               for a in jax.tree_util.tree_leaves((params, state, loss))):
        raise RuntimeError('Flax/Optax training probe returned non-finite values.')
    versions.update(flax=flax.__version__, optax=optax.__version__)
print('__SPADES_SETUP__' + json.dumps({'versions': versions,
                                      'devices': [str(d) for d in devices]}))
'''
    source = (f'REQUIRE_GPU = {bool(require_gpu)!r}\n'
              f'INCLUDE_TRAINING = {bool(include_training)!r}\n' + probe)
    started = time.monotonic()
    process = subprocess.Popen([sys.executable, '-u', '-c', source],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, env=os.environ.copy())
    while True:
        try:
            output, _ = process.communicate(timeout=15)
            break
        except subprocess.TimeoutExpired:
            elapsed = time.monotonic() - started
            print(f'Checking installed GPU stack: {elapsed:.0f}s elapsed...', flush=True)
            if elapsed >= 120:
                process.kill()
                output, _ = process.communicate()
                return False, {}, 'GPU stack probe timed out after 120s.\n' + output[-6000:]
    if process.returncode:
        return False, {}, output[-6000:]
    for line in reversed(output.splitlines()):
        if line.startswith('__SPADES_SETUP__'):
            return True, json.loads(line[len('__SPADES_SETUP__'):]), output[-6000:]
    return False, {}, 'Dependency probe returned no result.\n' + output[-6000:]


def _install_packages(requirements, constraints=None, upgrade=False):
    if not requirements:
        return
    started = time.monotonic()
    print('Installing only: ' + ', '.join(requirements), flush=True)
    # pip displays dependency resolution and download progress; no silent -q installs.
    with tempfile.TemporaryDirectory(prefix='spades_setup_') as temporary:
        command = [sys.executable, '-m', 'pip', 'install',
                   '--disable-pip-version-check', '--progress-bar', 'on']
        if upgrade:
            # Only Python add-ons are requested. Constraints freeze working GPU packages.
            command.extend(['--upgrade', '--upgrade-strategy', 'only-if-needed'])
        if constraints:
            path = Path(temporary) / 'constraints.txt'
            path.write_text(''.join(f'{name}=={version}\n'
                                    for name, version in constraints.items()))
            command.extend(['--constraint', str(path)])
        command.extend(requirements)
        try:
            subprocess.check_call(command)
        except subprocess.CalledProcessError as exc:
            raise RuntimeError('Dependency installation failed; the resolver output above '
                               'contains the cause. Start a fresh GPU runtime before retrying. '
                               'Use SETUP_MODE="pinned" only if the native stack needs repair.') from exc
    importlib.invalidate_caches()
    print(f'Package installation finished in {time.monotonic() - started:.1f}s.', flush=True)


def _run_fast_setup():
    started = time.monotonic()
    if SETUP_MODE not in ('auto', 'pinned'):
        raise ValueError('SETUP_MODE must be "auto" or "pinned".')
    _mount_drive()
    checkpoint_path, saved_stack = _read_setup_manifest()
    if saved_stack:
        print('Checkpoint requires:', saved_stack, flush=True)
    if SETUP_MODE == 'pinned' and saved_stack and saved_stack != PINNED_STACK:
        raise ValueError('This checkpoint uses a different stack. Select SETUP_MODE="auto" '
                         'to restore its saved versions exactly.')
    _require_physical_gpu()  # Fail before any package download in a CPU Colab runtime.
    installed = _installed_versions()
    model_present = all(name in installed for name in PINNED_STACK)
    healthy, probe, detail = _probe_stack(include_training=model_present,
                                         require_gpu=REQUIRE_GPU)
    # Separate a missing/broken add-on from a broken JAX/CUDA installation.
    jax_healthy = healthy
    if not healthy and model_present:
        jax_healthy, _, jax_detail = _probe_stack(include_training=False,
                                                require_gpu=REQUIRE_GPU)
    else:
        jax_detail = detail
    target = saved_stack or (PINNED_STACK if SETUP_MODE == 'pinned' else None)
    if (SETUP_MODE == 'auto' and target and not healthy and
            all(installed.get(name) == version for name, version in target.items())):
        raise RuntimeError('Installed versions already match this checkpoint, but their '
                           'training probe failed. No CUDA packages were downloaded. '
                           'Start a fresh T4 runtime, or select SETUP_MODE="pinned" if '
                           'this checkpoint uses the original stack and needs repair.\n' + detail)
    requirements = []
    constraints = {}
    addon_repair = False
    preserved = None
    if target:
        replace_jax = not jax_healthy or installed.get('jax') != target['jax']
        if replace_jax:
            print('Restoring the required JAX stack. This can download CUDA libraries; '
                  'subsequent healthy sessions skip this installation.', flush=True)
            jax_requirement = f'jax[cuda12]=={target["jax"]}' if REQUIRE_GPU else f'jax=={target["jax"]}'
            requirements.append(jax_requirement)
        else:
            # Preserve the working GPU installation, including its matching jaxlib.
            constraints.update(_preserve_working_stack(installed))
            preserved = dict(constraints)
        requirements.extend(f'{name}=={target[name]}' for name in ('flax', 'optax')
                            if installed.get(name) != target[name] or not healthy)
        constraints.update(target)
    elif jax_healthy:
        if model_present and not healthy:
            print('GPU JAX works; repairing the incompatible Flax/Optax Python packages. '
                  'Installed JAX, JAXlib, NumPy, CUDA plugins, and NVIDIA libraries stay fixed.',
                  flush=True)
            requirements.extend(_training_repair_requirements(installed))
            addon_repair = True
        elif not model_present:
            # A missing addon prevented the full training probe. Resolve a compatible
            # pair so an already-installed but incompatible companion is repaired too.
            requirements.extend(_training_repair_requirements(installed))
            addon_repair = True
        constraints = _preserve_working_stack(installed)
        preserved = dict(constraints)
    else:
        raise RuntimeError('The installed JAX stack cannot use this runtime. No CUDA packages '
                           'were downloaded. Start a fresh T4 GPU runtime, or set '
                           'SETUP_MODE="pinned" to install the original compatible stack.\n' + jax_detail)
    requirements.extend(name for name in ('numpy', 'pandas', 'matplotlib')
                        if name not in installed)
    if requirements:
        if addon_repair:
            _install_packages(requirements, constraints, upgrade=True)
        else:
            _install_packages(requirements, constraints)
    after = _installed_versions()
    if preserved is not None:
        after_protected = _accelerator_versions()
        after_protected.update(after)
        changed = {name: (version, after_protected.get(name))
                   for name, version in preserved.items() if after_protected.get(name) != version}
        if changed:
            raise RuntimeError('Protected JAX/GPU packages changed unexpectedly: ' + str(changed)
                               + '. Do not start training; use a fresh runtime.')
    changed_loaded = [name for name, version in _loaded_versions.items()
                      if version and after.get(name) != version]
    if changed_loaded:
        raise RuntimeError('Already-imported packages changed: ' + ', '.join(changed_loaded)
                           + '. Choose Runtime > Restart session, then rerun from this setup '
                           'cell. Matching installed versions will be reused.')
    if requirements or not model_present:
        healthy, probe, detail = _probe_stack(include_training=True, require_gpu=REQUIRE_GPU)
    if not healthy:
        raise RuntimeError('The installed JAX/Flax/Optax stack failed its training check. '
                           'See the captured diagnostic below.\n' + detail)
    selected = probe['versions']
    if target and any(selected.get(name) != version for name, version in target.items()):
        raise RuntimeError(f'Installed versions do not match the required stack: '
                           f'{selected} versus {target}. Restart the runtime and rerun setup.')
    if not requirements:
        print('Reused the working installed stack; pip was not run.', flush=True)
    print('Selected versions:', selected, '| Devices:', probe['devices'], flush=True)
    print(f'Setup completed in {time.monotonic() - started:.1f}s. Continue to Imports.', flush=True)
    return {'versions': selected, 'checkpoint_path': checkpoint_path}


# Run setup.
SETUP_REPORT = _run_fast_setup()
SELECTED_STACK = SETUP_REPORT['versions']
SETUP_CHECKPOINT_PATH = SETUP_REPORT['checkpoint_path']


## 2 · Imports and GPU check

The default training path uses float32 for numerical stability. A missing GPU raises an error instead of silently beginning a three-hour CPU run.

The selected stack must import successfully and expose a GPU. This cell enables JAX’s persistent compilation cache in your own Drive folder before the engine and trainer compile. JAX validates cache keys against the executable, library versions, compiler options, and hardware; a cache miss simply recompiles. Cache reuse does not alter the model or training algorithm.

In [ ]:
import os, time, json, math
import numpy as np
import jax
import jax.numpy as jnp
import flax
from flax import linen as nn
import optax

if 'SELECTED_STACK' not in globals():
    raise RuntimeError('Run the setup cell first so installed and checkpoint versions are checked.')
_actual_stack = {'jax': jax.__version__, 'flax': flax.__version__, 'optax': optax.__version__}
if _actual_stack != SELECTED_STACK:
    raise RuntimeError(f'Imported versions {_actual_stack} differ from setup {SELECTED_STACK}. '
                       'Restart the runtime and rerun from Setup.')
jax.config.update('jax_enable_x64', False)
devices = jax.devices()
print('JAX', jax.__version__, '| Flax', flax.__version__, '| Optax', optax.__version__)
print('Devices:', devices)
if REQUIRE_GPU and not any(d.platform == 'gpu' for d in devices):
    raise RuntimeError('No JAX GPU detected. Select a T4 GPU runtime, then rerun setup in a fresh runtime.')

# Cache only expensive compilations. JAX's own cache keys include source, flags,
# compiler version and device information; compatibility checks remain enabled.
# Keep this folder private: compiled artifacts must come only from your own runs.
COMPILATION_CACHE_DIR = str(Path(LEGACY_CHECKPOINT_DIR).parent / '.jax_compilation_cache')
try:
    Path(COMPILATION_CACHE_DIR).mkdir(parents=True, exist_ok=True)
    jax.config.update('jax_compilation_cache_dir', COMPILATION_CACHE_DIR)
    jax.config.update('jax_persistent_cache_min_compile_time_secs', 5.0)
    print('Persistent compilation cache:', COMPILATION_CACHE_DIR)
except (AttributeError, OSError, ValueError) as exc:
    print('Persistent compilation cache unavailable; normal compilation remains enabled:', exc)


## 3 · Exact game engine and information boundaries

Cards are `suit × 13 + rank`, with clubs/diamonds/hearts/spades and 2 through ace. Actions 0–51 select a card, 52–65 bid 0–13, 66 declines or skips, and 67 declares Double Nil. Every hand has 64 scheduled turns: four blind polls, four pass slots, four bid slots, then 52 card plays. Unused slots are forced no-ops and receive no policy or entropy loss.

The engine enforces follow suit and allows spades to lead immediately. Double Nil eligibility is frozen at the start of the hand, hands stay hidden during blind polling, and both swap selections are collected before either is delivered. Failed Nil/Double Nil invalidates the **entire** team contract. Scores, bag remainders, rotating dealer, and the exact ten-hand tie-break rule persist across hands.

The actor sees its own hand and known exchange, bids, scores, bags, table, observed voids, and complete public card-play chronology. It never receives the real opposing or partner hands. The critic has its own parameters and may use full state **during training only**. This is centralized value training with decentralized actions.

In [ ]:
# Pure JAX rules engine. Card IDs: clubs, diamonds, hearts, spades; 2 through ace.
from typing import NamedTuple

N_ACTIONS = 68
NOOP, DOUBLE_NIL = 66, 67
BLIND, PASS, BID, PLAY = 0, 1, 2, 3
GAMMA_HAND = 0.997
CARD_IDS = jnp.arange(52, dtype=jnp.int32)
CARD_SUITS, CARD_RANKS = CARD_IDS // 13, CARD_IDS % 13
TEAM_PLAYERS = jnp.array([[0, 2], [1, 3]], dtype=jnp.int32)


class State(NamedTuple):
    hands: jax.Array
    bids: jax.Array
    tricks: jax.Array
    scores: jax.Array
    bags: jax.Array
    dealer: jax.Array
    turn: jax.Array
    current_player: jax.Array
    table: jax.Array
    table_players: jax.Array
    trick_pos: jax.Array
    leader: jax.Array
    played_by: jax.Array
    played_at: jax.Array
    voids: jax.Array
    pending_pass: jax.Array
    passed: jax.Array
    received: jax.Array
    eligible: jax.Array
    tiebreak: jax.Array
    hand_number: jax.Array


def _new_hand(key, scores, bags, dealer, tiebreak, hand_number):
    # Interleave the shuffled deck clockwise, beginning left of the dealer.
    cards = jax.random.permutation(key, 52)
    owners = (dealer + 1 + jnp.arange(52, dtype=jnp.int32)) % 4
    hands = jnp.zeros((4, 52), dtype=jnp.bool_).at[owners, cards].set(True)
    first = (dealer + 1) % 4
    return State(
        hands=hands, bids=jnp.full(4, -2, jnp.int32),
        tricks=jnp.zeros(4, jnp.int32), scores=scores, bags=bags,
        dealer=dealer, turn=jnp.int32(0), current_player=first,
        table=jnp.full(4, -1, jnp.int32),
        table_players=jnp.full(4, -1, jnp.int32), trick_pos=jnp.int32(0),
        leader=first, played_by=jnp.full(52, -1, jnp.int32),
        played_at=jnp.full(52, -1, jnp.int32),
        voids=jnp.zeros((4, 4), jnp.bool_),
        pending_pass=jnp.full(4, -1, jnp.int32),
        passed=jnp.full(4, -1, jnp.int32),
        received=jnp.full(4, -1, jnp.int32),
        eligible=scores[::-1] - scores >= 150,
        tiebreak=tiebreak, hand_number=hand_number,
    )


def reset(key):
    """Start a fresh match, selecting a dealer uniformly at random."""
    dealer_key, deal_key = jax.random.split(key)
    return _new_hand(deal_key, jnp.zeros(2, jnp.int32),
                     jnp.zeros(2, jnp.int32),
                     jax.random.randint(dealer_key, (), 0, 4, dtype=jnp.int32),
                     jnp.int32(0), jnp.int32(0))


def deal_next(state, key):
    """Deal the next hand; carry scores/bags and rotate the dealer clockwise."""
    return _new_hand(key, state.scores, state.bags, (state.dealer + 1) % 4,
                     state.tiebreak, state.hand_number + 1)


def phase(state):
    return jnp.minimum(state.turn // 4, PLAY)


def actor(state):
    return jnp.where(state.turn < 12,
                     (state.dealer + 1 + state.turn % 4) % 4,
                     state.current_player)


def legal_mask(state):
    """Exact legal actions, including forced no-ops in the fixed 64-turn schedule."""
    p, ph = actor(state), phase(state)
    hand = state.hands[p]
    passing = (state.bids[p] == -1) | (state.bids[(p + 2) % 4] == -1)
    led_suit = jnp.maximum(state.table[0], 0) // 13
    follows = CARD_SUITS == led_suit
    must_follow = (state.trick_pos > 0) & jnp.any(hand & follows)
    play_cards = hand & (~must_follow | follows)
    card_actions = ((ph == PASS) & passing & hand) | ((ph == PLAY) & play_cards)
    bid_actions = jnp.full(14, (ph == BID) & (state.bids[p] != -1))
    decline = ((ph == BLIND) | ((ph == PASS) & ~passing)
               | ((ph == BID) & (state.bids[p] == -1)))
    blind = (ph == BLIND) & state.eligible[p % 2]
    return jnp.concatenate((card_actions, bid_actions,
                            jnp.array([decline, blind], dtype=jnp.bool_)))


def score_hand(bids, tricks, bags):
    """Score both complete contracts, then apply each cumulative ten-bag penalty.

    A failed nil invalidates the ENTIRE partnership contract. Its partner's
    extra tricks neither rescue the contract nor add bags.
    """
    b, t = bids[TEAM_PLAYERS], tricks[TEAM_PLAYERS]
    normal = b > 0
    both_normal = jnp.all(normal, axis=1)
    bid_sum, trick_sum = jnp.sum(b, axis=1), jnp.sum(t, axis=1)
    ordinary_success = trick_sum >= bid_sum
    individual_success = jnp.all(jnp.where(normal, t >= b, t == 0), axis=1)
    success = jnp.where(both_normal, ordinary_success, individual_success)
    value = jnp.sum(jnp.where(normal, 10 * b, jnp.where(b == -1, 200, 100)), axis=1)
    excess = jnp.where(both_normal, trick_sum - bid_sum,
                       jnp.sum(jnp.where(normal, t - b, 0), axis=1))
    bags_added = jnp.where(success, jnp.maximum(excess, 0), 0)
    total_bags = bags + bags_added
    delta = jnp.where(success, value + bags_added, -value) - 100 * (total_bags // 10)
    return delta, total_bags % 10, success, bags_added


def resolve_match(scores, tiebreak):
    """Return (done, winner, next_tiebreak); winner 2 denotes the ten-hand draw."""
    tied = scores[0] == scores[1]
    target_reached = jnp.any(scores >= 500)
    active = (tiebreak > 0) | target_reached
    draw = (tiebreak >= 10) & tied
    done = active & (~tied | draw)
    winner = jnp.where(done, jnp.where(tied, 2, jnp.argmax(scores)), -1).astype(jnp.int32)
    next_tiebreak = jnp.where(active & tied & ~done, tiebreak + 1, tiebreak)
    return done, winner, next_tiebreak


def _blind_turn(state, action):
    p = actor(state)
    bids = state.bids.at[p].set(jnp.where(action == DOUBLE_NIL, -1, state.bids[p]))
    return state._replace(bids=bids, turn=state.turn + 1)


def _pass_turn(state, action):
    p = actor(state)
    pending = state.pending_pass.at[p].set(jnp.where(action < 52, action, -1))
    state = state._replace(pending_pass=pending, turn=state.turn + 1)

    def exchange(s):
        received = s.pending_pass[jnp.array([2, 3, 0, 1])]
        outgoing = jax.nn.one_hot(s.pending_pass, 52, dtype=jnp.bool_)
        incoming = jax.nn.one_hot(received, 52, dtype=jnp.bool_)
        return s._replace(hands=(s.hands & ~outgoing) | incoming,
                          passed=s.pending_pass, received=received,
                          pending_pass=jnp.full(4, -1, jnp.int32))

    # Complete all exchanges only after all selections have been collected.
    return jax.lax.cond(state.turn == 8, exchange, lambda s: s, state)


def _bid_turn(state, action):
    p = actor(state)
    bids = state.bids.at[p].set(jnp.where(action == NOOP, state.bids[p], action - 52))
    return state._replace(bids=bids, turn=state.turn + 1)


def trick_winner(table, table_players):
    """Winner of a complete trick; only led-suit cards or trump can win."""
    suits, ranks = table // 13, table % 13
    strength = jnp.where(suits == 3, 26 + ranks,
                         jnp.where(suits == suits[0], 13 + ranks, -1))
    return table_players[jnp.argmax(strength)]


def _play_turn(state, action):
    p, pos = state.current_player, state.trick_pos
    table = state.table.at[pos].set(action)
    players = state.table_players.at[pos].set(p)
    led_suit = jnp.maximum(table[0], 0) // 13
    is_void = (pos > 0) & (action // 13 != led_suit)
    voids = state.voids.at[p, led_suit].set(state.voids[p, led_suit] | is_void)
    end_trick = pos == 3
    winner = trick_winner(table, players)
    tricks = state.tricks.at[jnp.maximum(winner, 0)].add(end_trick.astype(jnp.int32))
    next_player = jnp.where(end_trick, winner, (p + 1) % 4)
    return state._replace(
        hands=state.hands.at[p, action].set(False), tricks=tricks,
        turn=state.turn + 1, current_player=next_player,
        table=jnp.where(end_trick, -1, table),
        table_players=jnp.where(end_trick, -1, players),
        trick_pos=jnp.where(end_trick, 0, pos + 1),
        leader=jnp.where(end_trick, winner, state.leader),
        played_by=state.played_by.at[action].set(p),
        played_at=state.played_at.at[action].set(state.turn - 12), voids=voids,
    )


def _potential(scores):
    return 0.5 * jnp.tanh((scores[0] - scores[1]).astype(jnp.float32) / 500.0)


def _complete_hand(before, progressed, valid, key):
    delta, bags, success, added = score_hand(progressed.bids, progressed.tricks,
                                            progressed.bags)
    scores = progressed.scores + delta
    done, winner, tiebreak = resolve_match(scores, progressed.tiebreak)
    scored = progressed._replace(scores=scores, bags=bags, tiebreak=tiebreak)
    result_reward = jnp.where(winner == 0, 1.0, jnp.where(winner == 1, -1.0, 0.0))
    reward = (result_reward + jnp.where(done, 0.0, GAMMA_HAND * _potential(scores))
              - _potential(before.scores))
    next_state = jax.lax.cond(done, lambda _: reset(key),
                              lambda _: deal_next(scored, key), operand=None)
    info = dict(
        discount=jnp.where(done, 0.0, GAMMA_HAND).astype(jnp.float32),
        hand_done=jnp.bool_(True), match_done=done, winner=winner,
        score_delta=delta, contract_success=success,
        nil_attempts=jnp.sum(scored.bids == 0),
        nil_successes=jnp.sum((scored.bids == 0) & (scored.tricks == 0)),
        dn_attempts=jnp.sum(scored.bids == -1),
        dn_successes=jnp.sum((scored.bids == -1) & (scored.tricks == 0)),
        bags_added=added, illegal=~valid, final_scores=scores,
    )
    return next_state, reward.astype(jnp.float32), done, info


def _advance_result(progressed, valid):
    info = dict(
        discount=jnp.float32(1.0), hand_done=jnp.bool_(False),
        match_done=jnp.bool_(False), winner=jnp.int32(-1),
        score_delta=jnp.zeros(2, jnp.int32),
        contract_success=jnp.zeros(2, jnp.bool_),
        nil_attempts=jnp.int32(0), nil_successes=jnp.int32(0),
        dn_attempts=jnp.int32(0), dn_successes=jnp.int32(0),
        bags_added=jnp.zeros(2, jnp.int32), illegal=~valid,
        final_scores=progressed.scores,
    )
    return progressed, jnp.float32(0.0), jnp.bool_(False), info


def step(state, action, key):
    """One turn, automatically resetting hands and completed matches.

    Returns (next_state, team0_reward, match_done, info). A supplied illegal
    action falls back to the first legal action and sets info['illegal']; use
    checked_step in interactive code to raise instead. Training always masks
    logits, so it should never exercise this fallback.
    """
    action = jnp.asarray(action, dtype=jnp.int32)
    mask = legal_mask(state)
    in_range = (action >= 0) & (action < N_ACTIONS)
    valid = in_range & mask[jnp.clip(action, 0, N_ACTIONS - 1)]
    safe_action = jnp.where(valid, action, jnp.argmax(mask)).astype(jnp.int32)
    progressed = jax.lax.switch(phase(state),
                               (_blind_turn, _pass_turn, _bid_turn, _play_turn),
                               state, safe_action)
    return jax.lax.cond(state.turn == 63,
                        lambda _: _complete_hand(state, progressed, valid, key),
                        lambda _: _advance_result(progressed, valid), operand=None)


def batch_step(states, actions, keys):
    """Efficient synchronized batch step for training.

    All environments MUST share their scheduled turn (reset them together).
    Every hand is exactly 64 turns, including terminal hands, so normal rollout
    collection preserves this invariant. The whole-batch branch avoids
    shuffling future hands on every ordinary move, which plain vmap(step)
    would do because JAX converts batched conditionals into selections.
    Use vmap(step) for asynchronous states, such as search rollouts.
    """
    actions = jnp.asarray(actions, dtype=jnp.int32)
    masks = jax.vmap(legal_mask)(states)
    clipped = jnp.clip(actions, 0, N_ACTIONS - 1)
    valid = ((actions >= 0) & (actions < N_ACTIONS)
              & jnp.take_along_axis(masks, clipped[:, None], axis=1)[:, 0])
    safe = jnp.where(valid, actions, jnp.argmax(masks, axis=1)).astype(jnp.int32)
    progressed = jax.lax.switch(
        jnp.minimum(states.turn[0] // 4, PLAY),
        tuple(jax.vmap(fn) for fn in (_blind_turn, _pass_turn, _bid_turn, _play_turn)),
        states, safe,
    )
    return jax.lax.cond(
        states.turn[0] == 63,
        lambda _: jax.vmap(_complete_hand)(states, progressed, valid, keys),
        lambda _: jax.vmap(_advance_result)(progressed, valid), operand=None,
    )


def checked_step(state, action, key):
    """Host-side strict action validation for human play/debugging."""
    a = int(action)
    if not 0 <= a < N_ACTIONS or not bool(np.asarray(legal_mask(state))[a]):
        raise ValueError(f'Illegal action {a}; legal actions: '
                         f'{np.flatnonzero(np.asarray(legal_mask(state))).tolist()}')
    return step(state, a, key)


def _observe_for(state, viewer):
    """Actor-relative information set; no other hand or sealed selection is read."""
    players = (viewer + jnp.arange(4, dtype=jnp.int32)) % 4
    teams = (viewer % 2 + jnp.arange(2, dtype=jnp.int32)) % 2
    visible = state.turn >= 4
    exchanged = state.turn >= 8
    hand = (state.hands[viewer] & visible).astype(jnp.float32)
    owner_relative = (state.played_by - viewer) % 4
    owners = jax.nn.one_hot(owner_relative, 4) * (state.played_by >= 0)[:, None]
    table_relative = (state.table_players - viewer) % 4
    table_owners = (jax.nn.one_hot(table_relative, 4)
                    * (state.table_players >= 0)[:, None])
    pieces = [
        hand,
        jax.nn.one_hot(phase(state), 4),
        jnp.array([state.turn / 63.0]),
        jax.nn.one_hot((state.dealer - viewer) % 4, 4),
        jax.nn.one_hot((state.leader - viewer) % 4, 4),
        jax.nn.one_hot(state.bids[players] + 2, 16).reshape(-1),
        state.tricks[players] / 13.0,
        state.scores[teams] / 500.0,
        state.bags[teams] / 10.0,
        state.eligible[teams].astype(jnp.float32),
        jnp.array([state.tiebreak / 10.0, state.tiebreak > 0], dtype=jnp.float32),
        jax.nn.one_hot(state.table, 52).reshape(-1),
        table_owners.reshape(-1),
        jax.nn.one_hot(state.trick_pos, 4),
        owners.reshape(-1),
        (state.played_at + 1) / 52.0,
        state.voids[players].astype(jnp.float32).reshape(-1),
        jax.nn.one_hot(state.passed[viewer], 52) * exchanged,
        jax.nn.one_hot(state.received[viewer], 52) * exchanged,
        hand.reshape(4, 13).sum(axis=1) / 13.0,
    ]
    return jnp.concatenate(pieces).astype(jnp.float32)


def observe(state):
    """Policy observation: current player's hand and public information only.

    During BLIND the entire hand is hidden. During PASS, teammates' sealed
    selections remain hidden; own sent/received cards appear after exchange.
    Exact public card ownership and chronological order preserve the complete
    card-play history for inference about the remaining hidden hands.
    """
    return _observe_for(state, actor(state))


def critic_observe(state):
    """Privileged, fixed team-0 view for the separate training-only value net."""
    return jnp.concatenate((
        _observe_for(state, jnp.int32(0)),
        state.hands.astype(jnp.float32).reshape(-1),
        jax.nn.one_hot(state.pending_pass, 52).reshape(-1),
        jax.nn.one_hot(actor(state), 4),
    )).astype(jnp.float32)


OBS_DIM = int(jax.eval_shape(observe, reset(jax.random.PRNGKey(0))).shape[0])
CRITIC_DIM = int(jax.eval_shape(critic_observe, reset(jax.random.PRNGKey(0))).shape[0])
CARD_RANK_NAMES = tuple('23456789TJQKA')
CARD_SUIT_NAMES = ('♣', '♦', '♥', '♠')


def card_name(card):
    card = int(card)
    return '—' if card < 0 else CARD_RANK_NAMES[card % 13] + CARD_SUIT_NAMES[card // 13]


def action_name(action):
    action = int(action)
    if action < 52:
        return card_name(action)
    if action < 66:
        return 'Nil' if action == 52 else f'Bid {action - 52}'
    return 'Decline / skip' if action == NOOP else 'Double Nil'

print(f'Rules engine ready: {OBS_DIM} policy features, {CRITIC_DIM} critic features, '
      f'{N_ACTIONS} actions, 64 scheduled turns/hand.')


## 4 · Opponent suite

| Style | Development purpose |
|---|---|
| random | Broad action and unusual-contract coverage |
| conservative / aggressive | Underbidding and overbidding pressure |
| nil_specialist | Frequent Nil contracts |
| partner_support | Covering a Nil partner and cooperative play |
| bag_averse | Tight contracts and cumulative bag penalties |
| nil_hunter | Attacking opposing Nil contracts |
| adaptive | Balanced reference policy |
| blind_gambler | Double Nil, swaps, and comeback situations |
| low_card / high_card | Simple policies that expose brittle responses |
| card_counter | Public card/void tracking and master-card play |
| contract_disruptor | Setting opponents, including after its own contract fails |

These are deliberately varied heuristics, not solved-game opponents. They obey the same information and legality rules as the actor. Historical neural policies supply opponents that change as training improves.

In [ ]:
# Observation-limited development opponents. All styles share one vectorized
# feature computation so a heterogeneous JAX batch remains efficient.
HEURISTIC_NAMES = (
    "random", "conservative", "aggressive", "nil_specialist",
    "partner_support", "bag_averse", "nil_hunter", "adaptive",
    "blind_gambler", "low_card", "high_card", "card_counter",
    "contract_disruptor",
)


def estimate_bid_and_nil_risk(hand):
    """Estimate tricks and nil danger using ONLY the acting player's hand.

    The estimate is a baseline, not a claim of optimal bidding. Nil thresholds
    are deliberately stricter than in ordinary Spades because a failed nil
    also loses the partner's entire contract in this ruleset.
    """
    cards = hand.reshape(4, 13).astype(jnp.float32)
    lengths = cards.sum(axis=1)
    side = cards[:3]
    side_len = lengths[:3]
    honors = (
        .94 * side[:, 12]
        + jnp.where(side_len >= 2, .61, .26) * side[:, 11]
        + jnp.where(side_len >= 3, .29, .06) * side[:, 10]
        + jnp.where(side_len >= 4, .11, .01) * side[:, 9]
    )
    # Very long side suits are likely to be ruffed before all their honors cash.
    side_tricks = jnp.sum(honors * jnp.clip(1.05 - .045 * side_len, .65, 1.))
    trump_values = jnp.array([.03, .05, .08, .12, .17, .23, .31,
                             .42, .56, .70, .83, .95, 1.])
    trump_tricks = jnp.sum(cards[3] * trump_values)
    trump_tricks += .36 * jnp.maximum(lengths[3] - 3., 0.)
    shortness = jnp.sum(jnp.where(side_len == 0, .65,
                                 jnp.where(side_len == 1, .38, .0)))
    ruffs = jnp.minimum(shortness, .40 * jnp.maximum(lengths[3] - 1., 0.))
    estimate = jnp.clip(side_tricks + trump_tricks + ruffs, 0., 13.)

    ranks = jnp.arange(13, dtype=jnp.float32)
    # Higher cards are less exposed in a long suit with several small exits.
    exposure = ((ranks + 1.) / 13.) ** 4
    low_exits = jnp.sum(cards * (ranks <= 5), axis=1)
    protection = jnp.sqrt(jnp.maximum(lengths, 1.)) + .30 * low_exits
    nil_risk = jnp.sum(cards * exposure / protection[:, None])
    nil_risk += .9 * jnp.sum(cards[3] * ((ranks + 1.) / 13.) ** 3)
    nil_risk += jnp.sum(cards * (ranks >= 10) * (lengths[:, None] <= 2)) * .45
    # The ace of trump can never be ducked. Long trump holdings also force wins.
    nil_risk += 10. * cards[3, 12] + .18 * jnp.maximum(lengths[3] - 3., 0.)
    return estimate, nil_risk


def _heuristic_blind(state, key, style, legal):
    # IMPORTANT: this phase never inspects any hand, including our own.
    me = actor(state)
    team = me % 2
    deficit = state.scores[1 - team] - state.scores[team]
    chance = jnp.array([.5, .0, .06, .12, .01, .01, .03,
                        .04, .60, .0, .0, .025, .08])[style]
    late_comeback = (state.scores[1 - team] >= 400) & (deficit >= 200)
    chance = jnp.where((style == 7) & late_comeback, .22, chance)
    already_blind = state.bids[(me + 2) % 4] == -1
    chance *= jnp.where(already_blind, jnp.where(style == 8, .6, .08), 1.)
    declare = legal[67] & (jax.random.uniform(key) < chance)
    return jnp.where(declare, 67, 66).astype(jnp.int32)


def _heuristic_pass(state, key, style, legal):
    me = actor(state)
    hand = state.hands[me]
    rank = jnp.arange(52) % 13
    suit = jnp.arange(52) // 13
    lengths = hand.reshape(4, 13).sum(axis=1)
    # A blind nil bidder exports the card most likely to force a trick.
    danger = rank.astype(jnp.float32) + 5. * (suit == 3)
    danger += 2. * ((lengths[suit] <= 2) & (rank >= 9))
    # A supporting partner gives a small card, keeping high covering cards.
    gift = -rank.astype(jnp.float32) - .15 * (suit == 3)
    gift += .15 * (lengths[suit] <= 2)
    score = jnp.where(state.bids[me] == -1, danger, gift)
    score += .02 * jax.random.uniform(key, (52,))
    chosen = jnp.argmax(jnp.where(legal[:52], score, -1.e9))
    return jnp.where(jnp.any(legal[:52]), chosen, 66).astype(jnp.int32)


def _heuristic_bid(state, key, style, legal):
    me = actor(state)
    partner = (me + 2) % 4
    estimate, nil_risk = estimate_bid_and_nil_risk(state.hands[me])
    bias = jnp.array([0., -.35, .75, -.25, -.15, .40, .20,
                     .0, .0, -.80, .80, .20, .10])[style]
    bid = jnp.clip(jnp.floor(estimate + .35 + bias), 1, 13).astype(jnp.int32)
    partner_bid = state.bids[partner]
    # Avoid creating a normal contract that is mathematically impossible.
    bid = jnp.minimum(bid, jnp.maximum(1, 13 - jnp.maximum(partner_bid, 0)))
    threshold = jnp.array([0., .72, .50, 1.32, .78, .55, .82,
                          .95, .90, -.1, -.1, .90, .85])[style]
    partner_special = (partner_bid == 0) | (partner_bid == -1)
    threshold *= jnp.where(partner_special, .25, 1.)
    nil = nil_risk < threshold
    action = jnp.where(nil, 52, 52 + bid)
    # Double-nil players have already committed and have a forced skip.
    return jnp.where(legal[66], 66, action).astype(jnp.int32)


def _heuristic_play(state, key, style, legal):
    me = actor(state)
    partner = (me + 2) % 4
    team = me % 2
    ids = jnp.arange(52)
    ranks = ids % 13
    suits = ids // 13
    hand = state.hands[me]
    lengths = hand.reshape(4, 13).sum(axis=1)
    leading = state.trick_pos == 0
    led = jnp.maximum(state.table[0], 0) // 13
    table_cards = jnp.maximum(state.table, 0)
    table_suits = table_cards // 13
    table_rank = table_cards % 13
    strength = jnp.where(table_suits == 3, 26 + table_rank,
                         jnp.where(table_suits == led, 13 + table_rank, -1))
    strength = jnp.where(state.table >= 0, strength, -100)
    table_index = jnp.argmax(strength)
    current_card = table_cards[table_index]
    current_winner = jnp.where(leading, -1, state.table_players[table_index])
    candidate_strength = jnp.where(suits == 3, 26 + ranks,
                                   jnp.where(suits == led, 13 + ranks, -1))
    would_win = leading | (candidate_strength > jnp.max(strength))

    # Card counting uses played-card ownership and our own cards only.
    unseen = ((state.played_by < 0) & ~hand).reshape(4, 13)
    unseen_count = unseen.sum(axis=1)
    higher = jnp.flip(jnp.cumsum(jnp.flip(unseen, axis=1), axis=1), axis=1) - unseen
    unseen_total = jnp.maximum(unseen_count.sum(), 1).astype(jnp.float32)
    seats = jnp.arange(4)
    remaining_cards = 13 - jnp.sum(state.played_by[:, None] == seats[None, :], axis=0)
    relative = (seats - me) % 4
    future = (relative > 0) & (relative <= 3 - state.trick_pos)

    def survival(cards, lead_suits):
        cs, cr = cards // 13, cards % 13
        high_fraction = higher[cs, cr].astype(jnp.float32) / unseen_total
        p_high = 1. - jnp.exp(-high_fraction[:, None] * remaining_cards[None, :])
        known_void = state.voids[:, cs].T
        p_high = jnp.where(known_void, 0., p_high)
        lead_fraction = unseen_count[lead_suits].astype(jnp.float32) / unseen_total
        p_void = jnp.exp(-lead_fraction[:, None] * remaining_cards[None, :])
        p_void = jnp.where(state.voids[:, lead_suits].T, 1., p_void)
        trump_fraction = unseen_count[3].astype(jnp.float32) / unseen_total
        p_trump = 1. - jnp.exp(-trump_fraction * remaining_cards)
        p_trump = jnp.where(state.voids[:, 3], 0., p_trump)
        p_ruff = p_void * p_trump[None, :] * (cs[:, None] != 3)
        p_beat = 1. - (1. - p_high) * (1. - p_ruff)
        return jnp.prod(jnp.where(future[None, :], 1. - p_beat, 1.), axis=1)

    all_cards = jnp.concatenate((ids, current_card[None]))
    lead_suits = jnp.concatenate((jnp.where(leading, suits, led), led[None]))
    survives = survival(all_cards, lead_suits)
    own_win_probability = would_win.astype(jnp.float32) * survives[:52]
    existing_win_probability = survives[52]
    partner_winning = current_winner == partner
    team_win_probability = own_win_probability + (
        (~would_win) * partner_winning * existing_win_probability
    )

    own_team = seats % 2 == team
    special = state.bids <= 0
    live_nil = special & (state.tricks == 0)
    failed_nil = special & (state.tricks > 0)
    remaining_tricks = 13 - jnp.maximum(state.turn - 12, 0) // 4
    normal_bid = jnp.where(state.bids > 0, state.bids, 0)
    normal_tricks = jnp.where(state.bids > 0, state.tricks, 0)
    our_need = jnp.maximum(jnp.sum(jnp.where(own_team, normal_bid - normal_tricks, 0)), 0)
    their_need = jnp.maximum(jnp.sum(jnp.where(~own_team, normal_bid - normal_tricks, 0)), 0)
    our_failed = jnp.any(own_team & failed_nil) | (our_need > remaining_tricks)
    their_failed = jnp.any(~own_team & failed_nil) | (their_need > remaining_tricks)
    own_nil = live_nil[me] & ~our_failed
    partner_nil = live_nil[partner] & ~our_failed
    opponents_nil = (~own_team) & live_nil & ~their_failed
    current_opp_nil = jnp.any(opponents_nil & (seats == current_winner))

    bid_drive = jnp.array([1., 1.20, 1.45, 1., 1.10, 1.0, .95,
                          1.20, 1.20, 1., 1., 1.20, .85])[style]
    bag_weight = jnp.array([.4, .55, .18, .45, .5, 1.6, .35,
                           .85, .45, .4, .2, .95, .65])[style]
    hunt_weight = jnp.array([0., .7, .65, 1., .7, .75, 2.3,
                            1.4, 1., .0, .0, 1.35, 2.0])[style]
    support_weight = jnp.array([0., 1., .8, 1.2, 1.8, 1., .9,
                               1.4, 1.1, 0., 0., 1.4, .9])[style]
    denial_weight = jnp.array([0., .18, .45, .20, .20, .20, .35,
                              .35, .30, .0, .0, .40, 1.0])[style]
    urgency = 1. + 2. * our_need / jnp.maximum(remaining_tricks, 1)
    our_drive = jnp.where(our_need > 0, bid_drive * urgency,
                         -bag_weight * (1. + state.bags[team] / 4.))
    # In this variant a failed special contract is already irrecoverable.
    # Stop chasing our own bid; only disruption of opponents has value then.
    our_drive = jnp.where(our_failed, 0., our_drive)
    deny = jnp.where(their_need > 0, denial_weight,
                     -denial_weight * (.5 + state.bags[1 - team] / 4.))
    deny = jnp.where(their_failed, 0., deny)
    score = (our_drive + deny) * team_win_probability

    master = higher.reshape(52) == 0
    rank_value = ranks.astype(jnp.float32) / 12.
    card_cost = .22 * rank_value + .22 * (suits == 3) + .08 * master
    score -= card_cost
    score += leading * (.025 * lengths[suits] +
                        .08 * (lengths[suits] == 1) * (lengths[3] >= 2))
    score += leading * (style == 11) * .25 * master * (suits != 3) * (our_need > 0)

    # Protect a live partner nil, particularly if they currently hold the trick.
    partner_future = ((partner - me) % 4) <= 3 - state.trick_pos
    cover = jnp.where(partner_winning, 9. * would_win,
                      jnp.where(leading | partner_future, 2.8 * own_win_probability, 0.))
    score += partner_nil * support_weight * cover

    # If an opposing nil currently holds the trick, leave it with them.
    nil_trap = current_opp_nil * (~would_win) * existing_win_probability * 7.
    # Lead a small card in a suit an opposing nil can still follow; a low
    # percentile card leaves that opponent fewer smaller cards to duck with.
    above_fraction = higher.reshape(52) / jnp.maximum(unseen_count[suits], 1)
    can_follow = ~state.voids[:, suits].T
    target_follow = jnp.sum(can_follow * opponents_nil[None, :], axis=1)
    lead_trap = leading * target_follow * above_fraction * .85
    score += hunt_weight * (nil_trap + lead_trap)

    # A nil bidder dumps dangerous cards when safely below the winning card.
    danger = rank_value + .55 * (suits == 3) + .30 * master
    danger += .18 * (lengths[suits] <= 2)
    nil_score = -9. * own_win_probability + 1.45 * danger * (1. - own_win_probability)
    nil_score -= .08 * rank_value
    nil_score += .25 * hunt_weight * (nil_trap + lead_trap)
    score = jnp.where(own_nil, nil_score, score)

    # Deliberately simple endpoints expose policies overfitted to clever bots.
    score = jnp.where(style == 9, -ranks.astype(jnp.float32) - .05 * (suits == 3), score)
    score = jnp.where(style == 10, ranks.astype(jnp.float32) + .05 * (suits == 3), score)
    score += .035 * jax.random.uniform(key, (52,))
    return jnp.argmax(jnp.where(legal[:52], score, -1.e9)).astype(jnp.int32)


def heuristic_action(state, key, style):
    """Legal action for one observation-limited opponent; supports jit/vmap.

    Private information is restricted to state.hands[actor(state)]. No opponent
    hand, pending pass, centralized observation, or undealt future is consulted.
    During blind polling even the acting player's hand is ignored. Actions can
    depend on public seat, bids, scores, bags, table, voids, and played cards.
    """
    style = jnp.clip(jnp.asarray(style, jnp.int32), 0, len(HEURISTIC_NAMES) - 1)
    legal = legal_mask(state)
    chosen = jax.lax.switch(
        phase(state),
        (_heuristic_blind, _heuristic_pass, _heuristic_bid, _heuristic_play),
        state, key, style, legal,
    )
    random_choice = jnp.argmax(jnp.where(legal, jax.random.uniform(key, (68,)), -1.)).astype(jnp.int32)
    chosen = jnp.where(style == 0, random_choice, chosen)
    # A common final mask protects even deliberately weak stress-test styles.
    return jnp.where(legal[chosen], chosen, jnp.argmax(legal)).astype(jnp.int32)


## 5 · Stronger card-play training and the existing PPO objective

The default residual actor and separate critic retain their parameter names, shapes, width, and information boundaries. The match-win reward and potential shaping are unchanged. The score-difference auxiliary stays disabled because it can reward delaying wins against a losing opponent.

**Coherent teaching:** the previous initializer changed among seven teacher styles every turn and ended after only 16 batches. V3 uses the `adaptive` teacher consistently and declines its random blind gamble. The initial target is **6,291,456 fresh scheduled teacher decisions**, with a separate **1,024-example minibatch and two supervised epochs**. This preserves meaningful optimizer work when PPO batches become large. Teacher agreement measures imitation, not playing strength.

**Continuing guidance:** learner-visited bidding/blind positions retain a 0.1 cross-entropy reference, and play/pass positions now receive a 0.3 reference. Opponent actions and forced no-ops receive no actor or teacher loss. The teacher sees only the acting player's visible hand and public information. The policy can override its advice, including declaring a legal Double Nil. This regularization can bias strategy toward an imperfect teacher; held-out match wins determine usefulness.

PPO retains three epochs, the existing learning rate, legal masking, clipping, separate critic, gradient clipping, and KL stopping. GPU sizing preserves **8 minibatches per PPO epoch** rather than multiplying sequential Adam steps with a fixed tiny minibatch. The default 128-turn horizon and per-turn GAE lambda 0.995 are retained. Experimental hand-based GAE is implemented but disabled: isolated tests did not establish a strength benefit.

The self-play/historical/scripted league mix and all 13 opponents remain. Very long training trajectories still refresh after 128 completed hands, bootstrap from the real continuing state, and cut the advantage trace. Such refreshes are never wins or draws. Human games keep the exact rules; evaluation limits remain explicitly unresolved outcomes.

In [ ]:
"""JAX PPO, persistent league matchmaking, and optional supervised initialization.

This cell follows the engine and opponents cells in the standalone notebook.
All values needed to resume training live in TrainingState; no Python RNG is used.
"""
from typing import NamedTuple
import math
import jax
import jax.numpy as jnp
import numpy as np
import flax.linen as nn
import optax

TRAINING_REVISION = 'strong-teacher-hand-gae-v3'


def learning_config(overrides=None):
    cfg = dict(seed=42, n_env=256, rollout_len=128, ppo_epochs=3,
               minibatch_size=4096, width=256, lr=2e-4,
               gae_lambda=0.995, gae_per_hand=False, clip_eps=0.20, value_clip=0.25,
               max_grad_norm=0.5, entropy_coef=0.015, value_coef=0.5,
               target_kl=0.04, snapshot_size=4,
               snapshot_interval_updates=100, selfplay_prob=0.50,
               snapshot_prob=0.25, bootstrap_epochs=2, bootstrap_minibatch_size=1024,
               teacher_warmup_decisions=6291456,
               hand_score_reward_coef=0.0, bidding_anchor_coef=0.1,
               play_anchor_coef=0.3, teacher_style=7, teacher_decline_blind=True,
               max_training_match_hands=128)
    if overrides:
        unknown = set(overrides) - set(cfg)
        if unknown:
            raise ValueError("Unknown training settings: " + ", ".join(sorted(unknown)))
        cfg.update(overrides)
    if int(cfg["seed"]) != cfg["seed"] or not 0 <= cfg["seed"] < 2 ** 32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    cfg["seed"] = int(cfg["seed"])
    for name in ("n_env", "rollout_len", "ppo_epochs", "minibatch_size", "width",
                 "snapshot_size", "snapshot_interval_updates", "bootstrap_epochs",
                 "max_training_match_hands", "bootstrap_minibatch_size", "teacher_warmup_decisions"):
        if int(cfg[name]) != cfg[name] or cfg[name] < 1:
            raise ValueError(f"{name} must be a positive integer")
        cfg[name] = int(cfg[name])
    if cfg["n_env"] * cfg["rollout_len"] % cfg["minibatch_size"]:
        raise ValueError("n_env * rollout_len must be divisible by minibatch_size")
    # Tiny CPU fixtures can use one smaller teacher minibatch. Production batches
    # keep teacher SGD independent of the much larger PPO minibatch size.
    if cfg['n_env'] * cfg['rollout_len'] % min(cfg['bootstrap_minibatch_size'], cfg['n_env'] * cfg['rollout_len']):
        raise ValueError('Teacher minibatch size must divide the rollout size')
    for name in ('gae_per_hand', 'teacher_decline_blind'):
        if not isinstance(cfg[name], (bool, np.bool_)):
            raise ValueError(f'{name} must be boolean')
    if int(cfg['teacher_style']) != cfg['teacher_style'] or not 0 <= cfg['teacher_style'] < len(HEURISTIC_NAMES):
        raise ValueError('teacher_style must identify an existing public-information heuristic')
    cfg['teacher_style'] = int(cfg['teacher_style'])
    if not (0 <= cfg["selfplay_prob"] <= 1 and 0 <= cfg["snapshot_prob"] <= 1
            and cfg["selfplay_prob"] + cfg["snapshot_prob"] <= 1):
        raise ValueError("selfplay_prob and snapshot_prob must be nonnegative and sum to <= 1")
    if not 0 < cfg["gae_lambda"] <= 1:
        raise ValueError("gae_lambda must lie in (0, 1]")
    for name in ("lr", "clip_eps", "value_clip", "max_grad_norm", "target_kl"):
        if not math.isfinite(cfg[name]) or cfg[name] <= 0:
            raise ValueError(f"{name} must be finite and positive")
    for name in ("entropy_coef", "value_coef", "hand_score_reward_coef", "bidding_anchor_coef", "play_anchor_coef"):
        if not math.isfinite(cfg[name]) or cfg[name] < 0:
            raise ValueError(f"{name} must be finite and nonnegative")
    return cfg


def _network_features(features):
    """Compress extreme score inputs while retaining exact ordinary-range values.

    Actor and critic observations both begin with the same public feature layout.
    Scores occupy positions 133:135 and are already divided by 500 in the engine.
    Scores in [-500, 500] pass through unchanged; larger magnitudes use logarithmic
    tails so million-point negative games cannot overwhelm card/phase features.
    The game state, legal actions, and actual point totals remain untouched.
    """
    score = features[..., 133:135]
    magnitude = jnp.abs(score)
    compressed = jnp.sign(score) * (jnp.minimum(magnitude, 1.)
                                    + jnp.log(jnp.maximum(magnitude, 1.)))
    return features.at[..., 133:135].set(compressed)


class ResidualTower(nn.Module):
    width: int
    output_size: int
    actor_output: bool = False

    @nn.compact
    def __call__(self, features):
        x = nn.silu(nn.Dense(self.width, name="input")(_network_features(features)))
        for i in range(3):
            h = nn.LayerNorm(name=f"norm_{i}")(x)
            h = nn.silu(nn.Dense(self.width, name=f"hidden_{i}")(h))
            h = nn.Dense(self.width, name=f"residual_{i}")(h)
            x = nn.silu(x + 0.5 * h)
        init = nn.initializers.normal(0.01) if self.actor_output else nn.initializers.zeros
        return nn.Dense(self.output_size, kernel_init=init, name="output")(x)


class ActorCritic(nn.Module):
    """Actor and critic have completely separate parameters and encoders."""
    width: int = 256

    def setup(self):
        self.actor_network = ResidualTower(self.width, N_ACTIONS, True)
        self.critic_network = ResidualTower(self.width, 1, False)

    def actor(self, obs):
        return self.actor_network(obs)

    def value(self, critic_obs):
        return self.critic_network(critic_obs)[..., 0]

    def __call__(self, obs, critic_obs):
        return self.actor(obs), self.value(critic_obs)


def model_for_config(cfg):
    return ActorCritic(width=int(cfg.get("width", 256)))


def policy_logits(params, obs, cfg):
    model = model_for_config(cfg)
    return model.apply({"params": params}, obs, method=model.actor)


actor_logits = policy_logits


def value_apply(params, critic_obs, cfg):
    model = model_for_config(cfg)
    return model.apply({"params": params}, critic_obs, method=model.value)


def _optimizer(cfg):
    return optax.chain(optax.clip_by_global_norm(cfg["max_grad_norm"]),
                       optax.adam(cfg["lr"], eps=1e-5))


class TrainingState(NamedTuple):
    params: object
    opt_state: object
    rng: object
    envs: object
    league_kind: object       # 0 current self-play; 1 historical; 2 scripted
    league_style: object
    league_snapshot: object
    learner_team: object     # 0 or 1, used only for mixed-opponent matches
    snapshot_params: object  # leading dimension snapshot_size
    snapshot_count: object
    snapshot_cursor: object
    snapshot_retiring: object
    last_snapshot_update: object
    updates: object
    decisions_per_update: object
    bootstrap_updates: object
    bootstrap_complete: object
    env_steps_low: object
    env_steps_high: object
    base_updates: object

    @property
    def env_steps(self):
        # Host-side exact arithmetic: a cumulative JAX int32 counter would wrap
        # after 2.1 billion decisions, and enabling x64 would slow a T4 down.
        prior = (int(self.env_steps_high) << 32) | int(self.env_steps_low)
        return prior + (int(self.updates) - int(self.base_updates)) * int(self.decisions_per_update)


class Transition(NamedTuple):
    obs: object
    critic_obs: object
    actions: object
    logp: object
    values: object
    rewards: object
    discounts: object
    policy_mask: object
    actor_sign: object
    legal: object
    teacher_actions: object
    teacher_mask: object


class PPOBatch(NamedTuple):
    obs: object
    critic_obs: object
    actions: object
    logp: object
    old_values: object
    returns: object
    advantages: object
    policy_mask: object
    legal: object
    teacher_actions: object
    teacher_mask: object


def _league_assignments(key, cfg, count, retiring):
    n = cfg["n_env"]
    k_kind, k_team, k_style, k_snapshot = jax.random.split(key, 4)
    u = jax.random.uniform(k_kind, (n,))
    kind = jnp.where(u < cfg["selfplay_prob"], 0,
                     jnp.where(u < cfg["selfplay_prob"] + cfg["snapshot_prob"], 1, 2))
    available = ((jnp.arange(cfg["snapshot_size"]) < count)
                 & (jnp.arange(cfg["snapshot_size"]) != retiring))
    slot = jax.random.categorical(k_snapshot, jnp.where(available, 0., -1e9), shape=(n,))
    kind = jnp.where((kind == 1) & ~jnp.any(available), 0, kind)
    team = jax.random.randint(k_team, (n,), 0, 2)
    style = jax.random.randint(k_style, (n,), 0, len(HEURISTIC_NAMES))
    return kind.astype(jnp.int32), style, slot.astype(jnp.int32), team


def initialize_training(cfg):
    cfg = learning_config(cfg)
    key, k_model, k_env, k_league = jax.random.split(jax.random.PRNGKey(cfg["seed"]), 4)
    example = reset(jax.random.PRNGKey(0))
    model = model_for_config(cfg)
    params = model.init(k_model, observe(example), critic_observe(example))["params"]
    envs = jax.vmap(reset)(jax.random.split(k_env, cfg["n_env"]))
    kind, style, slot, team = _league_assignments(k_league, cfg, jnp.int32(1), jnp.int32(-1))
    bank = jax.tree_util.tree_map(lambda p: jnp.repeat(p[None], cfg["snapshot_size"], axis=0), params)
    return TrainingState(params, _optimizer(cfg).init(params), key, envs,
                         kind, style, slot, team, bank, jnp.int32(1), jnp.int32(1 % cfg["snapshot_size"]),
                         jnp.int32(-1), jnp.int32(0), jnp.int32(0),
                         jnp.int32(cfg["n_env"] * cfg["rollout_len"]), jnp.int32(0), jnp.bool_(False),
                         jnp.uint32(0), jnp.uint32(0), jnp.int32(0))


def _masked_log_probs(logits, legal):
    return jax.nn.log_softmax(jnp.where(legal, logits, -1e9), axis=-1)


def _tree_finite(tree):
    return jnp.all(jnp.stack([jnp.all(jnp.isfinite(x)) for x in jax.tree_util.tree_leaves(tree)]))


_UPDATE_CHECK_NAMES = ("diagnostics_finite", "gradients_finite", "grad_norm_finite",
                       "params_finite", "optimizer_finite")


def _update_checks(diagnostics, grads, grad_norm, params, opt_state):
    # Adam's moments can overflow even when the resulting parameters are finite.
    # Check the entire candidate before committing any part of the update.
    return jnp.stack((jnp.all(jnp.isfinite(diagnostics)), _tree_finite(grads),
                      jnp.isfinite(grad_norm), _tree_finite(params), _tree_finite(opt_state)))


def _reduce_update_rows(rows):
    """Sum diagnostics/counts and AND safety flags without floating-point votes.

    Compiled float32 count / count can use an approximate reciprocal and round
    below 1. An averaged flag compared to exactly 1 can then report a
    failure despite every minibatch being finite. Flags always stay boolean;
    attempted/accepted counts stay int32 throughout both scan reductions.
    """
    diagnostics, counts, checks = rows
    return jnp.sum(diagnostics, axis=0), jnp.sum(counts, axis=0), jnp.all(checks, axis=0)


def _update_metrics(rows, names):
    diagnostics, counts, checks = _reduce_update_rows(rows)
    metrics = dict(zip(names, diagnostics / jnp.maximum(counts[0], 1)))
    metrics.update(zip(_UPDATE_CHECK_NAMES, checks))
    metrics.update(attempted_updates=counts[0], accepted_updates=counts[1],
                   rejected_updates=counts[0] - counts[1], finite=jnp.all(checks))
    return metrics


def _masked_mean(x, mask):
    return jnp.sum(x * mask) / jnp.maximum(jnp.sum(mask), 1.)


def _refresh_snapshot(state, cfg):
    """Retire a bank slot, drain its active matches, and only then replace it.

    A match's historical opponent therefore cannot change underneath it. If all
    slots are busy, publication waits for that slot's existing matches to finish.
    """
    due = state.updates - state.last_snapshot_update >= cfg["snapshot_interval_updates"]
    unused = state.snapshot_count < cfg["snapshot_size"]
    candidate = jnp.where(unused, state.snapshot_count, state.snapshot_cursor)
    retiring = jnp.where((state.snapshot_retiring < 0) & due, candidate, state.snapshot_retiring)
    active = jnp.any((state.league_kind == 1) & (state.league_snapshot == retiring))
    ready = (retiring >= 0) & ~active

    def publish(s):
        bank = jax.tree_util.tree_map(lambda b, p: b.at[retiring].set(p), s.snapshot_params, s.params)
        return s._replace(snapshot_params=bank,
                          snapshot_count=jnp.minimum(s.snapshot_count + (retiring >= s.snapshot_count),
                                                     cfg["snapshot_size"]),
                          snapshot_cursor=(retiring + 1) % cfg["snapshot_size"],
                          snapshot_retiring=jnp.int32(-1), last_snapshot_update=s.updates)

    state = state._replace(snapshot_retiring=retiring)
    return jax.lax.cond(ready, publish, lambda s: s, state)


_ROLLOUT_METRIC_NAMES = (
    "matches", "team0_wins", "team1_wins", "draws", "hands",
    "nil_attempts", "nil_successes", "dn_attempts", "dn_successes",
    "illegal_actions", "final_score0_sum", "final_score1_sum",
    "opponent_matches", "learner_wins", "learner_draws",
    "scripted_matches", "scripted_wins", "historical_matches", "historical_wins",
    "successful_contracts", "learner_contracts", "learner_successful_contracts",
    "learner_hand_score_sum", "opponent_hand_score_sum", "opponent_contracts",
    "learner_bid_seats", "learner_nil_bids", "learner_dn_bids",
    "learner_nil_made", "learner_dn_made", "learner_normal_bid_sum",
    "learner_normal_bids", "hand_reward_abs_sum", "score_bonus_abs_sum",
    "entropy_blind_sum", "entropy_bid_sum", "entropy_play_sum",
    "entropy_blind_count", "entropy_bid_count", "entropy_play_count", "training_truncations")


def _training_rewards(engine_rewards, info, cfg):
    """Team-0 reward with an explicit dense score auxiliary for learning.

    The engine's match-win reward and potential term remain present. The extra
    term uses actual scored points (including failed contracts and bag penalties)
    and stays informative even in long, deeply negative matches. It changes the
    training objective; it is not policy-invariant potential shaping. Evaluation
    and human games always use the original rules and true match winners.
    """
    point_difference = (info['score_delta'][..., 0].astype(jnp.float32)
                        - info['score_delta'][..., 1].astype(jnp.float32))
    return engine_rewards + cfg['hand_score_reward_coef'] * point_difference / 100.


def teacher_action(state, key, cfg):
    """Coherent public-information advice, with no random blind gamble to imitate.

    The heuristic only reads the acting player's visible hand and public state.
    Declining blind is guidance, never a restriction on legal policy actions.
    PPO can still learn to declare Double Nil in appropriate game situations.
    """
    advice = heuristic_action(state, key, jnp.int32(cfg['teacher_style']))
    if cfg['teacher_decline_blind']:
        advice = jnp.where(phase(state) == BLIND, NOOP, advice)
    return advice


def _gae_trace_discount(discounts, cfg):
    # Rewards and the engine's gamma are hand-based. Apply lambda on that same
    # timescale so a hand does not incur 64 extra credit-decay factors.
    if cfg['gae_per_hand']:
        return discounts * jnp.where(discounts < 1., cfg['gae_lambda'], 1.)
    return discounts * cfg['gae_lambda']


def _refresh_training_trajectories(params, envs, rewards, discounts, done, hand_done, key, cfg):
    """Refresh very long training samples with proper time-limit bootstrapping.

    This is a rollout sampling boundary, never a game winner or draw. The critic
    bootstraps from the real continuing game before fresh games are installed;
    zeroing the trace discount then prevents GAE crossing into unrelated deals.
    Engine games, evaluation, and human play have no such sampling boundary.
    """
    truncated = hand_done & ~done & (envs.hand_number >= cfg['max_training_match_hands'])

    def refresh(_):
        model = model_for_config(cfg)
        continuation = model.apply({'params': params}, jax.vmap(critic_observe)(envs), method=model.value)
        fresh = jax.vmap(reset)(jax.random.split(jax.random.fold_in(key, 1), cfg['n_env']))
        result = jax.tree_util.tree_map(
            lambda new, old: jnp.where(truncated.reshape((cfg['n_env'],) + (1,) * (old.ndim - 1)),
                                      new, old), fresh, envs)
        return (result, rewards + jnp.where(truncated, discounts * continuation, 0.),
                jnp.where(truncated, 0., discounts))

    envs, rewards, discounts = jax.lax.cond(jnp.any(truncated), refresh,
        lambda _: (envs, rewards, discounts), operand=None)
    return envs, rewards, discounts, truncated


def make_train_step(cfg):
    """Return step(state) -> (state, scalar_metrics).

    Rollout and optimization compile separately. Games persist across rollouts
    and sessions until a true result or a bootstrapped training sampling boundary.
    """
    cfg = learning_config(cfg)
    model, tx = model_for_config(cfg), _optimizer(cfg)
    n, horizon = cfg["n_env"], cfg["rollout_len"]
    total = n * horizon
    n_minibatches = total // cfg["minibatch_size"]

    @jax.jit
    def collect(state):
        def one_step(s, unused):
            key, k_action, k_env, k_heuristic, k_league = jax.random.split(s.rng, 5)
            obs = jax.vmap(observe)(s.envs)
            central_obs = jax.vmap(critic_observe)(s.envs)
            legal = jax.vmap(legal_mask)(s.envs)
            who = jax.vmap(actor)(s.envs)
            logits, values = model.apply({"params": s.params}, obs, central_obs)
            learner_turn = (s.league_kind == 0) | ((who % 2) == s.learner_team)
            historical_turn = (s.league_kind == 1) & ~learner_turn

            def historical_logits(_):
                # Batch by bank slot, never materialize one model per environment.
                bank_logits = jax.vmap(lambda p: model.apply({"params": p}, obs, method=model.actor))(
                    s.snapshot_params)
                return bank_logits[s.league_snapshot, jnp.arange(n)]

            old_logits = jax.lax.cond(jnp.any(historical_turn), historical_logits,
                                      lambda _: jnp.zeros_like(logits), operand=None)
            acting_logits = jnp.where(historical_turn[:, None], old_logits, logits)
            actions = jax.random.categorical(k_action, jnp.where(legal, acting_logits, -1e9), axis=-1)
            scripted_turn = (s.league_kind == 2) & ~learner_turn
            scripted = jax.lax.cond(jnp.any(scripted_turn),
                lambda _: jax.vmap(heuristic_action)(s.envs, jax.random.split(k_heuristic, n), s.league_style),
                lambda _: jnp.zeros((n,), jnp.int32), operand=None)
            actions = jnp.where(scripted_turn, scripted, actions).astype(jnp.int32)
            logs = _masked_log_probs(logits, legal)
            logp = jnp.take_along_axis(logs, actions[:, None], axis=-1)[:, 0]
            next_envs, rewards, done, info = batch_step(s.envs, actions, jax.random.split(k_env, n))
            engine_rewards = rewards
            rewards = _training_rewards(engine_rewards, info, cfg)
            hand_rewards = rewards
            next_envs, rewards, trace_discounts, truncated = _refresh_training_trajectories(
                s.params, next_envs, rewards, info['discount'], done, info['hand_done'], k_env, cfg)
            # Continue public-information guidance on the learner's own visited
            # positions, including card play that the old bidding-only anchor omitted.
            teacher_phase = (s.envs.turn < 4) | ((s.envs.turn >= 8) & (s.envs.turn < 12))
            teacher_actions = jax.lax.cond(jnp.any(teacher_phase) | (cfg['play_anchor_coef'] > 0),
                lambda _: jax.vmap(lambda env, key: teacher_action(env, key, cfg))(
                    s.envs, jax.random.split(k_heuristic, n)),
                lambda _: jnp.full(n, NOOP, jnp.int32), operand=None)
            teacher_mask = teacher_phase & learner_turn & (jnp.sum(legal, axis=-1) > 1)
            transition = Transition(obs, central_obs, actions, logp, values, rewards,
                                    trace_discounts, learner_turn & (jnp.sum(legal, axis=-1) > 1),
                                    jnp.where(who % 2 == 0, 1., -1.), legal, teacher_actions, teacher_mask)
            new_assignments = _league_assignments(k_league, cfg, s.snapshot_count, s.snapshot_retiring)
            old_assignments = (s.league_kind, s.league_style, s.league_snapshot, s.learner_team)
            kind, style, slot, team = [jnp.where(done | truncated, new, old)
                                      for new, old in zip(new_assignments, old_assignments)]
            mixed = (s.league_kind != 0) & done
            won = info["winner"] == s.learner_team
            hand_end = info['hand_done']
            learner_teams = ((s.league_kind[:, None] == 0)
                             | (jnp.arange(2)[None, :] == s.learner_team[:, None]))
            our_contracts = hand_end[:, None] & learner_teams
            other_contracts = hand_end[:, None] & ~learner_teams
            learner_seats = hand_end[:, None] & learner_teams[:, jnp.arange(4) % 2]
            nil_bids = learner_seats & (s.envs.bids == 0)
            dn_bids = learner_seats & (s.envs.bids == -1)
            normal_bids = learner_seats & (s.envs.bids > 0)
            # The last trick's winner has not yet been counted in s.envs.tricks.
            final_table = s.envs.table.at[jnp.arange(n), s.envs.trick_pos].set(actions)
            final_players = s.envs.table_players.at[jnp.arange(n), s.envs.trick_pos].set(who)
            last_winners = jax.vmap(trick_winner)(final_table, final_players)
            scored_tricks = s.envs.tricks + jax.nn.one_hot(last_winners, 4, dtype=jnp.int32)
            actionable = learner_turn & (jnp.sum(legal, axis=-1) > 1)
            entropies = -jnp.sum(jnp.exp(logs) * logs, axis=-1)
            phases = jax.vmap(phase)(s.envs)
            entropy_masks = jnp.stack([actionable & (phases == ph) for ph in (BLIND, BID, PLAY)])
            entropy_sums = jnp.sum(entropy_masks * entropies[None, :], axis=1)
            entropy_counts = jnp.sum(entropy_masks, axis=1)
            metric = jnp.stack([
                jnp.sum(done), jnp.sum(done & (info["winner"] == 0)),
                jnp.sum(done & (info["winner"] == 1)), jnp.sum(done & (info["winner"] == 2)),
                jnp.sum(info["hand_done"]), jnp.sum(info["nil_attempts"]), jnp.sum(info["nil_successes"]),
                jnp.sum(info["dn_attempts"]), jnp.sum(info["dn_successes"]), jnp.sum(info["illegal"]),
                jnp.sum(jnp.where(done, info["final_scores"][:, 0], 0)),
                jnp.sum(jnp.where(done, info["final_scores"][:, 1], 0)),
                jnp.sum(mixed), jnp.sum(mixed & won), jnp.sum(mixed & (info["winner"] == 2)),
                jnp.sum(done & (s.league_kind == 2)), jnp.sum(done & (s.league_kind == 2) & won),
                jnp.sum(done & (s.league_kind == 1)), jnp.sum(done & (s.league_kind == 1) & won),
                jnp.sum(info["contract_success"]),
                jnp.sum(our_contracts), jnp.sum(our_contracts & info['contract_success']),
                jnp.sum(jnp.where(our_contracts, info['score_delta'], 0)),
                jnp.sum(jnp.where(other_contracts, info['score_delta'], 0)), jnp.sum(other_contracts),
                jnp.sum(learner_seats), jnp.sum(nil_bids), jnp.sum(dn_bids),
                jnp.sum(nil_bids & (scored_tricks == 0)), jnp.sum(dn_bids & (scored_tricks == 0)),
                jnp.sum(jnp.where(normal_bids, s.envs.bids, 0)), jnp.sum(normal_bids),
                jnp.sum(jnp.abs(hand_rewards)), jnp.sum(jnp.abs(hand_rewards - engine_rewards)),
                *entropy_sums, *entropy_counts,
                jnp.sum(truncated),
            ]).astype(jnp.float32)
            s = s._replace(rng=key, envs=next_envs, league_kind=kind, league_style=style,
                           league_snapshot=slot, learner_team=team)
            return s, (transition, metric)

        state, (trajectory, metric_rows) = jax.lax.scan(one_step, state, None, length=horizon)
        last_values = model.apply({"params": state.params}, jax.vmap(critic_observe)(state.envs),
                                  method=model.value)

        def gae_step(carry, tr):
            next_advantage, next_value = carry
            delta = tr.rewards + tr.discounts * next_value - tr.values
            advantage = delta + _gae_trace_discount(tr.discounts, cfg) * next_advantage
            return (advantage, tr.values), advantage

        _, advantages = jax.lax.scan(gae_step, (jnp.zeros(n), last_values), trajectory, reverse=True)
        returns = advantages + trajectory.values
        signed_adv = advantages * trajectory.actor_sign
        mask = trajectory.policy_mask.astype(jnp.float32)
        mean_adv = _masked_mean(signed_adv, mask)
        var_adv = _masked_mean((signed_adv - mean_adv) ** 2, mask)
        normalized_adv = (signed_adv - mean_adv) / jnp.sqrt(var_adv + 1e-8)
        batch = PPOBatch(trajectory.obs, trajectory.critic_obs, trajectory.actions, trajectory.logp,
                         trajectory.values, returns, normalized_adv, mask, trajectory.legal,
                         trajectory.teacher_actions, trajectory.teacher_mask)
        batch = jax.tree_util.tree_map(lambda a: a.reshape((total,) + a.shape[2:]), batch)
        metrics = dict(zip(_ROLLOUT_METRIC_NAMES, jnp.sum(metric_rows, axis=0)))
        metrics.update(policy_samples=jnp.sum(mask), learner_fraction=jnp.mean(mask),
                       reward_mean=jnp.mean(trajectory.rewards),
                       raw_adv_std=jnp.sqrt(var_adv),
                       explained_variance=1. - jnp.var(returns - trajectory.values) /
                                           jnp.maximum(jnp.var(returns), 1e-8),
                       return_mean=jnp.mean(returns), value_mean=jnp.mean(trajectory.values))
        return state, batch, metrics

    _loss_names = ("loss", "policy_loss", "value_loss", "entropy", "approx_kl",
                   "clip_fraction", "bidding_anchor_loss", "play_anchor_loss",
                   "teacher_accuracy_bid", "teacher_accuracy_play", "grad_norm")

    def loss_fn(params, batch):
        logits, values = model.apply({"params": params}, batch.obs, batch.critic_obs)
        logs = _masked_log_probs(logits, batch.legal)
        chosen = jnp.take_along_axis(logs, batch.actions[:, None], axis=-1)[:, 0]
        log_ratio = chosen - batch.logp
        # Guard exponent overflow; ordinary PPO ratios are nowhere near these bounds.
        ratio = jnp.exp(jnp.clip(log_ratio, -20., 20.))
        unclipped = ratio * batch.advantages
        clipped = jnp.clip(ratio, 1. - cfg["clip_eps"], 1. + cfg["clip_eps"]) * batch.advantages
        policy_loss = -_masked_mean(jnp.minimum(unclipped, clipped), batch.policy_mask)
        clipped_values = batch.old_values + jnp.clip(values - batch.old_values,
                                                      -cfg["value_clip"], cfg["value_clip"])
        value_loss = .5 * jnp.mean(jnp.maximum((values - batch.returns) ** 2,
                                               (clipped_values - batch.returns) ** 2))
        entropy = _masked_mean(-jnp.sum(jnp.exp(logs) * logs, axis=-1), batch.policy_mask)
        approx_kl = _masked_mean((ratio - 1.) - log_ratio, batch.policy_mask)
        clip_fraction = _masked_mean((jnp.abs(ratio - 1.) > cfg["clip_eps"]).astype(jnp.float32),
                                     batch.policy_mask)
        teacher_logp = jnp.take_along_axis(logs, batch.teacher_actions[:, None], axis=-1)[:, 0]
        bidding_mask = batch.teacher_mask * batch.policy_mask
        card_mask = (batch.actions < 52) * batch.policy_mask
        bidding_anchor_loss = -_masked_mean(teacher_logp, bidding_mask)
        # When guidance is disabled collect deliberately skips generating labels.
        play_anchor_loss = (-_masked_mean(teacher_logp, card_mask)
                            if cfg['play_anchor_coef'] > 0 else jnp.float32(0.))
        agreement = (jnp.argmax(logs, axis=-1) == batch.teacher_actions).astype(jnp.float32)
        bid_accuracy = _masked_mean(agreement, bidding_mask)
        play_accuracy = (_masked_mean(agreement, card_mask)
                         if cfg['play_anchor_coef'] > 0 else jnp.float32(0.))
        loss = (policy_loss + cfg["value_coef"] * value_loss - cfg["entropy_coef"] * entropy
                + cfg['bidding_anchor_coef'] * bidding_anchor_loss
                + cfg['play_anchor_coef'] * play_anchor_loss)
        return loss, jnp.stack([loss, policy_loss, value_loss, entropy, approx_kl,
                               clip_fraction, bidding_anchor_loss, play_anchor_loss,
                               bid_accuracy, play_accuracy])

    @jax.jit
    def optimize(state, batch):
        def epoch(carry, unused):
            params, opt_state, key, stopped = carry
            key, k_shuffle = jax.random.split(key)
            permutation = jax.random.permutation(k_shuffle, total)
            minibatches = jax.tree_util.tree_map(
                lambda a: a[permutation].reshape((n_minibatches, cfg["minibatch_size"]) + a.shape[1:]), batch)

            def minibatch(carry, mini):
                params, opt_state, key, stopped = carry

                def update(args):
                    params, opt_state = args
                    (_, diagnostics), grads = jax.value_and_grad(loss_fn, has_aux=True)(params, mini)
                    grad_norm = optax.global_norm(grads)
                    updates, new_opt_state = tx.update(grads, opt_state, params)
                    new_params = optax.apply_updates(params, updates)
                    checks = _update_checks(diagnostics, grads, grad_norm, new_params, new_opt_state)
                    finite = jnp.all(checks)
                    params, opt_state = jax.lax.cond(finite,
                        lambda _: (new_params, new_opt_state), lambda _: (params, opt_state), operand=None)
                    # Stop subsequent minibatches after a large on-policy KL drift.
                    stop = ~finite | (diagnostics[4] > 2. * cfg["target_kl"])
                    row = (jnp.concatenate((diagnostics, grad_norm[None])),
                           jnp.array([1, finite.astype(jnp.int32)], jnp.int32), checks)
                    return (params, opt_state, key, stop), row

                return jax.lax.cond(stopped,
                    lambda _: ((params, opt_state, key, stopped),
                               (jnp.zeros(len(_loss_names)), jnp.zeros(2, jnp.int32),
                                jnp.ones(len(_UPDATE_CHECK_NAMES), jnp.bool_))),
                    update, (params, opt_state))

            carry, rows = jax.lax.scan(minibatch, (params, opt_state, key, stopped), minibatches)
            return carry, _reduce_update_rows(rows)

        (params, opt_state, key, stopped), rows = jax.lax.scan(
            epoch, (state.params, state.opt_state, state.rng, jnp.bool_(False)), None,
            length=cfg["ppo_epochs"])
        metrics = _update_metrics(rows, _loss_names)
        metrics["kl_early_stop"] = stopped & metrics["finite"]
        state = state._replace(params=params, opt_state=opt_state, rng=key,
                               updates=state.updates + 1)
        state = _refresh_snapshot(state, cfg)
        metrics.update(updates=state.updates,
                       snapshot_count=state.snapshot_count, snapshot_retiring=state.snapshot_retiring)
        return state, metrics

    def train_step(state):
        state, batch, rollout_metrics = collect(state)
        state, update_metrics = optimize(state, batch)
        metrics = dict(rollout_metrics)
        metrics.update(update_metrics)
        metrics["env_steps"] = state.env_steps
        matches = jnp.maximum(metrics["matches"], 1.)
        metrics["mean_final_score0"] = metrics.pop("final_score0_sum") / matches
        metrics["mean_final_score1"] = metrics.pop("final_score1_sum") / matches
        metrics["illegal_rate"] = metrics["illegal_actions"] / total
        metrics["hand_rate"] = metrics["hands"] / total
        metrics["match_rate"] = metrics["matches"] / total
        metrics["contract_success_rate"] = metrics["successful_contracts"] / jnp.maximum(2. * metrics["hands"], 1.)
        metrics['learner_contract_success_rate'] = metrics['learner_successful_contracts'] / jnp.maximum(metrics['learner_contracts'], 1.)
        metrics['learner_hand_score'] = metrics['learner_hand_score_sum'] / jnp.maximum(metrics['learner_contracts'], 1.)
        metrics['opponent_hand_score'] = metrics['opponent_hand_score_sum'] / jnp.maximum(metrics['opponent_contracts'], 1.)
        metrics['mean_normal_bid'] = metrics['learner_normal_bid_sum'] / jnp.maximum(metrics['learner_normal_bids'], 1.)
        metrics['nil_bid_rate'] = metrics['learner_nil_bids'] / jnp.maximum(metrics['learner_bid_seats'], 1.)
        metrics['dn_bid_rate'] = metrics['learner_dn_bids'] / jnp.maximum(metrics['learner_bid_seats'], 1.)
        metrics['learner_nil_success_rate'] = metrics['learner_nil_made'] / jnp.maximum(metrics['learner_nil_bids'], 1.)
        metrics['learner_dn_success_rate'] = metrics['learner_dn_made'] / jnp.maximum(metrics['learner_dn_bids'], 1.)
        metrics['hand_reward_mean_abs'] = metrics['hand_reward_abs_sum'] / jnp.maximum(metrics['hands'], 1.)
        metrics['score_bonus_mean_abs'] = metrics['score_bonus_abs_sum'] / jnp.maximum(metrics['hands'], 1.)
        for ph in ('blind', 'bid', 'play'):
            metrics['entropy_' + ph] = metrics['entropy_' + ph + '_sum'] / jnp.maximum(metrics['entropy_' + ph + '_count'], 1.)
        metrics.update(completed_matches=metrics['matches'],
                       mean_match_hands=jnp.mean(state.envs.hand_number.astype(jnp.float32)),
                       max_match_hands=jnp.max(state.envs.hand_number),
                       mean_score0=jnp.mean(state.envs.scores[:, 0].astype(jnp.float32)),
                       mean_score1=jnp.mean(state.envs.scores[:, 1].astype(jnp.float32)),
                       score_gap_saturated_fraction=jnp.mean((jnp.abs(
                           state.envs.scores[:, 0] - state.envs.scores[:, 1]) >= 1500).astype(jnp.float32)))
        return state, metrics

    # Diagnostic hooks also let the notebook's optional verification exercise
    # the actual loss/rollout rather than a second implementation of PPO.
    train_step.collect = collect
    train_step.optimize = optimize
    train_step.loss_fn = loss_fn
    return train_step


def make_bootstrap_step(cfg):
    """Coherent teacher initialization before PPO and on-policy guidance.

    Each call gathers one fresh rollout of scripted games and optimizes actor
    cross-entropy. Teachers are imperfect; their advice is a training aid.
    Call finish_bootstrap once afterward to start clean self-play matches.
    """
    cfg = learning_config(cfg)
    model, tx = model_for_config(cfg), _optimizer(cfg)
    n, horizon = cfg["n_env"], cfg["rollout_len"]
    total = n * horizon
    teacher_minibatch = min(cfg['bootstrap_minibatch_size'], total)
    nb = total // teacher_minibatch

    @jax.jit
    def collect(state):
        def teacher_turn(carry, unused):
            envs, key = carry
            key, k_action, k_step = jax.random.split(key, 3)
            obs, legal = jax.vmap(observe)(envs), jax.vmap(legal_mask)(envs)
            labels = jax.vmap(lambda env, key: teacher_action(env, key, cfg))(
                envs, jax.random.split(k_action, n))
            envs, _, _, _ = batch_step(envs, labels, jax.random.split(k_step, n))
            return (envs, key), (obs, legal, labels)

        (envs, key), data = jax.lax.scan(teacher_turn, (state.envs, state.rng), None, length=horizon)
        data = jax.tree_util.tree_map(lambda a: a.reshape((total,) + a.shape[2:]), data)
        return state._replace(envs=envs, rng=key), data

    @jax.jit
    def optimize(state, data):
        def supervised_loss(params, mini):
            obs, legal, labels = mini
            logits = model.apply({"params": params}, obs, method=model.actor)
            logs = _masked_log_probs(logits, legal)
            selected = jnp.take_along_axis(logs, labels[:, None], axis=-1)[:, 0]
            mask = (jnp.sum(legal, axis=-1) > 1).astype(jnp.float32)
            loss = -_masked_mean(selected, mask)
            accuracy = _masked_mean((jnp.argmax(logs, axis=-1) == labels).astype(jnp.float32), mask)
            return loss, accuracy

        def epoch(carry, unused):
            params, opt_state, key, stopped = carry
            key, k_shuffle = jax.random.split(key)
            permutation = jax.random.permutation(k_shuffle, total)
            minis = jax.tree_util.tree_map(
                lambda a: a[permutation].reshape((nb, teacher_minibatch) + a.shape[1:]), data)

            def one(carry, mini):
                params, opt_state, stopped = carry

                def update(args):
                    params, opt_state = args
                    (loss, accuracy), grads = jax.value_and_grad(supervised_loss, has_aux=True)(params, mini)
                    grad_norm = optax.global_norm(grads)
                    updates, candidate_opt = tx.update(grads, opt_state, params)
                    candidate = optax.apply_updates(params, updates)
                    diagnostics = jnp.array([loss, accuracy, grad_norm])
                    checks = _update_checks(diagnostics, grads, grad_norm, candidate, candidate_opt)
                    finite = jnp.all(checks)
                    params, opt_state = jax.lax.cond(finite, lambda _: (candidate, candidate_opt),
                                                    lambda _: (params, opt_state), operand=None)
                    return (params, opt_state, ~finite), (diagnostics,
                        jnp.array([1, finite.astype(jnp.int32)], jnp.int32), checks)

                return jax.lax.cond(stopped,
                    lambda _: ((params, opt_state, stopped),
                               (jnp.zeros(3), jnp.zeros(2, jnp.int32),
                                jnp.ones(len(_UPDATE_CHECK_NAMES), jnp.bool_))),
                    update, (params, opt_state))

            (params, opt_state, stopped), rows = jax.lax.scan(one, (params, opt_state, stopped), minis)
            return (params, opt_state, key, stopped), _reduce_update_rows(rows)

        (params, opt_state, key, _), rows = jax.lax.scan(
            epoch, (state.params, state.opt_state, state.rng, jnp.bool_(False)),
            None, length=cfg["bootstrap_epochs"])
        metrics = _update_metrics(rows, ("bootstrap_loss", "bootstrap_accuracy", "bootstrap_grad_norm"))
        metrics["bootstrap_updates"] = state.bootstrap_updates + 1
        return state._replace(params=params, opt_state=opt_state, rng=key,
                              bootstrap_updates=state.bootstrap_updates + 1), metrics

    def bootstrap_step(state):
        state, data = collect(state)
        return optimize(state, data)

    bootstrap_step.collect = collect
    bootstrap_step.optimize = optimize
    return bootstrap_step


def finish_bootstrap(state, cfg):
    """Commit warmed weights to the league and reset games/optimizer for PPO."""
    cfg = learning_config(cfg)
    key, k_env, k_league = jax.random.split(state.rng, 3)
    kind, style, slot, team = _league_assignments(k_league, cfg, jnp.int32(1), jnp.int32(-1))
    bank = jax.tree_util.tree_map(lambda p: jnp.repeat(p[None], cfg["snapshot_size"], axis=0), state.params)
    return state._replace(opt_state=_optimizer(cfg).init(state.params), rng=key,
                          envs=jax.vmap(reset)(jax.random.split(k_env, cfg["n_env"])),
                          league_kind=kind, league_style=style, league_snapshot=slot, learner_team=team,
                          snapshot_params=bank, snapshot_count=jnp.int32(1),
                          snapshot_cursor=jnp.int32(1 % cfg["snapshot_size"]),
                          snapshot_retiring=jnp.int32(-1), last_snapshot_update=state.updates,
                          bootstrap_complete=jnp.bool_(True))


def recover_legacy_training(state, cfg):
    """Retain the policy/counters and prepare an explicit trainer migration.

    Reset the critic fitted to pathological trajectories, Adam moments, deeply
    negative games, and stale league bank. Actor parameter tensors transfer
    exactly before the teacher refresh; extreme-score input encoding changes.
    The caller records this change and saves into a separate recovery folder.
    """
    cfg = learning_config(cfg)
    if int(state.decisions_per_update) != cfg['n_env'] * cfg['rollout_len']:
        raise ValueError('Recovery must keep the saved environment/rollout configuration.')
    fresh = initialize_training(cfg)
    old_actor = jax.tree_util.tree_leaves(state.params['actor_network'])
    new_actor = jax.tree_util.tree_leaves(fresh.params['actor_network'])
    if len(old_actor) != len(new_actor) or any(a.shape != b.shape for a, b in zip(old_actor, new_actor)):
        raise ValueError('Recovery must keep the saved model architecture.')
    params = dict(fresh.params)
    params['actor_network'] = state.params['actor_network']
    fresh = fresh._replace(params=params, rng=state.rng, updates=state.updates,
                            decisions_per_update=state.decisions_per_update,
                            env_steps_low=state.env_steps_low, env_steps_high=state.env_steps_high,
                            base_updates=state.base_updates)
    # This creates fresh games and snapshots from the transferred policy with
    # a fresh critic, and derives their random keys from the saved RNG stream.
    fresh = finish_bootstrap(fresh, cfg)
    return fresh._replace(bootstrap_updates=jnp.int32(0), bootstrap_complete=jnp.bool_(False))


def upgrade_training_workload(state, cfg, reset_critic=False):
    """Transfer learned weights into an explicit new batch/teacher experiment.

    Old files are never modified. Only the actor/critic parameters and random
    stream transfer; optimizer, games and league are initialized for the new
    workload. Lifetime decisions and update numbers remain exact after resizing.
    """
    cfg = learning_config(cfg)
    fresh = initialize_training(cfg)
    for name in ('actor_network', 'critic_network'):
        old, old_structure = jax.tree_util.tree_flatten(state.params[name])
        new, new_structure = jax.tree_util.tree_flatten(fresh.params[name])
        if old_structure != new_structure or any(a.shape != b.shape for a, b in zip(old, new)):
            raise ValueError('Workload migration must preserve the saved network width and architecture.')
    params = dict(state.params)
    if reset_critic:
        params['critic_network'] = fresh.params['critic_network']
    decisions = state.env_steps
    if not 0 <= decisions < 2 ** 64:
        raise ValueError('Lifetime decisions exceed the supported unsigned 64-bit counter.')
    fresh = fresh._replace(params=params, rng=state.rng, updates=state.updates,
                          env_steps_low=jnp.uint32(decisions & 0xffffffff),
                          env_steps_high=jnp.uint32(decisions >> 32), base_updates=state.updates)
    fresh = finish_bootstrap(fresh, cfg)
    return fresh._replace(bootstrap_updates=jnp.int32(0), bootstrap_complete=jnp.bool_(False))


## 5b · GPU memory profiler

These helpers profile the actual compiled rollout, PPO optimizer, and separate teacher rollout/optimizer buffers. They include the retained state needed for rollback, apply a safety factor, and respect physical and allocator limits. They run when configuration requests a new workload; exact-resume shapes remain unchanged.

`GPU_TUNING['target_gib']=14.0` requests 14 GiB. The live dashboard will distinguish estimated working memory, active arrays, allocator reservations, and total board usage.
GPU admissions require current live/pool/allocator evidence and retain at least1.45×estimate+256MiB. Exact resume characterizes only its saved shape. A compiler or admission failure stops without automatic retry.


In [ ]:
"""Size real rollout/optimizer workloads for available GPU memory.

No dummy allocations or GPU-memory preallocation are used. Compilation is done
with abstract arrays, so an oversized candidate is rejected before its rollout
buffers are allocated. Exact checkpoint resumes retain their saved shapes.
"""
import csv
import gc
import math
import numbers
import os
import subprocess

GPU_TUNING = dict(enabled=True, target_gib=14.0, reserve_mib=1024,
                  safety_factor=1.45, extra_guard_mib=256, initial_envs=256, env_quantum=128,
                  maximum_envs=16384, maximum_profiles=3,
                  minibatches_per_epoch=8)
_GPU_TUNED_STEP = None
_GPU_TUNED_CONFIG = None
_GPU_PROFILE_REPORT = None
_GPU_LAST_ADMISSION = None
_GPU_TUNED_RUNTIME = None
RUNTIME_SAFETY_REVISION = "checkpoint-memory-safety-v2-kaggle-uuid-v1"


_GPU_COUNTER_FIELDS = (
    ('bytes_in_use', 'jax_live_bytes'),
    ('peak_bytes_in_use', 'jax_peak_live_bytes'),
    ('pool_bytes', 'jax_pool_bytes'),
    ('peak_pool_bytes', 'jax_peak_pool_bytes'),
    ('bytes_reserved', 'jax_reserved_bytes'),
    ('peak_bytes_reserved', 'jax_peak_reserved_bytes'),
    ('bytes_limit', 'jax_limit_bytes'),
    ('largest_free_block_bytes', 'jax_largest_free_block_bytes'),
)


def _gpu_counter_record(value):
    """Retain raw values in JSON-safe form without silently fixing counters."""
    if isinstance(value, numbers.Integral) and not isinstance(value, bool):
        return int(value)
    if value is None or isinstance(value, (str, bool)):
        return value
    if isinstance(value, numbers.Real):
        value = float(value)
        return value if math.isfinite(value) else repr(value)
    return repr(value)


def _gpu_snapshot_allocator_fields(stats):
    """Pure mapper for independently testable telemetry preservation."""
    if not isinstance(stats, dict):
        return dict(jax_memory_stats_raw={}, jax_memory_stats_error='not_a_dictionary')
    report = {'jax_memory_stats_raw': {
        str(key): _gpu_counter_record(value) for key, value in stats.items()}}
    for source, target in _GPU_COUNTER_FIELDS:
        if source in stats:
            # Retaining invalid input allows the budget helper to fail closed;
            # int(nan), int(inf), and string-to-int coercions are not attempted.
            report[target] = _gpu_counter_record(stats[source])
    return report


def _gpu_memory_metric_values(snapshot):
    """Display only valid byte counters; raw malformed telemetry stays in reports."""
    result = {}
    for name, value in snapshot.items():
        if (not name.endswith('_bytes') or isinstance(value, bool)
                or not isinstance(value, numbers.Integral) or value < 0):
            continue
        try:
            gib = value / 2**30
        except OverflowError:
            continue
        if math.isfinite(gib):
            result[name.removesuffix('_bytes') + '_gib'] = gib
    return result


def _gpu_selected_identity():
    """Selected physical board identity; logical device0 never means physical0."""
    platform = globals().get('_SPADES_KAGGLE_PLATFORM')
    if not isinstance(platform, dict):
        raise RuntimeError('Kaggle selected physical board metadata required.')
    selected = platform['selected_gpu_uuid']
    if os.environ.get('CUDA_VISIBLE_DEVICES') != selected:
        raise RuntimeError('Selected GPU visibility changed; no launch.')
    return (selected, platform['selected_gpu_pci_bus_id'],
            platform['selected_gpu_name'], platform['selected_gpu_total_bytes'],
            platform['driver_version'], os.environ.get('CUDA_VISIBLE_DEVICES'))


def _gpu_memory_snapshot(device=None):
    """UUID-bound one-board telemetry; all original allocator guards retained."""
    identity = _gpu_selected_identity()
    if device is None:
        devices = jax.devices()
        if len(devices) != 1 or devices[0].platform != 'gpu' or 'T4' not in str(devices[0].device_kind):
            raise RuntimeError('Exactly one JAX-visible selected T4 required.')
        device = devices[0]
    report = dict(platform=device.platform, selected_gpu_uuid=identity[0])
    if device.platform != 'gpu':
        return report
    try:
        stats = device.memory_stats() or {}
    except (AttributeError, RuntimeError) as exc:
        stats = {}
        report['jax_memory_stats_error'] = type(exc).__name__
    report.update(_gpu_snapshot_allocator_fields(stats))
    try:
        query = subprocess.run(
            ['nvidia-smi', '--id=' + identity[0],
             '--query-gpu=uuid,index,pci.bus_id,name,memory.total,memory.used,memory.free,driver_version',
             '--format=csv,noheader,nounits'], capture_output=True, text=True,
            check=False, timeout=5)
        rows = list(csv.reader(query.stdout.strip().splitlines()))
        if query.returncode != 0 or len(rows) != 1 or len(rows[0]) != 8:
            raise ValueError('Selected UUID board telemetry unavailable/ambiguous')
        uuid, index, pci, name, total, used, free, driver = [value.strip() for value in rows[0]]
        values = [float(value) for value in (total, used, free)]
        if (uuid != identity[0] or pci != identity[1] or name != identity[2] or driver != identity[4]
                or not index.isdecimal()
                or not all(math.isfinite(value) and value >= 0 and value.is_integer() for value in values)
                or int(values[0] * 2**20) != identity[3]
                or values[1] > values[0] or values[2] > values[0]
                or values[1] + values[2] > values[0]):
            raise ValueError('Selected UUID/PCI/name/driver/capacity identity changed or telemetry invalid')
        report.update(gpu_name=name, gpu_index=int(index), gpu_uuid=uuid, gpu_pci_bus_id=pci,
                      board_total_bytes=int(values[0] * 2**20),
                      board_used_bytes=int(values[1] * 2**20),
                      board_free_bytes=int(values[2] * 2**20))
    except (OSError, ValueError, OverflowError, subprocess.TimeoutExpired) as exc:
        report['selected_board_telemetry_error'] = type(exc).__name__ + ': ' + str(exc)
    return report


def _gpu_memory_budget_details(snapshot, options):
    """Additional allocation budget, its raw counter source, and any fallback.

    pool_bytes is the current BFC allocator pool total, including live bytes.
    bytes_reserved is accepted as a legacy total only when the caller explicitly
    declares that interpretation verified for its backend. The counters are
    never added together. Peak counters never provide current capacity.
    """
    target_value, reserve_value = options['target_gib'], options['reserve_mib']
    if any(isinstance(value, bool) or not isinstance(value, numbers.Real)
           for value in (target_value, reserve_value)):
        raise ValueError('GPU target and reserve must be finite numbers.')
    target, reserve = float(target_value) * 2**30, float(reserve_value) * 2**20
    if not (math.isfinite(target) and target > 0 and math.isfinite(reserve) and reserve >= 0):
        raise ValueError('GPU target and reserve must be finite; target must be positive.')
    legacy_verified = options.get('legacy_reserved_is_total_pool', False)
    if not isinstance(legacy_verified, bool):
        raise ValueError('Legacy reserved semantics must be explicitly True or False.')

    counters = {name: _gpu_counter_record(snapshot[name]) for name in (
        'board_total_bytes', 'board_used_bytes', 'board_free_bytes',
        'jax_live_bytes', 'jax_pool_bytes', 'jax_peak_pool_bytes',
        'jax_reserved_bytes', 'jax_peak_reserved_bytes', 'jax_peak_live_bytes',
        'jax_limit_bytes', 'jax_largest_free_block_bytes') if name in snapshot}
    result = dict(status='starting', available_new_working_bytes=None,
                  requested_target_bytes=int(target), board_reserve_bytes=int(reserve),
                  raw_counters=counters, pool_counter_source=None,
                  pool_counter_snapshot_field=None, pool_counter_value_bytes=None,
                  pool_counter_status='unavailable', reusable_idle_pool_bytes=0,
                  legacy_reserved_is_total_pool=legacy_verified,
                  overlap_credit_bytes=0, reasons=[])

    def invalid(reason):
        result.update(status='invalid_telemetry', available_new_working_bytes=0)
        result['reasons'].append(reason)
        return result

    def valid_bytes(value):
        return (not isinstance(value, bool) and isinstance(value, numbers.Integral)
                and value >= 0)

    for name in ('board_total_bytes', 'board_free_bytes'):
        if name not in snapshot:
            result.update(status='telemetry_unavailable')
            result['reasons'].append('missing_' + name)
            return result
    core = ('board_total_bytes', 'board_free_bytes', 'board_used_bytes',
            'jax_live_bytes', 'jax_limit_bytes')
    for name in core:
        if name in snapshot and not valid_bytes(snapshot[name]):
            return invalid('invalid_' + name)
    total, free = int(snapshot['board_total_bytes']), int(snapshot['board_free_bytes'])
    if total <= 0 or free > total:
        return invalid('invalid_board_total_or_free_bounds')
    if 'board_used_bytes' in snapshot:
        used = int(snapshot['board_used_bytes'])
        if used > total or used + free > total:
            return invalid('invalid_board_used_bounds')
        # Retain reserved/rounding discrepancy. It is never reusable capacity.
        result['board_total_minus_used_minus_free_bytes'] = total - used - free

    live_known = 'jax_live_bytes' in snapshot
    live = int(snapshot['jax_live_bytes']) if live_known else total - free
    if live > total - free:
        return invalid('live_exceeds_occupied_board_capacity')
    if not live_known:
        result['reasons'].append('missing_live_uses_occupied_board_upper_bound')
    result.update(live_bytes_deducted=live, live_counter_known=live_known)
    limit = snapshot.get('jax_limit_bytes')
    if limit is not None:
        limit = int(limit)
        if limit > total:
            return invalid('allocator_limit_exceeds_board_total')
        result['allocator_remaining_bytes'] = max(0, limit - live)
    else:
        result['reasons'].append('allocator_limit_unavailable')

    pool_field = None
    if 'jax_pool_bytes' in snapshot:
        pool_field = 'jax_pool_bytes'
        raw_source = 'pool_bytes'
    elif 'jax_reserved_bytes' in snapshot and legacy_verified:
        pool_field = 'jax_reserved_bytes'
        raw_source = 'bytes_reserved'
    elif 'jax_reserved_bytes' in snapshot:
        result['pool_counter_status'] = 'legacy_semantics_unverified'
        result['reasons'].append('bytes_reserved_is_not_assumed_to_be_a_total_pool')

    idle = 0
    if pool_field is not None:
        value = snapshot[pool_field]
        result.update(pool_counter_source=raw_source,
                      pool_counter_snapshot_field=pool_field,
                      pool_counter_value_bytes=_gpu_counter_record(value))
        reason = None
        if not live_known:
            reason = 'pool_credit_requires_known_live_bytes'
        elif not valid_bytes(value):
            reason = 'pool_counter_is_not_nonnegative_integer_bytes'
        elif value < live:
            reason = 'pool_counter_is_below_live_bytes'
        elif value > total - free:
            reason = 'pool_counter_exceeds_occupied_board_capacity'
        elif limit is not None and value > limit:
            reason = 'pool_counter_exceeds_allocator_limit'
        if reason:
            result['pool_counter_status'] = 'unusable_zero_credit'
            result['reasons'].append(reason)
        else:
            idle = int(value) - live
            result['pool_counter_status'] = 'validated_current_total'
    else:
        result['reasons'].append('no_validated_current_pool_counter_zero_credit')

    physical = max(0, min(total - live, free + idle) - reserve)
    limits = [int(target), int(physical)]
    if limit is not None:
        limits.append(limit - live)
    result.update(status='ok', reusable_idle_pool_bytes=idle,
                  physical_remaining_after_reserve_bytes=int(physical),
                  available_new_working_bytes=max(0, int(min(limits))))
    if result['available_new_working_bytes'] == 0:
        result['status'] = 'no_capacity'
    return result


def _gpu_working_budget(snapshot, options):
    """Compatible scalar API; detailed decisions remain separately inspectable."""
    return _gpu_memory_budget_details(snapshot, options)['available_new_working_bytes']


def _gpu_candidate_config(base_cfg, n_env, options):
    cfg = dict(base_cfg)
    n_env = int(n_env)
    cfg['n_env'] = n_env
    total = n_env * int(cfg['rollout_len'])
    # Preserve PPO optimizer cadence while scaling data parallelism. A fixed
    # small minibatch on a huge rollout would perform hundreds of sequential
    # Adam steps, trigger KL early stopping, and waste most sampled decisions.
    minibatches = int(options.get('minibatches_per_epoch', 8))
    if minibatches < 1 or total % minibatches:
        raise ValueError('GPU candidate decisions must divide into whole minibatches.')
    cfg['minibatch_size'] = total // minibatches
    return learning_config(cfg)


def _gpu_compiled_bytes(analysis):
    """XLA executable buffer accounting; aliases occupy memory only once."""
    names = ('argument_size_in_bytes', 'output_size_in_bytes',
             'temp_size_in_bytes', 'alias_size_in_bytes')
    if analysis is None or any(getattr(analysis, name, None) is None for name in names):
        raise RuntimeError('This JAX backend does not expose executable memory accounting.')
    values = {name: int(getattr(analysis, name)) for name in names}
    if any(value < 0 for value in values.values()):
        raise RuntimeError('The compiler returned invalid memory accounting.')
    values['total_bytes'] = max(0, values['argument_size_in_bytes']
        + values['output_size_in_bytes'] + values['temp_size_in_bytes']
        - values['alias_size_in_bytes'])
    return values


def _gpu_tree_bytes(tree):
    return sum(math.prod(x.shape) * np.dtype(x.dtype).itemsize
               for x in jax.tree_util.tree_leaves(tree))


def _gpu_profile_shape(cfg, options=None):
    """Compile the actual code, including snapshots, finite guards and shuffle."""
    opts = dict(GPU_TUNING) if options is None else dict(options)
    profile_admissions = []
    step = make_train_step(cfg)
    abstract_state = jax.eval_shape(lambda: initialize_training(cfg))
    state_bytes = _gpu_tree_bytes(abstract_state)
    profile_admissions.append(_gpu_memory_admission(state_bytes, opts, "before collect profile"))
    after_collect, abstract_batch, _ = jax.eval_shape(step.collect, abstract_state)
    collect_executable = step.collect.lower(abstract_state).compile()
    collect_bytes = _gpu_compiled_bytes(collect_executable.memory_analysis())
    profile_admissions.append(_gpu_memory_admission(collect_bytes["total_bytes"] + state_bytes, opts, "before optimize profile"))
    optimize_executable = step.optimize.lower(after_collect, abstract_batch).compile()
    optimize_bytes = _gpu_compiled_bytes(optimize_executable.memory_analysis())
    # Match the actual two-stage supervised runtime. Jitting the whole Python
    # wrapper could fuse away buffers that remain live between the real collect
    # and optimize calls, underestimating the teacher initialization peak.
    bootstrap = make_bootstrap_step(cfg)
    after_teacher, abstract_teacher_data = jax.eval_shape(bootstrap.collect, abstract_state)
    profile_admissions.append(_gpu_memory_admission(max(collect_bytes["total_bytes"], optimize_bytes["total_bytes"]) + state_bytes, opts, "before teacher collect profile"))
    teacher_collect_executable = bootstrap.collect.lower(abstract_state).compile()
    teacher_collect_bytes = _gpu_compiled_bytes(teacher_collect_executable.memory_analysis())
    profile_admissions.append(_gpu_memory_admission(max(collect_bytes["total_bytes"], optimize_bytes["total_bytes"], teacher_collect_bytes["total_bytes"]) + state_bytes, opts, "before teacher optimize profile"))
    teacher_optimize_executable = bootstrap.optimize.lower(after_teacher, abstract_teacher_data).compile()
    teacher_optimize_bytes = _gpu_compiled_bytes(teacher_optimize_executable.memory_analysis())
    state_bytes = _gpu_tree_bytes(abstract_state)
    # The caller retains the last completed state for rollback throughout the
    # entire update. Optimize also receives the post-rollout state and batch:
    # those buffers, shuffled copies, gradients, Adam and finite checks are
    # already in optimize's executable accounting. This extra full state is
    # conservative because some unchanged leaves alias between both states.
    peak = max(collect_bytes['total_bytes'], optimize_bytes['total_bytes'],
               teacher_collect_bytes['total_bytes'], teacher_optimize_bytes['total_bytes']) + state_bytes
    report = dict(n_env=cfg['n_env'], rollout_len=cfg['rollout_len'],
                  minibatch_size=cfg['minibatch_size'], width=cfg['width'],
                  state_bytes=state_bytes, profile_admissions=profile_admissions,
                  rollout_batch_bytes=_gpu_tree_bytes(abstract_batch),
                  collect=collect_bytes, optimize=optimize_bytes,
                  bootstrap_collect=teacher_collect_bytes, bootstrap_optimize=teacher_optimize_bytes,
                  estimated_peak_bytes=int(peak))
    # Keep the selected jitted functions alive to reuse their compilation. The
    # standalone bootstrap executables are used only for memory estimation.
    step.bootstrap = bootstrap
    del collect_executable, optimize_executable, teacher_collect_executable, teacher_optimize_executable, bootstrap
    gc.collect()
    return report, step


def _gpu_guarded_bytes(estimated_bytes, options):
    """The measured calibration is a floor at every allocation admission."""
    if (isinstance(estimated_bytes, bool) or not isinstance(estimated_bytes, numbers.Integral)
            or estimated_bytes <= 0):
        raise ValueError('The compiler/state estimate must be positive integer bytes.')
    safety, guard = options['safety_factor'], options.get('extra_guard_mib', 256)
    if any(isinstance(x, bool) or not isinstance(x, numbers.Real) for x in (safety, guard)):
        raise ValueError('GPU safety factor and guard must be finite numbers.')
    if not (math.isfinite(safety) and safety >= 1.45
            and math.isfinite(guard) and guard >= 256):
        raise ValueError('GPU admissions require at least 1.45 times the estimate plus 256 MiB.')
    return math.ceil(int(estimated_bytes) * float(safety)) + math.ceil(float(guard) * 2**20)


def _gpu_runtime_identity():
    """Live function objects, stack and revision must still match the profile."""
    functions = tuple(globals().get(name) for name in (
        'make_train_step', 'make_bootstrap_step', 'initialize_training', 'finish_bootstrap'))
    versions = tuple(getattr(globals().get(name), '__version__', None)
                     for name in ('jax', 'flax', 'optax'))
    return functions + versions + (globals().get('TRAINING_REVISION'), RUNTIME_SAFETY_REVISION) + _gpu_selected_identity()


def _gpu_runtime_summary():
    """Serializable diagnostics; live object equality is checked separately."""
    return dict(function_ids={name: id(globals().get(name)) for name in (
        'make_train_step', 'make_bootstrap_step', 'initialize_training', 'finish_bootstrap')},
        stack={name: getattr(globals().get(name), '__version__', None)
               for name in ('jax', 'flax', 'optax')},
        training_revision=globals().get('TRAINING_REVISION'),
        runtime_safety_revision=RUNTIME_SAFETY_REVISION)


def _gpu_current_budget_details(snapshot, options):
    """Production admissions require current live and allocator-ceiling evidence.

    The pure accounting helper retains conservative compatibility fallbacks for
    diagnostics. Production refuses to train or allocate without these counters.
    """
    details = _gpu_memory_budget_details(snapshot, options)
    missing = [name for name in ('board_total_bytes', 'board_free_bytes',
               'jax_live_bytes', 'jax_limit_bytes') if name not in snapshot]
    details['admission_missing_required_fields'] = missing
    if missing:
        details.update(status='telemetry_unavailable', available_new_working_bytes=0)
        details['reasons'].append('production_requires_current_board_live_and_allocator_limit')
    return details


def _gpu_memory_admission(estimated_bytes, options, stage, *, snapshot=None):
    """Refresh board/pool capacity; give no credit for live input overlap."""
    global _GPU_LAST_ADMISSION
    guarded = _gpu_guarded_bytes(estimated_bytes, options)
    snapshot = _gpu_memory_snapshot() if snapshot is None else snapshot
    report = dict(stage=str(stage), estimated_peak_bytes=int(estimated_bytes),
                  guarded_peak_bytes=guarded, snapshot=snapshot,
                  overlap_credit_bytes=0)
    if snapshot.get('platform') != 'gpu':
        report['status'] = 'no_gpu_unchanged'
        _GPU_LAST_ADMISSION = report
        return report
    details = _gpu_current_budget_details(snapshot, options)
    budget = details['available_new_working_bytes']
    report.update(budget_details=details, budget_bytes=budget)
    report['status'] = ('admitted' if details['status'] == 'ok'
                        and budget is not None and guarded <= budget else 'rejected')
    _GPU_LAST_ADMISSION = report
    if report['status'] != 'admitted':
        raise RuntimeError(f'GPU memory admission rejected before {stage}: '
                           f'{guarded} guarded bytes, {budget} available; '
                           f'{details["status"]}. Last accepted state is retained; '
                           'inspect _GPU_LAST_ADMISSION before any retry.')
    return report


def _gpu_choose_configuration(base_cfg, budget_bytes, options, profiler, budget_provider=None):
    """Bounded shape search. Compiler errors stop immediately, without retry.

    A capacity gate raised during partial compilation also stops the entire
    search, even if an earlier fully compiled candidate fit. This conservative
    fail-fast policy avoids continuing after any admission refusal.
    """
    quantum, max_env = int(options['env_quantum']), int(options['maximum_envs'])
    attempts = int(options['maximum_profiles'])
    _gpu_guarded_bytes(1, options)
    if quantum < 1 or max_env < quantum or attempts < 1:
        raise ValueError('Invalid GPU profiling limits.')
    minimum = min(quantum, max_env)
    n_env = max(minimum, min(max_env, int(options['initial_envs']) // quantum * quantum))
    candidates, tried, best = [], set(), None
    upper_failure = max_env + quantum
    for _ in range(attempts):
        if n_env in tried:
            break
        tried.add(n_env)
        cfg = _gpu_candidate_config(base_cfg, n_env, options)
        # Do not interpret compiler RuntimeError/MemoryError as a size signal.
        # No next shape or live state operation runs after either failure.
        profile, handle = profiler(cfg)
        estimate = profile['estimated_peak_bytes']
        guarded = _gpu_guarded_bytes(estimate, options)
        if budget_provider is not None:
            budget_bytes = budget_provider()
        if (isinstance(budget_bytes, bool) or not isinstance(budget_bytes, numbers.Integral)
                or budget_bytes <= 0):
            raise RuntimeError('Fresh GPU memory telemetry has no usable capacity.')
        profile = dict(profile, guarded_peak_bytes=guarded,
                       admission_budget_bytes=int(budget_bytes),
                       fits_budget=guarded <= budget_bytes)
        candidates.append(profile)
        if guarded <= budget_bytes:
            if best is None or n_env > best[0]['n_env']:
                best = (cfg, profile, handle)
            next_env = int(n_env * budget_bytes / guarded) // quantum * quantum
            next_env = min(max_env, upper_failure - quantum, max(n_env + quantum, next_env))
            if guarded >= .90 * budget_bytes or next_env <= n_env:
                break
        else:
            upper_failure = min(upper_failure, n_env)
            next_env = int(n_env * budget_bytes / guarded * .95) // quantum * quantum
            next_env = max(minimum, min(n_env - quantum, next_env))
        if next_env in tried or next_env < minimum:
            break
        n_env = next_env
    # An earlier accepted candidate may have lost capacity while later shapes
    # compiled. Its caller performs a separate fresh final admission.
    return best, candidates, []


def configure_gpu_workload(base_cfg, *, exact_resume=False, options=None):
    """Profile new shapes or characterize exactly the saved resume shape.

    Exact resume never changes model, optimizer, RNG, games, minibatches or
    counters. Its saved shape must pass characterization before an update.
    """
    global _GPU_TUNED_STEP, _GPU_TUNED_CONFIG, _GPU_PROFILE_REPORT, _GPU_TUNED_RUNTIME
    _GPU_TUNED_STEP = _GPU_TUNED_CONFIG = _GPU_PROFILE_REPORT = _GPU_TUNED_RUNTIME = None
    runtime_identity = _gpu_runtime_identity()
    cfg = learning_config(base_cfg)
    opts = dict(GPU_TUNING)
    if options:
        opts.update(options)
    _gpu_guarded_bytes(1, opts)
    snapshot = _gpu_memory_snapshot()
    report = dict(requested_gib=float(opts['target_gib']), snapshot=snapshot,
                  exact_resume=bool(exact_resume), selected_config=dict(cfg),
                  options=dict(opts), profiles=[], failures=[], runtime_signature=_gpu_runtime_summary())
    _GPU_PROFILE_REPORT = report
    if snapshot.get('platform') != 'gpu':
        report['status'] = 'exact_resume_unchanged' if exact_resume else 'no_gpu_unchanged'
        print('GPU workload sizing skipped: this runtime has no JAX GPU.')
        return cfg, report
    initial_details = _gpu_current_budget_details(snapshot, opts)
    report['initial_budget_details'] = initial_details
    if (initial_details['status'] != 'ok'
            or not initial_details['available_new_working_bytes']):
        report['status'] = 'rejected'
        raise RuntimeError('GPU workload profiling requires valid current memory telemetry and capacity.')
    budget_snapshots = []
    def refreshed_budget():
        fresh = _gpu_memory_snapshot()
        details = _gpu_current_budget_details(fresh, opts)
        budget_snapshots.append(dict(snapshot=fresh, budget_details=details))
        if details['status'] != 'ok':
            raise RuntimeError('GPU memory telemetry invalid or unavailable after profiling.')
        return details['available_new_working_bytes']
    report['profile_budget_snapshots'] = budget_snapshots
    def profile(candidate):
        print(f'Compiling memory profile: {candidate["n_env"]} games × '
              f'{candidate["rollout_len"]} steps; minibatch {candidate["minibatch_size"]}.', flush=True)
        # Check a fresh capacity snapshot before compilation as well. The
        # compiler itself is abstract; a compile failure still stops this call.
        if not refreshed_budget():
            raise RuntimeError('No GPU memory capacity remains before compilation.')
        return _gpu_profile_shape(candidate, options=opts)
    try:
        if exact_resume or not opts['enabled']:
            selected, step = profile(cfg)
            selected = dict(selected,
                guarded_peak_bytes=_gpu_guarded_bytes(selected['estimated_peak_bytes'], opts),
                admission_budget_bytes=refreshed_budget())
            selected['fits_budget'] = selected['guarded_peak_bytes'] <= selected['admission_budget_bytes']
            report['profiles'] = [selected]
        else:
            best, profiles, failures = _gpu_choose_configuration(cfg,
                initial_details['available_new_working_bytes'], opts, profile,
                budget_provider=refreshed_budget)
            report.update(profiles=profiles, failures=failures)
            if best is None:
                raise RuntimeError('No profiled training shape fits the fresh guarded GPU memory budget.')
            cfg, selected, step = best
        report.update(selected_config=dict(cfg), selected_profile=selected)
        final = _gpu_memory_admission(selected['estimated_peak_bytes'], opts,
            'selected exact resume' if exact_resume else 'selected workload')
        if runtime_identity != _gpu_runtime_identity():
            raise RuntimeError('Trainer definitions, runtime revision or library stack changed during profiling.')
        report.update(status='exact_resume_profiled_unchanged' if exact_resume else 'profiled',
                      final_admission=final)
    except (RuntimeError, MemoryError) as exc:
        report.update(status='rejected', error=f'{type(exc).__name__}: {exc}')
        _GPU_TUNED_STEP = _GPU_TUNED_CONFIG = None
        raise
    _GPU_TUNED_CONFIG, _GPU_TUNED_STEP = dict(cfg), step
    _GPU_TUNED_RUNTIME = runtime_identity
    print(f'GPU workload admitted: {cfg["n_env"]} environments, '
          f'{cfg["n_env"] * cfg["rollout_len"]:,} decisions/update; '
          f'{selected["guarded_peak_bytes"] / 2**30:.2f} GiB with calibration and guard.', flush=True)
    return cfg, report


def gpu_admit_workload(cfg, stage):
    """Require the current exact configuration's profile at each outer boundary."""
    snapshot = _gpu_memory_snapshot()
    if snapshot.get('platform') != 'gpu':
        return dict(stage=str(stage), status='no_gpu_unchanged')
    cfg = learning_config(cfg)
    report = _GPU_PROFILE_REPORT or {}
    if (report.get('status') not in ('profiled', 'exact_resume_profiled_unchanged')
            or dict(cfg) != report.get('selected_config')
            or _GPU_TUNED_RUNTIME != _gpu_runtime_identity()
            or 'selected_profile' not in report):
        raise RuntimeError(f'No matching admitted GPU memory profile before {stage}; '
                           'characterize this exact configuration before training.')
    return _gpu_memory_admission(report['selected_profile']['estimated_peak_bytes'],
        report['options'], stage, snapshot=snapshot)


def gpu_admit_checkpoint_load(cfg, state_bytes):
    """Bound array transfer before any saved leaf is placed on the GPU.

    This is a state-transfer admission, not a characterization of training or
    eager initialization. The loader uses an abstract template, avoiding the
    former extra real template allocation. Training receives its separate full
    exact-shape profile and stage checks after loading.
    """
    return _gpu_memory_admission(state_bytes, dict(GPU_TUNING), 'checkpoint array transfer')


def gpu_train_step(cfg):
    """Reuse profiled functions only when the complete configuration matches."""
    return (_GPU_TUNED_STEP if _GPU_TUNED_STEP is not None
            and dict(cfg) == _GPU_TUNED_CONFIG
            and _GPU_TUNED_RUNTIME == _gpu_runtime_identity() else make_train_step(cfg))


def gpu_bootstrap_step(cfg):
    """Reuse the profiled teacher collect/optimize objects for this exact config."""
    return (_GPU_TUNED_STEP.bootstrap if _GPU_TUNED_STEP is not None
            and dict(cfg) == _GPU_TUNED_CONFIG
            and _GPU_TUNED_RUNTIME == _gpu_runtime_identity()
            and hasattr(_GPU_TUNED_STEP, 'bootstrap') else make_bootstrap_step(cfg))


def gpu_memory_metrics():
    """Only valid finite scalar byte counters enter dashboard arithmetic."""
    snapshot = _gpu_memory_snapshot()
    result = _gpu_memory_metric_values(snapshot)
    if snapshot.get('platform') == 'gpu':
        result['gpu_target_gib'] = float((_GPU_PROFILE_REPORT or {}).get(
            'requested_gib', GPU_TUNING['target_gib']))
    if _GPU_PROFILE_REPORT and 'selected_profile' in _GPU_PROFILE_REPORT:
        result['gpu_estimated_working_gib'] = (
            _GPU_PROFILE_REPORT['selected_profile']['estimated_peak_bytes'] / 2**30)
    return result


## 6 · Full-match benchmarks and human table

Evaluation uses duplicate deals with partnership reversal. The per-hand shuffle key is independent of policy choices, making comparisons reproducible. Results include wins, losses, true ties, unfinished matches, score/contract statistics, Wilson intervals, and a bootstrap interval over complete deal pairs.

`max_hands` is an evaluation time limit, **not a rule of the game**. Capped matches are unresolved; they never count as wins or draws. Increase the cap if unresolved games are material. Small samples and all-win bootstrap intervals can be misleading; consult the Wilson interval and use more held-out pairs.

The periodic development benchmark uses a fixed seed against `adaptive`. The saved best-development model is selected by the lower Wilson bound only when every game resolves. Reserve new seeds and broader opponents for assessment after selecting a checkpoint.

In [ ]:
# Full-match evaluation and a private-information-safe human table.
# This cell only defines helpers; it does not start evaluation or interactive play.
from functools import lru_cache
import math
import time
import pandas as pd


def _style_id(style):
    if isinstance(style, str):
        if style not in HEURISTIC_NAMES:
            raise ValueError(f"Unknown opponent {style!r}; choose from {HEURISTIC_NAMES}")
        return HEURISTIC_NAMES.index(style)
    number = int(style)
    if not 0 <= number < len(HEURISTIC_NAMES):
        raise ValueError(f"Opponent index must be in [0, {len(HEURISTIC_NAMES) - 1}]")
    return number


def _wilson_interval(successes, count, z=1.959963984540054):
    """Wilson interval; fractional successes give draws half credit."""
    if count == 0:
        return float('nan'), float('nan')
    p = successes / count
    denominator = 1 + z * z / count
    center = (p + z * z / (2 * count)) / denominator
    half = z * math.sqrt(p * (1 - p) / count + z * z / (4 * count * count)) / denominator
    return max(0., center - half), min(1., center + half)


def _paired_interval(outcomes, seed):
    """Percentile bootstrap over complete deal pairs, preserving pair dependence."""
    pair_outcomes = np.asarray(outcomes, dtype=float).reshape(-1, 2)
    complete = np.isfinite(pair_outcomes).all(axis=1)
    pair_means = pair_outcomes[complete].mean(axis=1)
    if len(pair_means) < 2:
        return float('nan'), float('nan'), int(len(pair_means))
    rng = np.random.default_rng(seed)
    # An all-win sample cannot establish certainty; also consult Wilson and sample size.
    sampled = rng.choice(pair_means, size=(4000, len(pair_means)), replace=True).mean(axis=1)
    low, high = np.quantile(sampled, [.025, .975])
    return float(low), float(high), int(len(pair_means))


@lru_cache(maxsize=12)
def _evaluation_kernel(width, snapshot_opponent, deterministic):
    """One cached executable per architecture/opponent kind/action convention."""
    inference_cfg = {'width': width}

    def choose(logits, masks, keys):
        logits = jnp.where(masks, logits, -jnp.inf)
        if deterministic:
            return jnp.argmax(logits, axis=-1).astype(jnp.int32)
        return jax.vmap(jax.random.categorical)(keys, logits).astype(jnp.int32)

    @jax.jit
    def run(params, other_params, pair_keys, learner_teams, style, max_hands):
        count = learner_teams.shape[0]
        reset_keys = jax.vmap(lambda k: jax.random.fold_in(k, 0))(pair_keys)
        states = jax.vmap(reset)(reset_keys)
        # The hand clock is independent of bidding, trick winners, and policy randomness.
        # Both sides of a duplicate pair get the same reset and every subsequent deal key.
        carry = (
            states,
            jnp.zeros(count, bool),                    # completed matches
            jnp.full(count, -1, jnp.int32),             # true engine winners only
            jnp.zeros(count, jnp.int32),               # played hands
            jnp.zeros((count, 2), jnp.int32),           # last scores before auto-reset
            jnp.zeros(count, jnp.int32),               # successful learner contracts
            jnp.zeros(count, jnp.int32),               # all illegal actions (must be 0)
            jnp.zeros(count, jnp.float32),             # learner hand score sum
            jnp.array(0, jnp.int32),                   # synchronized hand index
        )

        def condition(c):
            return (c[-1] < max_hands) & (~jnp.all(c[1]))

        def hand(c):
            state, completed, winners, hands, scores, made, illegal, score_sum, h = c
            active = ~completed
            deal_keys = jax.vmap(lambda k: jax.random.fold_in(k, h + 1))(pair_keys)

            def turn(s, t):
                action_keys = jax.vmap(
                    lambda k: jax.random.fold_in(k, 1_000_000 + h * 64 + t)
                )(pair_keys)
                observations = jax.vmap(observe)(s)
                masks = jax.vmap(legal_mask)(s)
                players = jax.vmap(actor)(s)
                learner_actions = choose(policy_logits(params, observations, inference_cfg), masks, action_keys)
                opponent_keys = jax.vmap(lambda k: jax.random.fold_in(k, 1))(action_keys)
                if snapshot_opponent:
                    opponent_actions = choose(
                        policy_logits(other_params, observations, inference_cfg), masks, opponent_keys
                    )
                else:
                    opponent_actions = jax.vmap(heuristic_action, in_axes=(0, 0, None))(
                        s, opponent_keys, style
                    )
                actions = jnp.where(players % 2 == learner_teams, learner_actions, opponent_actions)
                # Keep every member on the same scheduled turn so the engine can
                # branch once per batch. Completed matches may play ignored extra
                # hands; active masks below freeze all recorded results and metrics.
                nxt, _, _, info = batch_step(s, actions, deal_keys)
                compact = (info['match_done'], info['winner'], info['final_scores'],
                           info['contract_success'], info['illegal'], info['score_delta'])
                return nxt, compact

            state, info = jax.lax.scan(turn, state, jnp.arange(64, dtype=jnp.int32))
            ended = info[0][-1] & active
            winners = jnp.where(ended, info[1][-1], winners)
            scores = jnp.where(active[:, None], info[2][-1], scores)
            row = jnp.arange(count)
            made += (info[3][-1][row, learner_teams] & active).astype(jnp.int32)
            illegal += jnp.sum(info[4].astype(jnp.int32), axis=0) * active
            score_sum += jnp.where(active, info[5][-1][row, learner_teams], 0)
            hands += active.astype(jnp.int32)
            return state, completed | ended, winners, hands, scores, made, illegal, score_sum, h + 1

        result = jax.lax.while_loop(condition, hand, carry)
        return result[1:8]
    return run


def evaluate_policy(params, cfg, styles=None, games_per_style=32, seed=20261014,
                    max_hands=200, snapshot_opponents=None, deterministic=True,
                    progress=True):
    """Play complete partnership matches against bots and/or frozen policy snapshots.

    Each opponent gets games_per_style matches (an even number): identical deals with
    the learner controlling seats 0+2, then seats 1+3. Both learner partners share the
    policy. Deals for hand h are keyed by h, never by the trajectory's RNG consumption.
    styles=None selects all bots; styles=[] selects only supplied snapshot_opponents.
    snapshot_opponents is {label: bare_flax_params}, using the same model width.

    max_hands is an evaluation resource limit, NOT a game rule. Unfinished matches
    remain unresolved, are excluded from win rates/CIs, and are never treated as wins
    or true draws. Their fraction and score margins are reported separately. Increasing
    the cap is essential when unresolved_fraction is material; conditioning on resolved
    games can bias comparisons. Genuine engine ties receive half credit.

    Wilson CIs are approximate (paired deals and half-credit draws). paired_ci_* uses a
    deal-pair bootstrap, excludes incomplete pairs, and can degenerate at all wins/losses.
    Use hundreds of pairs and multiple unseen seeds before making strength claims.
    The executable is compiled on first use; runtime_seconds includes that compilation.
    """
    games_per_style = int(games_per_style)
    max_hands = int(max_hands)
    if games_per_style < 2 or games_per_style % 2:
        raise ValueError('games_per_style must be an even integer >= 2 for duplicate deals.')
    if max_hands < 1:
        raise ValueError('max_hands must be positive.')
    if styles is None:
        styles = list(range(len(HEURISTIC_NAMES)))
    selected = [_style_id(s) for s in styles]
    if len(set(selected)) != len(selected):
        raise ValueError('Choose each heuristic opponent at most once.')
    snapshots = {} if snapshot_opponents is None else dict(snapshot_opponents)
    jobs = [(HEURISTIC_NAMES[s], s, None) for s in selected]
    jobs += [(f'snapshot:{name}', 0, weights) for name, weights in snapshots.items()]
    if not jobs:
        raise ValueError('Select at least one heuristic or snapshot opponent.')
    width = int(cfg.get('width', 256))
    roots = jax.random.split(jax.random.PRNGKey(int(seed)), games_per_style // 2)
    pair_keys = jnp.repeat(roots, 2, axis=0)
    learner_teams = jnp.tile(jnp.array([0, 1], jnp.int32), games_per_style // 2)
    teams_np = np.asarray(learner_teams)
    rows = []
    for label, style, opponent_weights in jobs:
        start = time.monotonic()
        kernel = _evaluation_kernel(width, opponent_weights is not None, bool(deterministic))
        outputs = jax.device_get(kernel(params, params if opponent_weights is None else opponent_weights,
                                        pair_keys, learner_teams, jnp.int32(style), jnp.int32(max_hands)))
        completed, winners, hands, scores, made, illegal, score_sum = map(np.asarray, outputs)
        resolved = completed.astype(bool)
        wins = resolved & (winners == teams_np)
        draws = resolved & (winners == 2)
        losses = resolved & ~wins & ~draws
        outcomes = np.where(resolved, wins.astype(float) + .5 * draws, np.nan)
        n_resolved = int(resolved.sum())
        credit = float(wins.sum() + .5 * draws.sum())
        low, high = _wilson_interval(credit, n_resolved)
        pair_low, pair_high, complete_pairs = _paired_interval(outcomes, int(seed))
        margins = scores[np.arange(games_per_style), teams_np] - scores[np.arange(games_per_style), 1 - teams_np]
        total_hands = int(hands.sum())
        row = dict(
            opponent=label, games=games_per_style, wins=int(wins.sum()), losses=int(losses.sum()),
            draws=int(draws.sum()), unresolved=int((~resolved).sum()),
            resolved_fraction=float(resolved.mean()), unresolved_fraction=float((~resolved).mean()),
            win_rate=credit / n_resolved if n_resolved else np.nan,
            ci_low=low, ci_high=high, paired_ci_low=pair_low, paired_ci_high=pair_high,
            complete_deal_pairs=complete_pairs,
            mean_hands=float(hands.mean()),
            completed_mean_hands=float(hands[resolved].mean()) if n_resolved else np.nan,
            completed_mean_score_margin=float(margins[resolved].mean()) if n_resolved else np.nan,
            unresolved_mean_score_margin=float(margins[~resolved].mean()) if (~resolved).any() else np.nan,
            mean_hand_score=float(score_sum.sum() / total_hands) if total_hands else np.nan,
            contract_success_rate=float(made.sum() / total_hands) if total_hands else np.nan,
            illegal_actions=int(illegal.sum()), seed=int(seed), max_hands=max_hands,
            deterministic=bool(deterministic), runtime_seconds=time.monotonic() - start,
        )
        rows.append(row)
        if progress:
            rate = 'n/a' if not n_resolved else f'{row["win_rate"]:.1%}'
            print(f'{label}: {row["wins"]}W {row["losses"]}L {row["draws"]}D, '
                  f'{row["unresolved"]} unresolved; resolved win rate {rate}; '
                  f'{row["runtime_seconds"]:.1f}s', flush=True)
    result = pd.DataFrame(rows)
    result.attrs['interpretation'] = (
        'Win rate gives true ties half credit and excludes unresolved capped matches. '
        'Wilson intervals are approximate; paired bootstrap resamples complete duplicate-deal pairs. '
        'Fixed-seed development benchmarks are selection data, not final holdout evidence.'
    )
    return result


def benchmark_selection_score(frame, max_unresolved_fraction=0.0):
    """Equal-opponent mean Wilson lower bound, or -inf when evaluation is unreliable.

    Use the SAME opponents, seed, cap and action mode when comparing checkpoints.
    This score is only a development selection criterion. A separate unseen-seed
    report is required after selecting the checkpoint; do not select on that report.
    """
    required = {'win_rate', 'ci_low', 'unresolved_fraction', 'illegal_actions'}
    if not required.issubset(frame.columns) or len(frame) == 0:
        raise ValueError('Pass a nonempty evaluate_policy result.')
    valid = (np.isfinite(frame['win_rate']).all()
             and np.isfinite(frame['ci_low']).all()
             and (frame['unresolved_fraction'] <= max_unresolved_fraction).all()
             and (frame['illegal_actions'] == 0).all())
    return float(frame['ci_low'].mean()) if valid else -math.inf


_CARD_RANKS = ('2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A')
_CARD_SUITS = ('C', 'D', 'H', 'S')


def card_text(card):
    card = int(card)
    return '--' if card < 0 else _CARD_RANKS[card % 13] + _CARD_SUITS[card // 13]


def _hand_text(hand):
    return ' | '.join(
        suit + ': ' + ' '.join(card_text(c) for c in range(s * 13, (s + 1) * 13) if hand[c])
        for s, suit in enumerate(_CARD_SUITS)
    )


def _bid_text(bid):
    bid = int(bid)
    return 'DN' if bid == -1 else ('?' if bid == -2 else ('Nil' if bid == 0 else str(bid)))


def _human_action(s, input_fn):
    p, ph = int(actor(s)), int(phase(s))
    mask = np.asarray(legal_mask(s))
    legal = np.flatnonzero(mask)
    if len(legal) == 1:
        # A forced card remains a play; a mandatory no-op needs no user interaction.
        if int(legal[0]) == 66:
            return 66
    if ph == 0:
        print('Your cards are face-down. Double Nil is a blind commitment.')
        options = 'n = decline' + (', d = Double Nil' if mask[67] else '')
    else:
        print('Your hand:', _hand_text(np.asarray(s.hands[p])))
        if ph == 1:
            print('Select one card to pass to your partner. Both selections stay hidden until exchange.')
        if ph == 2:
            print('Bids by seat:', ', '.join(f'{i}: {_bid_text(b)}' for i, b in enumerate(np.asarray(s.bids))))
        if ph == 3:
            print('Tricks by seat:', np.asarray(s.tricks).tolist())
            visible = [(int(player), card_text(card)) for player, card in
                       zip(np.asarray(s.table_players), np.asarray(s.table)) if int(card) >= 0]
            print('Current trick:', visible if visible else '(you lead)')
        options = ('bid 0 (Nil) through 13' if ph == 2
                   else 'legal cards: ' + ' '.join(card_text(c) for c in legal if c < 52))
    while True:
        raw = input_fn(options + ' [q to quit]: ').strip().upper()
        if raw in ('Q', 'QUIT', 'EXIT'):
            return None
        try:
            if ph == 0:
                action = 67 if raw in ('D', 'DN', 'DOUBLE', 'DOUBLE NIL') else (66 if raw in ('N', 'NO', 'DECLINE') else -1)
            elif ph == 2:
                bid = 0 if raw in ('N', 'NIL') else int(raw)
                action = 52 + bid if 0 <= bid <= 13 else -1
            else:
                token = raw.replace('♣', 'C').replace('♦', 'D').replace('♥', 'H').replace('♠', 'S')
                rank = '10' if token[:-1] == 'T' else token[:-1]
                action = _CARD_SUITS.index(token[-1]) * 13 + _CARD_RANKS.index(rank)
            if 0 <= action < len(mask) and mask[action]:
                return int(action)
        except (ValueError, IndexError):
            pass
        print('That action is not legal. Choose one of the displayed options.')


@lru_cache(maxsize=4)
def _human_inference(width):
    return jax.jit(lambda p, o: policy_logits(p, o, {'width': width}))


def human_play_match(params, cfg, opponents=('adaptive', 'card_counter'), seed=0,
                     human_seat=0, search=False, determinizations=16, input_fn=input):
    """Play a full match with an AI partner; q quits without manufacturing an outcome.

    Seats across the table are partners. Each opposing seat can be a HEURISTIC_NAMES
    entry, integer style, or 'ai' for the supplied policy. search=True uses the optional
    search_action helper for AI play turns; that can be substantially slower. Only your
    own hand and public information are printed. No hand is shown in the blind phase,
    and another player's pending pass is never shown. Scores and bags carry over until
    the engine declares a winner or a true ten-tiebreak-hand draw.

    The returned engine state is for developer debugging and contains hidden cards;
    do not inspect it while playing. This interactive notebook is not a secure server.
    """
    human_seat = int(human_seat)
    if human_seat not in range(4):
        raise ValueError('human_seat must be 0, 1, 2, or 3.')
    if len(opponents) != 2:
        raise ValueError('Supply exactly two opponent styles (one per opposing seat).')
    opposing_seats = [s for s in range(4) if s % 2 != human_seat % 2]
    seat_styles = {s: ('ai' if style == 'ai' else _style_id(style))
                   for s, style in zip(opposing_seats, opponents)}
    if search and 'search_action' not in globals():
        raise RuntimeError('Run the optional search helper cell before requesting search=True.')
    infer = _human_inference(int(cfg.get('width', 256)))
    step_compiled = jax.jit(step)
    bot_compiled = jax.jit(heuristic_action)
    root_key = jax.random.PRNGKey(int(seed))
    state = reset(jax.random.fold_in(root_key, 0))
    hands_played, turn_number = 0, 0
    print(f'You are seat {human_seat}; your AI partner is seat {(human_seat + 2) % 4}.')
    print('Cards: C clubs, D diamonds, H hearts, S spades; enter AS for ace of spades, 10H for ten of hearts.')
    while True:
        if int(state.turn) == 0:
            print(f'\nHand {hands_played + 1} | dealer {int(state.dealer)} | '
                  f'scores {np.asarray(state.scores).tolist()} | bags {np.asarray(state.bags).tolist()}')
        p, ph = int(actor(state)), int(phase(state))
        key = jax.random.fold_in(root_key, 1_000_000 + turn_number)
        if p == human_seat:
            action = _human_action(state, input_fn)
            if action is None:
                print('Session ended by you. The match is unfinished.')
                return {'completed': False, 'winner': None, 'hands': hands_played,
                        'scores': np.asarray(state.scores), 'state': state}
        elif p % 2 == human_seat % 2 or seat_styles[p] == 'ai':
            if search and ph == 3:
                action, _ = search_action(state, params, cfg, seed=int(seed) + turn_number,
                                          determinizations=int(determinizations))
                action = int(action)
            else:
                logits = infer(params, observe(state))
                action = int(jnp.argmax(jnp.where(legal_mask(state), logits, -jnp.inf)))
        else:
            action = int(bot_compiled(state, key, jnp.int32(seat_styles[p])))
        if not bool(np.asarray(legal_mask(state))[action]):
            raise RuntimeError('An AI returned an illegal action; stop and inspect the policy/search helper.')
        if ph == 0 and action == 67:
            print(f'Seat {p} declares Double Nil.')
        elif ph == 2 and action != 66:
            print(f'Seat {p} bids {_bid_text(action - 52)}.')
        elif ph == 3:
            print(f'Seat {p} plays {card_text(action)}.')
        before_phase = ph
        state, _, done, info = step_compiled(state, jnp.int32(action),
                                           jax.random.fold_in(root_key, hands_played + 1))
        turn_number += 1
        # Reveal only the human's completed exchange, and only after every pass is collected.
        if before_phase == 1 and int(phase(state)) == 2 and int(state.received[human_seat]) >= 0:
            print(f'Your exchange is complete: you received {card_text(state.received[human_seat])}.')
        if bool(info['hand_done']):
            hands_played += 1
            print(f'Hand result: score changes {np.asarray(info["score_delta"]).tolist()}, '
                  f'final scores {np.asarray(info["final_scores"]).tolist()}.')
        if bool(done):
            winner = int(info['winner'])
            print('Match tied after ten additional tied hands.' if winner == 2 else
                  ('Your partnership wins!' if winner == human_seat % 2 else 'The opposing partnership wins.'))
            return {'completed': True, 'winner': winner, 'hands': hands_played,
                    'scores': np.asarray(info['final_scores']), 'state': state}


## 7 · Optional information-set card search

`search_action` samples hypothetical hidden deals consistent with the actor's own cards, publicly observed voids, remaining hand sizes, and its known passed card. It compares legal cards by simulated continuation to the next hand and the trained value estimate. It never evaluates the real hidden deal. Candidate comparisons share random samples to reduce variance.

This is optional and slower than direct policy play. The sampler is nonuniform and does not condition on bidding likelihood; determinization can also suffer from strategy-fusion bias. Search may improve or weaken a policy, so compare it on independent deals before relying on it. Start with 8–16 samples late in a hand. Bidding and blind decisions use the direct policy.

In [ ]:
# Optional imperfect-information search. This is inference only, never a trainer.
from functools import lru_cache


def _search_sample_states(state, seed, count):
    """Sample legal hidden deals using only this actor's information set.

    Randomized bipartite matching exactly enforces known voids, public remaining
    hand sizes, and a known unplayed card passed to our partner. It is an
    approximate, NONUNIFORM belief sampler: bidding and past policy likelihoods
    are not used. No opponent hand or sealed pass selection is ever consulted.
    """
    p = int(actor(state))
    if int(phase(state)) != PLAY:
        raise ValueError('Hidden-deal search is available only during card play.')
    count = int(count)
    if count < 1:
        raise ValueError('At least one determinization is required.')
    rng = np.random.default_rng(int(seed))
    own = np.asarray(state.hands[p], dtype=bool)
    played = np.asarray(state.played_by, dtype=np.int32)
    voids = np.asarray(state.voids, dtype=bool)
    bids = np.asarray(state.bids, dtype=np.int32)
    own_passed = int(state.passed[p])
    own_received = int(state.received[p])
    capacities = 13 - np.array([(played == q).sum() for q in range(4)])
    if own.sum() != capacities[p] or np.any(own & (played >= 0)):
        raise ValueError('Own hand and public play history disagree.')
    if np.any(own & voids[p, np.arange(52) // 13]):
        raise ValueError('Own hand contradicts the public void information.')
    unknown = np.flatnonzero((played < 0) & ~own)
    capacities[p] = 0
    partner = (p + 2) % 4
    fixed_pass = own_passed if own_passed >= 0 and played[own_passed] < 0 else -1
    if fixed_pass >= 0:
        if own[fixed_pass] or voids[partner, fixed_pass // 13]:
            raise ValueError('Known passed card contradicts the public information.')
        capacities[partner] -= 1
        unknown = unknown[unknown != fixed_pass]
    if np.any(capacities < 0) or capacities.sum() != len(unknown):
        raise ValueError('Public remaining hand sizes are inconsistent.')
    slots = np.repeat(np.arange(4), capacities)
    legal_slots = [np.flatnonzero(~voids[slots, card // 13]) for card in unknown]
    if any(len(choices) == 0 for choices in legal_slots):
        raise ValueError('No hidden deal satisfies the public void constraints.')

    samples = []
    for _ in range(count):
        # A complete augmenting-path matching (not greedy dealing) avoids
        # falsely rejecting feasible void configurations late in the hand.
        adjacency = [rng.permutation(choices) for choices in legal_slots]
        slot_card = np.full(len(slots), -1, dtype=np.int32)
        order = sorted(rng.permutation(len(unknown)), key=lambda i: len(adjacency[i]))

        def augment(card_index, visited):
            for slot in adjacency[card_index]:
                if visited[slot]:
                    continue
                visited[slot] = True
                previous = slot_card[slot]
                if previous < 0 or augment(previous, visited):
                    slot_card[slot] = card_index
                    return True
            return False

        for index in order:
            if not augment(index, np.zeros(len(slots), dtype=bool)):
                raise ValueError('No hidden deal satisfies all public constraints.')
        hands = np.zeros((4, 52), dtype=bool)
        hands[p] = own
        if fixed_pass >= 0:
            hands[partner, fixed_pass] = True
        for slot, index in enumerate(slot_card):
            hands[slots[slot], unknown[index]] = True

        # Remove EVERY hidden pass field. For our partnership the reciprocal
        # sent/received pair is known; for the other partnership generate a
        # possible exchange from their sampled post-exchange thirteen cards.
        passed = np.full(4, -1, dtype=np.int32)
        received = np.full(4, -1, dtype=np.int32)
        if bids[p] == -1 or bids[partner] == -1:
            if own_passed < 0 or own_received < 0:
                raise ValueError('Completed Double Nil exchange lacks own card history.')
            passed[p], received[p] = own_passed, own_received
            passed[partner], received[partner] = own_received, own_passed
        opponent = (p + 1) % 4
        opposite = (opponent + 2) % 4
        if bids[opponent] == -1 or bids[opposite] == -1:
            for q in (opponent, opposite):
                original_post_exchange = np.flatnonzero(hands[q] | (played == q))
                if len(original_post_exchange) != 13:
                    raise ValueError('Sampled post-exchange hand does not contain thirteen cards.')
                received[q] = rng.choice(original_post_exchange)
            passed[opponent], passed[opposite] = received[opposite], received[opponent]
        samples.append(state._replace(
            hands=jnp.asarray(hands), pending_pass=jnp.full(4, -1, jnp.int32),
            passed=jnp.asarray(passed), received=jnp.asarray(received)))
    return jax.tree_util.tree_map(lambda *xs: jnp.stack(xs), *samples)


@lru_cache(maxsize=8)
def _search_rollout_kernel(width, heuristic_fraction, temperature):
    """Compile once per architecture/search setting; 13 candidate slots are fixed."""
    model_cfg = {'width': width}

    @jax.jit
    def run(params, starts, candidates, random_key):
        n = starts.turn.shape[0]
        states = jax.tree_util.tree_map(lambda x: jnp.repeat(x, 13, axis=0), starts)
        first_actions = jnp.tile(candidates, n)
        # Common random numbers pair candidate comparisons within each deal.
        keys = jnp.repeat(jax.random.split(random_key, n), 13, axis=0)
        batch = n * 13
        initial = (states, keys, jnp.ones(batch, jnp.bool_),
                   jnp.zeros(batch, jnp.float32), jnp.ones(batch, jnp.float32))

        def tick(carry, time_index):
            current, current_keys, active, returns, continuation = carry
            split = jax.vmap(lambda k: jax.random.split(k, 5))(current_keys)
            next_keys, policy_keys, mix_keys, heuristic_keys, step_keys = [
                split[:, i] for i in range(5)]

            def choose_rollout_actions(_):
                observations = jax.vmap(observe)(current)
                masks = jax.vmap(legal_mask)(current)
                logits = actor_logits(params, observations, model_cfg)
                masked = jnp.where(masks, logits / temperature, -1e9)
                chosen = jax.vmap(jax.random.categorical)(policy_keys, masked)
                if heuristic_fraction > 0.0:
                    baseline = jax.vmap(lambda s, k: heuristic_action(s, k, 7))(
                        current, heuristic_keys)
                    use_baseline = jax.vmap(lambda k: jax.random.bernoulli(
                        k, heuristic_fraction))(mix_keys)
                    chosen = jnp.where(use_baseline, baseline, chosen)
                return chosen.astype(jnp.int32)

            actions = jax.lax.cond(time_index == 0, lambda _: first_actions,
                                   choose_rollout_actions, operand=None)
            proposed, rewards, _, info = batch_step(current, actions, step_keys)
            # The next deal returned at the hand boundary is retained for value
            # bootstrapping. Never let its play affect this hand's comparison.
            frozen = jax.tree_util.tree_map(
                lambda new, old: jnp.where(
                    active.reshape((batch,) + (1,) * (new.ndim - 1)), new, old),
                proposed, current)
            finishes = active & info['hand_done']
            returns = returns + jnp.where(active, rewards, 0.0)
            continuation = jnp.where(finishes, info['discount'], continuation)
            active = active & ~info['hand_done']
            return (frozen, next_keys, active, returns, continuation), None

        # Fixed shape supports late and early positions without recompilation.
        # Once all branches finish, skip remaining scan iterations entirely.
        def masked_tick(carry, time_index):
            return jax.lax.cond(jnp.any(carry[2]),
                                lambda c: tick(c, time_index),
                                lambda c: (c, None), carry)

        final, _ = jax.lax.scan(masked_tick, initial, jnp.arange(64))
        final_states, _, unfinished, rewards, discounts = final
        # Critic sees ONLY sampled future deals. It never receives the actual
        # opponents' hidden cards at the searched position.
        bootstrap = value_apply(params, jax.vmap(critic_observe)(final_states), model_cfg)
        objective = rewards + discounts * bootstrap
        return objective.reshape(n, 13), unfinished.reshape(n, 13)

    return run


def search_action(state, params, cfg, seed=0, determinizations=16):
    """Select a card via optional information-set Monte Carlo rollouts.

    Call with the same learned parameters/config used by policy inference.
    Scores use the EXACT training objective: actual hand reward (match outcome
    plus potential shaping), then the learned discounted next-state value.
    All seats follow the learned actor, optionally mixed with the adaptive
    baseline via cfg['search_heuristic_fraction'] (default 0). The rollout
    sampling temperature defaults to 0.6 and may be set with
    cfg['search_temperature']. All candidate cards share sampled deals and RNG.

    This optional search is expensive, particularly on the first compilation;
    start with 8-16 deals in late-hand analysis. It is not a strength guarantee.
    Nonuniform beliefs, an approximate value function, and determinization's
    strategy-fusion bias can make it weaker than the direct policy. Benchmark
    it on held-out seeds before adopting it. It searches card play only; other
    phases and infeasible information sets fall back to the legal greedy actor.
    """
    mask = np.asarray(legal_mask(state), dtype=bool)
    logits = np.asarray(actor_logits(params, observe(state), cfg))
    fallback = int(np.argmax(np.where(mask, logits, -np.inf)))
    if int(phase(state)) != PLAY:
        return fallback, {'searched': False, 'reason': 'non-play phase'}
    legal = np.flatnonzero(mask)
    if len(legal) <= 1:
        return fallback, {'searched': False, 'reason': 'forced card'}
    determinizations = int(determinizations)
    if determinizations < 1:
        raise ValueError('determinizations must be positive.')
    fraction = float(cfg.get('search_heuristic_fraction', 0.0))
    temperature = float(cfg.get('search_temperature', 0.6))
    if not 0 <= fraction <= 1 or not np.isfinite(temperature) or temperature <= 0:
        raise ValueError('Search requires a fraction in [0,1] and positive finite temperature.')
    try:
        samples = _search_sample_states(state, seed, determinizations)
    except ValueError as error:
        return fallback, {'searched': False, 'reason': str(error)}
    # Fixed padding limits recompilation to changes in determinization count.
    candidates = np.full(13, legal[0], dtype=np.int32)
    candidates[:len(legal)] = legal
    kernel = _search_rollout_kernel(int(cfg.get('width', 256)), fraction, temperature)
    objectives, unfinished = kernel(params, samples, jnp.asarray(candidates),
                                    jax.random.PRNGKey(int(seed) ^ 0x5EAD))
    if np.any(np.asarray(unfinished)):
        raise RuntimeError('Search failed to reach the next hand boundary within 64 turns.')
    team_sign = 1.0 if int(actor(state)) % 2 == 0 else -1.0
    values = np.asarray(objectives)[:, :len(legal)] * team_sign
    if not np.isfinite(values).all():
        return fallback, {'searched': False, 'reason': 'non-finite search values'}
    means = values.mean(axis=0)
    # Resolved contracts often make several cards exactly equivalent. Preserve
    # the learned preference when simulated returns cannot distinguish them.
    tied = np.isclose(means, means.max(), rtol=0.0, atol=1e-8)
    best = int(np.argmax(np.where(tied, logits[legal], -np.inf)))
    # This SE describes rollout sampling only; it does not measure belief bias.
    errors = (values.std(axis=0, ddof=1) / np.sqrt(determinizations)
              if determinizations > 1 else np.full(len(legal), np.nan))
    paired_errors = ((values - values[:, best:best + 1]).std(axis=0, ddof=1)
                     / np.sqrt(determinizations)
                     if determinizations > 1 else np.full(len(legal), np.nan))
    return int(legal[best]), {
        'searched': True, 'determinizations': determinizations,
        'legal_actions': legal.tolist(), 'mean_objective': means.tolist(),
        'sampling_standard_error': errors.tolist(),
        'paired_difference_standard_error': paired_errors.tolist(),
        'greedy_policy_action': fallback,
        'belief_method': 'approximate nonuniform randomized constrained matching',
    }


def search_privacy_selftest(state, seed=123):
    """Changing every unseen hand/pass must leave sampled states bit-identical."""
    if int(phase(state)) != PLAY:
        raise ValueError('Provide a PLAY state for this privacy self-test.')
    p = int(actor(state))
    others = [q for q in range(4) if q != p]
    changed_hands = state.hands.at[jnp.asarray(others)].set(
        state.hands[jnp.asarray(others[::-1])])
    changed_passed = state.passed.at[jnp.asarray(others)].set(jnp.int32(51))
    changed_received = state.received.at[jnp.asarray(others)].set(jnp.int32(0))
    altered = state._replace(hands=changed_hands,
                            pending_pass=jnp.arange(4, dtype=jnp.int32),
                            passed=changed_passed, received=changed_received)
    left = _search_sample_states(state, seed, 3)
    right = _search_sample_states(altered, seed, 3)
    assert all(np.array_equal(np.asarray(a), np.asarray(b)) for a, b in zip(
        jax.tree_util.tree_leaves(left), jax.tree_util.tree_leaves(right)))
    return True


print('Optional information-set search ready: search_action(..., determinizations=16).')


## 8 · Actual strength, GPU memory, and durable checkpoints

The dashboard now has eight panels: training losses; policy diagnostics; contract success; net points per hand; Nil/Double Nil declarations; throughput; fixed-opponent win rate with Wilson intervals; and GPU memory. Training learner points include self-play, while opponent points come from non-self-play games; their difference is not a direct head-to-head score margin. Use the fixed benchmark to compare strength. The header identifies the trainer revision and actual workload. Active JAX arrays, the allocator pool, driver-reported board usage, and the target are distinct measurements.

On an imported run, the incoming actor is benchmarked against the fixed `adaptive` opponent **before** teacher refresh. That record is checkpointed so resuming does not repeat it. Another benchmark becomes due when warmup finishes, followed by the existing 30-minute cadence. The best model is selected by the lower Wilson win-rate bound only when every match resolves with no illegal actions. If later training regresses, the stronger evaluated checkpoint remains available in **15 · Inspect progress**.

The checkpoint now has 214 default state leaves: the original 211 plus three exact counter scalars for batch resizing. Legacy checkpoints are read without modifying them. New saves preserve model, Adam state, RNG, games, league, counters, and teacher progress; future v3 resumes restore them exactly. JSON/NumPy ZIPs remain checksum-verified and pickle-free. The newest three regular saves and one best-development save are retained in the new folder.

Checkpoints remain scheduled every 90 minutes, at normal segment end, and on handled interruption. Numerical guards reject bad candidates and keep the last accepted state; Drive failures attempt a local emergency checkpoint. Forced runtime termination can recover only durable saves. Contract rates and self-play points alone do not establish strength.
Every accepted PPO update also appends a compact `optimizer_work` record to history and metrics.jsonl. The three-rollout optimizer-health rule commits a numerically valid candidate, retains an `optimizer_health_stop_*.zip` save with no cleanup, and stops before evaluation; it never labels skipped minibatches as executed.


In [ ]:
# Safe, complete checkpoints and the time-budgeted notebook training runner.
import io
import json
import os
import sys
import time
import hashlib
import tempfile
import zipfile
from pathlib import Path
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, clear_output

CHECKPOINT_FORMAT = 1
CHECKPOINT_STATE_SCHEMA = 2
_LEGACY_TRAINING_REVISIONS = frozenset(('potential-only-v1', 'score-feedback-bidding-anchor-v2'))
_COUNTER_STATE_FIELDS = ('env_steps_low', 'env_steps_high', 'base_updates')
RULESET_ID = 'modified-spades-500-blind-swap-team-failure-v1'

def _json_safe(value):
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [_json_safe(v) for v in value]
    if isinstance(value, (np.generic,)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def _checkpoint_creation_key(path):
    """Order saves by creation time, including explicit lower-update resumes."""
    path = Path(path)
    try:
        stamp = datetime.strptime(path.stem.rsplit('_', 1)[-1], '%Y%m%dT%H%M%S%f')
        created = stamp.replace(tzinfo=timezone.utc).timestamp()
    except ValueError:
        # Imported/older names may lack our UTC suffix; use their file time.
        created = path.stat().st_mtime_ns / 1e9
    return created, path.name


def _validate_checkpoint_history(history):
    if not isinstance(history, list) or any(not isinstance(row, dict) for row in history):
        raise ValueError('Checkpoint history must be a list of JSON objects.')
    try:
        json.dumps(history, allow_nan=False)
    except (TypeError, ValueError) as exc:
        raise ValueError('Checkpoint history must contain finite JSON values.') from exc


def _validate_checkpoint_manifest(metadata):
    """Reject malformed archive metadata before accessing it or allocating state."""
    if not isinstance(metadata, dict):
        raise ValueError('Checkpoint manifest must be a JSON object.')
    if type(metadata.get('format')) is not int or metadata['format'] != CHECKPOINT_FORMAT or metadata.get('ruleset') != RULESET_ID:
        raise ValueError('Checkpoint format or game rules do not match this notebook.')
    versions = metadata.get('versions')
    if not isinstance(versions, dict) or any(
            not isinstance(versions.get(name), str) or not versions[name]
            for name in ('jax', 'flax', 'optax')):
        raise ValueError('Checkpoint versions must specify JAX, Flax, and Optax strings.')
    saved_cfg = metadata.get('cfg')
    if not isinstance(saved_cfg, dict) or any(name not in saved_cfg for name in ('n_env', 'rollout_len')):
        raise ValueError('Checkpoint config must be an object with its saved batch dimensions.')
    try:
        learning_config(saved_cfg)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError('Checkpoint training config is invalid: ' + str(exc)) from exc
    if any(type(metadata.get(name)) is not int or metadata[name] < 0
           for name in ('updates', 'env_steps')):
        raise ValueError('Checkpoint manifest counters must be exact nonnegative integers.')
    seconds = metadata.get('cumulative_seconds', 0.)
    try:
        valid_seconds = type(seconds) in (int, float) and np.isfinite(float(seconds)) and seconds >= 0
    except (TypeError, ValueError, OverflowError):
        valid_seconds = False
    if not valid_seconds:
        raise ValueError('Checkpoint cumulative_seconds must be finite and nonnegative.')
    checksum = metadata.get('sha256')
    if not isinstance(checksum, str) or len(checksum) != 64 or any(c not in '0123456789abcdef' for c in checksum):
        raise ValueError('Checkpoint checksum must be a SHA256 hexadecimal string.')
    specs = metadata.get('leaves')
    if not isinstance(specs, list):
        raise ValueError('Checkpoint leaves must be a list of array specifications.')
    for spec in specs:
        if (not isinstance(spec, dict) or not isinstance(spec.get('shape'), list)
                or any(type(size) is not int or size < 0 for size in spec['shape'])
                or not isinstance(spec.get('dtype'), str)):
            raise ValueError('Checkpoint contains an invalid array specification.')
        try:
            dtype = np.dtype(spec['dtype'])
        except (TypeError, ValueError) as exc:
            raise ValueError('Checkpoint contains an invalid array dtype.') from exc
        if dtype.hasobject:
            raise ValueError('Checkpoint arrays cannot contain Python objects.')
    revision = metadata.get('training_revision', 'potential-only-v1')
    if not isinstance(revision, str):
        raise ValueError('Checkpoint training revision must be a string.')
    return saved_cfg


def save_checkpoint(state, cfg, history, directory, cumulative_seconds=0., tag='checkpoint', keep=3):
    """Single zip; JSON + non-pickled NumPy arrays; validate before atomic rename."""
    directory = Path(directory)
    retention_count = None if keep is None else max(1, int(keep))
    leaves, _ = jax.tree_util.tree_flatten(jax.device_get(state))
    arrays = {f'leaf_{i:05d}': np.asarray(x) for i, x in enumerate(leaves)}
    for name, array in arrays.items():
        if array.dtype.hasobject or (np.issubdtype(array.dtype, np.inexact) and not np.isfinite(array).all()):
            raise ValueError(f'Refusing to save invalid training state at {name}; existing checkpoints are unchanged.')
    safe_history = _json_safe(history)
    _validate_checkpoint_history(safe_history)
    data = io.BytesIO()
    np.savez(data, **arrays)
    payload = data.getvalue()
    manifest = {
        'format': CHECKPOINT_FORMAT, 'ruleset': RULESET_ID,
        'cfg': _json_safe(cfg), 'cumulative_seconds': float(cumulative_seconds),
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'updates': int(state.updates), 'env_steps': int(state.env_steps),
        'sha256': hashlib.sha256(payload).hexdigest(),
        'leaves': [{'shape': list(x.shape), 'dtype': str(x.dtype)} for x in arrays.values()],
        'versions': {'jax': jax.__version__, 'flax': flax.__version__, 'optax': optax.__version__},
        'training_revision': TRAINING_REVISION,
        'training_state_schema': CHECKPOINT_STATE_SCHEMA,
    }
    _validate_checkpoint_manifest(manifest)
    if not 0 <= int(state.base_updates) <= int(state.updates):
        raise ValueError('Refusing to save invalid cumulative decision counter bounds.')
    if int(state.decisions_per_update) != int(cfg['n_env']) * int(cfg['rollout_len']):
        raise ValueError('Refusing to save a decision counter inconsistent with the batch config.')
    directory.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
    name = f'{tag}_{int(state.updates):09d}_{stamp}.zip'
    destination = directory / name
    temporary = directory / ('.' + name + '.partial')
    try:
        with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as z:
            z.writestr('manifest.json', json.dumps(manifest, indent=2, allow_nan=False))
            z.writestr('state.npz', payload)
            z.writestr('history.json', json.dumps(safe_history, allow_nan=False))
        # Detect incomplete writes before making the new file discoverable.
        with zipfile.ZipFile(temporary, 'r') as z:
            if hashlib.sha256(z.read('state.npz')).hexdigest() != manifest['sha256']:
                raise IOError('Checkpoint checksum verification failed.')
        os.replace(temporary, destination)
    finally:
        if temporary.exists():
            temporary.unlink()
    # Only delete older checkpoints after this complete file is committed.
    if retention_count is not None:
        previous = sorted((p for p in directory.glob(f'{tag}_*.zip') if p != destination),
                          key=_checkpoint_creation_key, reverse=True)
        # The committed destination always occupies one retained slot, even if
        # it resumes an older update or the host clock has moved backwards.
        for p in previous[retention_count - 1:]:
            p.unlink()
    return destination


def load_checkpoint(path, cfg=None):
    """Recreate the exact pytree via its saved config; refuse incompatible changes."""
    with zipfile.ZipFile(path, 'r') as z:
        metadata = json.loads(z.read('manifest.json'))
        saved_cfg = _validate_checkpoint_manifest(metadata)
        installed_stack = {'jax': jax.__version__, 'flax': flax.__version__, 'optax': optax.__version__}
        saved_stack = metadata.get('versions', {})
        mismatch = {name: (saved_stack.get(name), version) for name, version in installed_stack.items()
                    if saved_stack.get(name) != version}
        if mismatch:
            raise ValueError('Checkpoint library versions differ from this runtime: '
                             + str(mismatch) + '. Set RESUME_FILE to this checkpoint in cell 1, '
                             'rerun setup, then restart the runtime if instructed.')
        if cfg is not None and not isinstance(cfg, dict):
            raise ValueError('Requested checkpoint config must be a dictionary.')
        if cfg is not None and _json_safe(cfg) != saved_cfg:
            differences = {k: (saved_cfg.get(k), _json_safe(cfg).get(k))
                           for k in saved_cfg.keys() | cfg.keys()
                           if saved_cfg.get(k) != _json_safe(cfg).get(k)}
            raise ValueError(f'Training config changed. Resume with saved configuration: {differences}')
        payload = z.read('state.npz')
        if hashlib.sha256(payload).hexdigest() != metadata['sha256']:
            raise IOError('Checkpoint is corrupt: array checksum mismatch.')
        history = json.loads(z.read('history.json'))
        _validate_checkpoint_history(history)
    revision = metadata.get('training_revision', 'potential-only-v1')
    schema = metadata.get('training_state_schema')
    legacy = (schema is None or type(schema) is int and schema == 1) and revision in _LEGACY_TRAINING_REVISIONS
    if not legacy and (type(schema) is not int or schema != CHECKPOINT_STATE_SCHEMA
                       or revision != TRAINING_REVISION):
        raise ValueError(f'Unsupported checkpoint training state schema/revision: {schema!r}/{revision!r}.')
    # Reconstruct shape/tree information without allocating a second device state.
    template = jax.eval_shape(lambda: initialize_training(saved_cfg))
    template_leaves, treedef = jax.tree_util.tree_flatten(template)
    if tuple(template._fields[-3:]) != _COUNTER_STATE_FIELDS:
        raise ValueError('This notebook has an incompatible training counter state layout.')
    expected_count = len(template_leaves) - (3 if legacy else 0)
    loaded = []
    try:
        archive = np.load(io.BytesIO(payload), allow_pickle=False)
    except EOFError as exc:
        raise ValueError('Checkpoint state payload is empty or truncated.') from exc
    if not isinstance(archive, np.lib.npyio.NpzFile):
        raise ValueError('Checkpoint state payload must be a NumPy array archive.')
    with archive as a:
        expected = [f'leaf_{i:05d}' for i in range(expected_count)]
        if sorted(a.files) != expected or len(metadata['leaves']) != expected_count:
            raise ValueError('Checkpoint pytree does not match the training state.')
        for name, spec, template_leaf in zip(expected, metadata['leaves'], template_leaves):
            x = a[name]
            if list(x.shape) != spec['shape'] or str(x.dtype) != spec['dtype']:
                raise ValueError(f'Invalid stored array specification: {name}')
            if x.shape != template_leaf.shape or x.dtype != np.dtype(template_leaf.dtype):
                raise ValueError(f'Architecture/state mismatch at {name}; use the original notebook/config.')
            if np.issubdtype(x.dtype, np.inexact) and not np.isfinite(x).all():
                raise ValueError(f'Checkpoint contains non-finite training state at {name}; '
                                 'restore an earlier finite checkpoint.')
            loaded.append(x)
    if legacy:
        # The old state ended before these three scalars. Zero offset/base
        # reproduces its exact updates * decisions_per_update counter. Never
        # pad arbitrary short payloads or guess missing model/optimizer leaves.
        loaded.extend((np.asarray(0, np.uint32), np.asarray(0, np.uint32), np.asarray(0, np.int32)))
    state = jax.tree_util.tree_unflatten(treedef, loaded)
    if not 0 <= int(state.base_updates) <= int(state.updates):
        raise ValueError('Checkpoint has invalid cumulative decision counter bounds.')
    batch_decisions = int(saved_cfg['n_env']) * int(saved_cfg['rollout_len'])
    if int(state.decisions_per_update) != batch_decisions or batch_decisions <= 0:
        raise ValueError('Checkpoint decision counter does not match its saved batch configuration.')
    if int(state.updates) != metadata['updates'] or int(state.env_steps) != metadata['env_steps']:
        raise ValueError('Checkpoint update/decision counters do not match their manifest.')
    # All array and counter validation precedes the first GPU transfer.
    gpu_admit_checkpoint_load(saved_cfg, sum(x.nbytes for x in loaded))
    state = jax.tree_util.tree_unflatten(treedef, [jnp.asarray(x) for x in loaded])
    return state, saved_cfg, history, metadata


def find_resume_checkpoint(directory):
    """Try newest regular checkpoint first. Fall back if a write/download was damaged."""
    paths = sorted(Path(directory).glob('checkpoint_*.zip'), key=_checkpoint_creation_key, reverse=True)
    for p in paths:
        try:
            result = load_checkpoint(p)
            print(f'Resumed {p.name}: {int(result[0].updates):,} updates, {int(result[0].env_steps):,} decisions.')
            return result, p
        except (OSError, ValueError, KeyError, zipfile.BadZipFile) as exc:
            print(f'Skipping unreadable checkpoint {p.name}: {exc}')
    if paths:
        raise RuntimeError('No valid checkpoint was found. Restore a backup or choose a new run folder.')
    return None, None


def _scalar_metrics(metrics):
    result = {}
    for name, value in jax.device_get(metrics).items():
        a = np.asarray(value)
        if a.size == 1:
            result[name] = float(a.reshape(()))
    return result


def _diagnostic_json(value):
    """Preserve numerical failures in strict JSON without writing NaN tokens."""
    if isinstance(value, dict):
        return {str(k): _diagnostic_json(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [_diagnostic_json(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return 'NaN' if np.isnan(value) else ('Infinity' if value > 0 else '-Infinity')
    return _json_safe(value)


def _nonfinite_state_summary(tree):
    """Inspect rejected state only; avoid copying model arrays on healthy steps."""
    leaves, _ = jax.tree_util.tree_flatten_with_path(jax.device_get(tree))
    bad = []
    total = 0
    for path, value in leaves:
        a = np.asarray(value)
        count = int(np.count_nonzero(~np.isfinite(a)))
        if count:
            total += count
            bad.append(dict(path=jax.tree_util.keystr(path), nonfinite_elements=count))
    return dict(nonfinite_elements=total, nonfinite_leaves=len(bad), first_bad_leaves=bad[:8])


def _check_training_numerics(state, candidate, scalars, stage, history, directory):
    """Reject a bad update and retain enough evidence to diagnose it after resume."""
    nonfinite = {name: value for name, value in scalars.items() if not np.isfinite(value)}
    failed_checks = {name: value for name, value in scalars.items()
                     if (name == 'finite' or name.endswith('_finite'))
                     and np.isfinite(value) and value < .5}
    if not nonfinite and not failed_checks:
        return
    last_update = int(state.updates)
    attempt = int(state.bootstrap_updates) + 1 if stage == 'bootstrap' else last_update + 1
    record = _diagnostic_json(dict(
        kind='numerical_failure', stage=stage, update=last_update,
        attempted_update=attempt, last_accepted_update=last_update,
        last_accepted_bootstrap_update=int(state.bootstrap_updates),
        created_utc=datetime.now(timezone.utc).isoformat(),
        nonfinite_metrics=nonfinite, failed_checks=failed_checks, metrics=scalars,
        candidate_params=_nonfinite_state_summary(candidate.params),
        candidate_optimizer=_nonfinite_state_summary(candidate.opt_state)))
    history.append(record)
    globals()['LAST_NUMERICAL_FAILURE'] = record
    try:
        _append_jsonl(directory, record)
    except OSError as exc:
        # The final checkpoint (or emergency checkpoint) still contains this event.
        print(f'Could not append failure diagnostics to metrics.jsonl: {exc}', flush=True)
    details = []
    if nonfinite:
        details.append('non-finite metrics: ' + ', '.join(
            f'{name}={value}' for name, value in record['nonfinite_metrics'].items()))
    if failed_checks:
        details.append('failed checks: ' + ', '.join(
            f'{name}={value:g}' for name, value in failed_checks.items()))
    raise FloatingPointError(
        f"Rejected {stage} update {attempt}: {'; '.join(details)}. "
        f'Keeping last accepted PPO update {last_update}. '
        'Full diagnostics are in LAST_NUMERICAL_FAILURE, metrics.jsonl, and checkpoint history.')


def _gpu_telemetry():
    """Host-only observability; unavailable GPU telemetry cannot stop training."""
    reader = globals().get('gpu_memory_metrics')
    if not callable(reader):
        return {}
    try:
        metrics = reader()
        if not isinstance(metrics, dict):
            raise TypeError('GPU telemetry reader must return a dictionary.')
        return _replace_nonfinite(metrics)
    except Exception as exc:
        return {'gpu_telemetry_status': f'{type(exc).__name__}: {exc}'}


def _gpu_health_rows(metrics):
    """Display working allocations separately from GPU driver/pool occupancy."""
    rows = []
    fields = [
        ('jax_live_gib', 'JAX active working memory'),
        ('jax_reserved_gib', 'JAX reserved memory pool'),
        ('board_used_gib', 'GPU occupied memory (includes reserved pool)'),
        ('gpu_target_gib', 'GPU memory budget'),
        ('gpu_estimated_working_gib', 'Estimated compiled working memory'),
    ]
    for key, label in fields:
        value = metrics.get(key)
        if isinstance(value, (int, float)) and np.isfinite(value):
            rows.append((label, f'{value:.2f} GiB'))
    if metrics.get('gpu_telemetry_status'):
        rows.append(('GPU telemetry', str(metrics['gpu_telemetry_status'])))
    return rows


def show_training_metrics(history, latest, elapsed, budget, checkpoint_path=None):
    """Display learning quality, bidding behavior, and throughput during training."""
    clear_output(wait=True)
    print(f"Segment {elapsed/3600:.2f}/{budget/3600:.2f} hours | "
          f"update {int(latest['update']):,} | decisions {int(latest['env_steps']):,} | "
          f"{latest.get('decisions_per_second', 0):,.0f} decisions/sec")
    print(f"Trainer: {latest.get('training_revision', TRAINING_REVISION)} | "
          f"parallel games {latest.get('n_env', '—')} × rollout {latest.get('rollout_len', '—')} | "
          f"minibatch {latest.get('minibatch_size', '—')} | network width {latest.get('width', '—')}")
    if checkpoint_path:
        print('Latest durable checkpoint:', checkpoint_path)
    recent = [{k: v for k, v in r.items() if k != 'kind'}
              for r in history if r.get('kind') == 'train'][-300:]
    if not recent:
        return
    df = pd.DataFrame(recent)
    updates = pd.to_numeric(df['update'], errors='coerce')
    fig, axes = plt.subplots(4, 2, figsize=(14, 13), sharex=True, constrained_layout=True)

    def plot_series(ax, series):
        lines = []
        for name, label, color in series:
            if name not in df:
                continue
            values = pd.to_numeric(df[name], errors='coerce')
            if not np.isfinite(values.to_numpy(dtype=float)).any():
                continue
            lines.extend(ax.plot(updates, values, label=label, color=color, linewidth=1.6))
        return lines

    def finish_panel(ax, title, lines, ylabel=None, rates=False):
        ax.set(title=title, xlabel='PPO update', ylabel=ylabel or '')
        ax.grid(alpha=.2)
        if rates:
            ax.set_ylim(-.025, 1.025)
            ax.set_yticks([0, .25, .5, .75, 1], ['0%', '25%', '50%', '75%', '100%'])
        if lines:
            ax.legend(handles=lines, fontsize=8, loc='best')
        else:
            ax.text(.5, .5, 'Not recorded by the earlier trainer',
                    ha='center', va='center', transform=ax.transAxes, color='0.45', fontsize=9)

    ax = axes[0, 0]
    lines = plot_series(ax, [('policy_loss', 'Policy', '#2878b5'),
                             ('value_loss', 'Value', '#d95319'),
                             ('bidding_anchor_loss', 'Bidding guide', '#719c34'),
                             ('play_anchor_loss', 'Card-play guide', '#9467bd')])
    finish_panel(ax, 'Training losses', lines, 'Loss')

    ax = axes[0, 1]
    lines = plot_series(ax, [('approx_kl', 'Approx. KL', '#2878b5'),
                             ('clip_fraction', 'Clipped fraction', '#d95319')])
    entropy_axis = ax.twinx()
    entropy_lines = plot_series(entropy_axis, [('entropy', 'Entropy (right)', '#9467bd')])
    entropy_axis.set_ylabel('Policy entropy', color='#9467bd')
    entropy_axis.tick_params(axis='y', colors='#9467bd')
    if entropy_lines:
        entropy_axis.set_ylim(bottom=0)
    finish_panel(ax, 'Policy diagnostics', lines + entropy_lines, 'KL / clipped fraction')

    ax = axes[1, 0]
    lines = plot_series(ax, [('learner_contract_success_rate', 'Learner contract', '#2878b5'),
                             ('contract_success_rate', 'All contracts', '#719c34'),
                             ('nil_success_rate', 'Nil success', '#d95319'),
                             ('dn_success_rate', 'Double Nil success', '#9467bd')])
    finish_panel(ax, 'Contract success', lines, rates=True)

    ax = axes[1, 1]
    lines = plot_series(ax, [('learner_hand_score', 'Learner (all training games)', '#2878b5'),
                             ('opponent_hand_score', 'Opponent (non-self-play)', '#d95319')])
    ax.axhline(0, color='0.4', linewidth=.8, alpha=.6)
    finish_panel(ax, 'Training points / hand (different game mixtures)', lines, 'Points / hand')

    ax = axes[2, 0]
    lines = plot_series(ax, [('nil_bid_rate', 'Nil declaration', '#d95319'),
                             ('dn_bid_rate', 'Double Nil declaration', '#9467bd')])
    finish_panel(ax, 'Nil and Double Nil declarations', lines, rates=True)

    ax = axes[2, 1]
    lines = plot_series(ax, [('decisions_per_second', 'Decisions / sec', '#2878b5')])
    finish_panel(ax, 'Training throughput', lines, 'Decisions / sec')


    # Benchmarks use fixed development deals; unresolved matches remain visible.
    ax = axes[3, 0]
    benchmark_rows = []
    for event in history:
        if event.get('kind') != 'evaluation' or 'update' not in event:
            continue
        for row in event.get('rows', []):
            if row.get('opponent') == 'adaptive':
                benchmark_rows.append(dict(row, update=event['update']))
    benchmark_rows = benchmark_rows[-100:]
    lines = []
    if benchmark_rows:
        benchmark = pd.DataFrame(benchmark_rows)
        x = pd.to_numeric(benchmark['update'], errors='coerce').to_numpy(dtype=float)
        for name, label, color in [('win_rate', 'Resolved win rate', '#2878b5'),
                                   ('unresolved_fraction', 'Unresolved fraction', '#d95319')]:
            if name in benchmark:
                y = pd.to_numeric(benchmark[name], errors='coerce').to_numpy(dtype=float)
                lines.extend(ax.plot(x, y, marker='o', markersize=3, label=label, color=color))
        if 'ci_low' in benchmark and 'ci_high' in benchmark:
            low = pd.to_numeric(benchmark['ci_low'], errors='coerce').to_numpy(dtype=float)
            high = pd.to_numeric(benchmark['ci_high'], errors='coerce').to_numpy(dtype=float)
            ax.fill_between(x, low, high, color='#2878b5', alpha=.15, label='95% Wilson interval')
            lines = list(ax.get_lines()) + list(ax.collections)
    finish_panel(ax, 'Fixed benchmark versus adaptive', lines, rates=True)
    if not benchmark_rows:
        # A new run has no benchmark until teacher initialization completes.
        for text in list(ax.texts):
            text.remove()
        ax.text(.5, .5, 'Benchmark follows teacher initialization, then every 30 min',
                ha='center', va='center', transform=ax.transAxes, color='0.45', fontsize=8)

    ax = axes[3, 1]
    lines = []
    for name, label, color in [('jax_live_gib', 'JAX active allocations', '#2878b5'),
                              ('jax_reserved_gib', 'JAX reserved pool', '#9467bd'),
                              ('board_used_gib', 'GPU occupied', '#719c34'),
                              ('gpu_target_gib', 'Memory budget', '#d95319')]:
        if name in df:
            values = pd.to_numeric(df[name], errors='coerce')
            if np.isfinite(values.to_numpy(dtype=float)).any():
                lines.extend(ax.plot(updates, values, label=label, color=color,
                                     linestyle='--' if name == 'gpu_target_gib' else '-'))
    finish_panel(ax, 'GPU memory: allocations and reserved pool', lines, 'GiB')

    # Keep historical points intact; indicate exactly where training behavior changed.
    finite_updates = updates[np.isfinite(updates.to_numpy(dtype=float))]
    if len(finite_updates):
        left, right = float(finite_updates.min()), float(finite_updates.max())
        for event in history:
            if event.get('kind') != 'trainer_upgrade':
                continue
            try:
                upgrade = float(event['update'])
            except (KeyError, TypeError, ValueError):
                continue
            if np.isfinite(upgrade) and left <= upgrade <= right:
                for ax in axes.flat:
                    ax.axvline(upgrade, color='0.35', linestyle='--', linewidth=1, alpha=.65)
                axes[0, 0].text(upgrade, .98, ' Trainer upgrade',
                    transform=axes[0, 0].get_xaxis_transform(), va='top', fontsize=8,
                    color='0.3', bbox=dict(facecolor='white', edgecolor='none', alpha=.8))
    display(fig)
    plt.close(fig)

    def metric_text(name, fmt='.3g'):
        value = latest.get(name)
        if value is None:
            return '—'
        try:
            value = float(value)
        except (TypeError, ValueError):
            return '—'
        return format(value, fmt) if np.isfinite(value) else str(value)

    health = [
        ('Match age: mean / max (hands)',
         metric_text('mean_match_hands', '.1f') + ' / ' + metric_text('max_match_hands', '.0f')),
        ('Completed matches in this update', metric_text('completed_matches', ',.0f')),
        ('Training trajectory refreshes in this update', metric_text('training_truncations', ',.0f')),
        ('Mean normal bid', metric_text('mean_normal_bid', '.2f')),
        ('Team scores: 0 / 1', metric_text('mean_score0', '.1f') + ' / ' + metric_text('mean_score1', '.1f')),
        ('Saturated score-gap fraction', metric_text('score_gap_saturated_fraction', '.1%')),
        ('Numerical failure events in run', str(sum(r.get('kind') == 'numerical_failure' for r in history))),
        ('Illegal action rate', metric_text('illegal_rate', '.2%')),
    ]
    for name, label in [('raw_adv_std', 'Raw advantage standard deviation'),
                        ('hand_reward_mean_abs', 'Mean absolute hand reward'),
                        ('grad_norm', 'Gradient norm')]:
        if name in latest:
            health.append((label, metric_text(name)))
    if any(name in latest for name in ('entropy_blind', 'entropy_bid', 'entropy_play')):
        health.append(('Entropy: blind / bid / play', ' / '.join(
            metric_text(name, '.2f') for name in ('entropy_blind', 'entropy_bid', 'entropy_play'))))
    phase_accuracy = [name for name in ('teacher_accuracy_blind', 'teacher_accuracy_bid',
                                       'teacher_accuracy_pass', 'teacher_accuracy_play') if name in latest]
    if phase_accuracy:
        health.append(('Teacher agreement: ' + ' / '.join(name.rsplit('_', 1)[-1] for name in phase_accuracy),
                       ' / '.join(metric_text(name, '.1%') for name in phase_accuracy)))
    health.extend(_gpu_health_rows(latest))
    display(pd.DataFrame(health[:20], columns=['Training health', 'Latest']).set_index('Training health'))
    evaluations = [r for r in history if r.get('kind') == 'evaluation' and r.get('rows')]
    if evaluations:
        print('Latest fixed development benchmark (strength evidence; self-play win rate is not a strength measure):')
        display(pd.DataFrame(evaluations[-1]['rows']))


def _append_jsonl(directory, record):
    # Logs are useful artifacts; the checkpoint's history is authoritative on resume.
    with open(Path(directory) / 'metrics.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(_json_safe(record), allow_nan=False) + '\n')


def _best_development_score(history):
    """Select only benchmarks belonging to the current recovered run.

    Older rows remain available for plots, but their best checkpoint lives in
    the original folder and cannot establish the recovered folder's threshold.
    """
    start = max((i + 1 for i, row in enumerate(history)
                 if row.get('kind') == 'trainer_upgrade'), default=0)
    scores = [row['selection_score'] for row in history[start:]
              if row.get('kind') == 'evaluation' and row.get('selection_score') is not None
              and np.isfinite(row['selection_score']) and row.get('rows')
              and all(result.get('unresolved', 0) == 0
                      and result.get('illegal_actions', 0) == 0 for result in row['rows'])]
    return max(scores, default=-float('inf'))


class OptimizerHealthStop(RuntimeError):
    """Accepted state is retained; inspect the durable stop checkpoint before retry."""


def _optimizer_exact_integer(value, label, *, minimum=0, allow_bool=False):
    """Accept scalar numeric counts, never fractional/nonfinite/coerced strings."""
    array = np.asarray(jax.device_get(value))
    if (array.shape != () or array.dtype.kind not in ('i', 'u', 'f', 'b')
            or array.dtype.kind == 'b' and not allow_bool):
        raise ValueError(f'{label} must be an exact scalar numeric integer.')
    number = array.item()
    if not np.isfinite(number) or number != int(number) or number < minimum:
        raise ValueError(f'{label} must be a finite integer >= {minimum}.')
    return int(number)


def _optimizer_cadence(cfg):
    values = {name: _optimizer_exact_integer(cfg[name], name, minimum=1)
              for name in ('n_env', 'rollout_len', 'minibatch_size', 'ppo_epochs')}
    decisions = values['n_env'] * values['rollout_len']
    if decisions % values['minibatch_size']:
        raise ValueError('Saved PPO cadence must divide into whole minibatches.')
    return values


def _optimizer_config_signature(cfg):
    """Fingerprint every saved learning setting, not only minibatch cadence."""
    return hashlib.sha256(json.dumps(_json_safe(dict(cfg)),sort_keys=True,
        separators=(',', ':'),allow_nan=False).encode()).hexdigest()


def _optimizer_work_counts(cfg, metrics):
    """Validate actual raw optimizer counters before accepting a candidate."""
    cadence = _optimizer_cadence(cfg)
    planned = cadence['ppo_epochs'] * (cadence['n_env'] * cadence['rollout_len'] // cadence['minibatch_size'])
    try:
        counts = {target: _optimizer_exact_integer(metrics[source], source)
            for source, target in (('attempted_updates', 'attempted'),
                ('accepted_updates', 'accepted'), ('rejected_updates', 'rejected'))}
        kl = _optimizer_exact_integer(metrics['kl_early_stop'], 'kl_early_stop', allow_bool=True)
    except KeyError as exc:
        raise ValueError('Missing optimizer work metric: ' + str(exc)) from exc
    if not 1 <= counts['accepted'] == counts['attempted'] <= planned or counts['rejected'] != 0:
        raise ValueError('Optimizer counts require1<=accepted==attempted<=planned and rejected=0.')
    if kl not in (0, 1) or counts['attempted'] < planned and not kl:
        raise ValueError('Skipped optimizer slots require an exact KL-stop flag.')
    return dict(planned=planned, **counts, skipped=planned-counts['attempted'],
        attempted_fraction=counts['attempted']/planned, accepted_fraction=counts['accepted']/planned,
        skipped_fraction=(planned-counts['attempted'])/planned, kl_early_stop=bool(kl))


def _optimizer_journal_rows(history, *, ignored=None, current_update=None):
    """Retain actual journaled work; a trainer migration starts a new lineage."""
    start = max((i+1 for i, row in enumerate(history)
                 if row.get('kind') == 'trainer_upgrade'), default=0)
    rows = []
    ignored = [] if ignored is None else ignored
    for index, row in enumerate(history[start:],start):
        if row.get('kind') in ('optimizer_health_stop','optimizer_health_acknowledgement'):
            # A recorded stop/explicit recovery boundary never stitches an old
            # partial health window to later optimizer work. Totals stay intact.
            if rows:
                rows[-1]['continuity_break_after']=True
            if row.get('kind')=='optimizer_health_acknowledgement' and not (
                    type(row.get('optimizer_health_version')) is int and row['optimizer_health_version']==1
                    and type(row.get('stop_update')) is int and row['stop_update']>=1
                    and isinstance(row.get('stop_id'),str) and len(row['stop_id'])==64
                    and all(c in '0123456789abcdef' for c in row['stop_id'])
                    and isinstance(row.get('config_signature'),str) and len(row['config_signature'])==64
                    and all(c in '0123456789abcdef' for c in row['config_signature'])
                    and isinstance(row.get('reason'),str) and len(row['reason'].strip())>=8):
                ignored.append(dict(history_index=index,kind='optimizer_health_acknowledgement',
                    reason='Ignored malformed or unrecognized optional health acknowledgement.'))
            continue
        if row.get('kind') != 'optimizer_work':
            continue
        try:
            if (type(row.get('optimizer_work_version')) is not int or row['optimizer_work_version'] != 1
                    or not isinstance(row.get('cadence'), dict)):
                raise ValueError('Unsupported or malformed optimizer-work journal.')
            signature=row.get('config_signature')
            if (not isinstance(signature,str) or len(signature)!=64
                    or any(c not in '0123456789abcdef' for c in signature)):
                raise ValueError('Missing or malformed full saved-config signature.')
            update = _optimizer_exact_integer(row.get('update'), 'journal update', minimum=1)
            if current_update is not None and update > current_update:
                raise ValueError('Optional optimizer journal runs ahead of the saved training state.')
            if rows and update <= rows[-1]['update']:
                raise ValueError('Optional optimizer journal update ID is duplicate or out of order.')
            restored = _optimizer_work_counts(row['cadence'], dict(
                attempted_updates=row.get('attempted'), accepted_updates=row.get('accepted'),
                rejected_updates=row.get('rejected'), kl_early_stop=row.get('kl_early_stop')))
            for name in ('planned', 'attempted', 'accepted', 'skipped', 'rejected'):
                actual = _optimizer_exact_integer(row.get(name), 'journal '+name)
                if actual != restored[name]:
                    raise ValueError('Optional journal counts do not match its cadence.')
        except (KeyError,TypeError,ValueError,OverflowError) as exc:
            ignored.append(dict(history_index=index,kind='optimizer_work',reason=str(exc)))
            if rows:
                rows[-1]['continuity_break_after']=True
            continue
        rows.append(dict(restored, kind='optimizer_work', optimizer_work_version=1,
                         update=update, cadence=_optimizer_cadence(row['cadence']),config_signature=signature))
    return rows


def _optimizer_health(rows):
    """Three consecutive same-cadence KL stops below25% trigger a safe stop."""
    window = rows[-3:]
    planned, accepted = sum(row['planned'] for row in window), sum(row['accepted'] for row in window)
    consecutive = (len(window) == 3 and all(window[i]['update']+1 == window[i+1]['update']
                    for i in range(2)) and all(row['cadence'] == window[0]['cadence']
                    and row['config_signature']==window[0]['config_signature'] for row in window)
                    and not any(row.get('continuity_break_after',False) for row in window[:-1]))
    all_kl = bool(consecutive and all(row['kl_early_stop'] for row in window))
    fraction = accepted/planned if planned else 0.0
    return dict(window_rollouts=len(window), planned=planned, accepted=accepted,
        aggregate_accepted_fraction=fraction, consecutive_same_cadence=bool(consecutive),
        all_three_kl_stopped=all_kl, should_stop=bool(all_kl and 4*accepted < planned))


def _optimizer_totals(rows):
    """Actual journal coverage only; never infer missing historic minibatches."""
    totals = {name: sum(row[name] for row in rows)
              for name in ('planned', 'attempted', 'accepted', 'skipped', 'rejected')}
    planned = totals['planned']
    return dict(totals, journaled_ppo_rollouts=len(rows),
        kl_stopped_rollouts=sum(bool(row['kl_early_stop']) for row in rows),
        attempted_fraction=totals['attempted']/planned if planned else 0.0,
        accepted_fraction=totals['accepted']/planned if planned else 0.0,
        skipped_fraction=totals['skipped']/planned if planned else 0.0,
        first_journaled_update=rows[0]['update'] if rows else None,
        last_journaled_update=rows[-1]['update'] if rows else None,
        interpretation='Observed logged optimizer minibatches; prior or missing updates are not inferred as lifetime work.')


def _optimizer_resume_journal(history, cfg, current_update, acknowledgement=None):
    """Use checkpoint history; require an explicit matching recovery after a stop."""
    current_update = _optimizer_exact_integer(current_update, 'current update')
    ignored=[]
    rows = _optimizer_journal_rows(history,ignored=ignored,current_update=current_update)
    start = max((i+1 for i, row in enumerate(history)
                 if row.get('kind') == 'trainer_upgrade'), default=0)
    stopped=None
    stopped_index=None
    signature=_optimizer_config_signature(cfg)
    for index,row in enumerate(history[start:],start):
        if row.get('kind')!='optimizer_health_stop':
            continue
        if (type(row.get('optimizer_health_version')) is int and row['optimizer_health_version']==1
                and row.get('config_signature')==signature and isinstance(row.get('reason'),str)
                and len(row['reason'].strip())>=8
                and isinstance(row.get('stop_id'),str) and len(row['stop_id'])==64
                and all(c in '0123456789abcdef' for c in row['stop_id'])
                and type(row.get('update')) is int and 1<=row['update']<=current_update):
            stopped=row
            stopped_index=index
        else:
            ignored.append(dict(history_index=index,kind='optimizer_health_stop',
                reason='Ignored unrecognized, malformed or different-config optional health-stop record.'))
    if ignored:
        history.append(dict(kind='optimizer_journal_notice',update=current_update,
            reason='Optional older health records ignored; rolling window reset across uncertain coverage.',
            ignored_records=ignored))
    if stopped is None:
        if acknowledgement is not None:
            raise ValueError('Optimizer health acknowledgement is stale: no matching current stop.')
        return rows
    acknowledged = any(row.get('kind')=='optimizer_health_acknowledgement'
        and type(row.get('optimizer_health_version')) is int and row['optimizer_health_version']==1
        and row.get('stop_id')==stopped['stop_id'] and type(row.get('stop_update')) is int
        and row['stop_update']==stopped['update'] and row.get('config_signature')==signature
        and isinstance(row.get('reason'),str) and len(row['reason'].strip())>=8
        for row in history[stopped_index+1:])
    if acknowledgement is not None:
        if (not isinstance(acknowledgement,dict)
                or type(acknowledgement.get('stop_update')) is not int
                or acknowledgement['stop_update']!=stopped['update']
                or acknowledgement.get('stop_id')!=stopped['stop_id']
                or acknowledgement.get('config_signature')!=signature
                or not isinstance(acknowledgement.get('reason'),str)
                or len(acknowledgement['reason'].strip())<8):
            raise ValueError('Optimizer health acknowledgement must match the latest stop ID/update/config and document its reviewed fix.')
        if not acknowledged:
            if current_update!=stopped['update']:
                raise ValueError('A new acknowledgement must recover exactly the stopped accepted update.')
            history.append(dict(kind='optimizer_health_acknowledgement',optimizer_health_version=1,
                stop_id=stopped['stop_id'],stop_update=stopped['update'],config_signature=signature,
                update=current_update,reason=acknowledgement['reason'].strip(),
                scope='Reset rolling health window only; model/optimizer/RNG/counters/config unchanged.'))
            if rows:
                rows[-1]['continuity_break_after']=True
            acknowledged=True
    if not acknowledged:
        raise OptimizerHealthStop('This native checkpoint records an optimizer health stop. '
            'Inspect its reason and recovery checkpoint; do not rerun unchanged. '
            'After a reviewed fix, supply a matching explicit RUN_CFG optimizer_health_acknowledgement. '
            + str(stopped.get('checkpoint_path',stopped.get('checkpoint_tag','optimizer_health_stop'))))
    return rows


def _optimizer_work_record(previous, candidate, cfg, metrics):
    work = _optimizer_work_counts(cfg, metrics)
    old_update=_optimizer_exact_integer(previous.updates,'previous PPO update')
    new_update=_optimizer_exact_integer(candidate.updates,'candidate PPO update')
    old_steps=_optimizer_exact_integer(previous.env_steps,'previous lifetime decisions')
    new_steps=_optimizer_exact_integer(candidate.env_steps,'candidate lifetime decisions')
    if new_update != old_update+1:
        raise ValueError('Accepted PPO candidate must advance exactly one update.')
    if new_steps != old_steps + int(cfg['n_env'])*int(cfg['rollout_len']):
        raise ValueError('Accepted PPO candidate lifetime decisions are inconsistent.')
    return dict(work, kind='optimizer_work', optimizer_work_version=1,
        update=new_update, env_steps=new_steps, cadence=_optimizer_cadence(cfg),
        config_signature=_optimizer_config_signature(cfg))


def train_segment(state, cfg, history, run_cfg, cumulative_seconds=0.):
    """Run one wall-clock budget, checkpoint periodically, finally, and on interruption.

    Stop/checkpoint occurs at a completed PPO update or benchmark boundary. Colab
    cannot guarantee recovery from forced VM termination; only completed Drive
    checkpoints survive. In-memory state is replaced only after finite metrics.
    """
    globals()['RECOVERED_TRAINING_STATE'] = state
    globals()['LAST_RUN_RECOVERY'] = (state, history, cumulative_seconds)
    folder = Path(run_cfg['checkpoint_dir'])
    folder.mkdir(parents=True, exist_ok=True)
    probe = folder / '.write_test'
    probe.write_text('ok')
    probe.unlink()
    duration = float(run_cfg.get('duration_hours', 3.)) * 3600
    checkpoint_period = float(run_cfg.get('checkpoint_hours', 1.5)) * 3600
    metric_period = float(run_cfg.get('metrics_every_seconds', 30.))
    eval_period = float(run_cfg.get('eval_every_minutes', 30.)) * 60
    if not all(np.isfinite(x) for x in (duration, checkpoint_period, metric_period, eval_period)):
        raise ValueError('All time intervals must be finite.')
    if min(duration, checkpoint_period, metric_period) <= 0:
        raise ValueError('Duration, checkpoint interval, and metric interval must be positive.')
    if int(run_cfg.get('bootstrap_updates', 16)) < 0:
        raise ValueError('bootstrap_updates must be nonnegative.')
    optimizer_rows = _optimizer_resume_journal(history, cfg, state.updates,
        run_cfg.get('optimizer_health_acknowledgement'))
    optimizer_totals = _optimizer_totals(optimizer_rows)
    step_factory = globals().get('gpu_train_step', make_train_step)
    step_fn = step_factory(cfg)
    bootstrap_fn = gpu_bootstrap_step(cfg)
    bootstrap_target = int(run_cfg.get('bootstrap_updates', 16))
    start = time.monotonic()
    next_save = start + checkpoint_period
    next_metrics = start
    next_eval = start + eval_period if eval_period > 0 else float('inf')
    last_path = None
    last_metric_time = start
    last_metric_steps = int(state.env_steps)
    best_score = _best_development_score(history)
    run_metadata = dict(training_revision=TRAINING_REVISION,
                        n_env=int(cfg['n_env']), rollout_len=int(cfg['rollout_len']),
                        minibatch_size=int(cfg['minibatch_size']), width=int(cfg['width']))
    latest = dict(run_metadata, update=int(state.updates), env_steps=int(state.env_steps))
    latest.update(_gpu_telemetry())
    print(f"Trainer: {TRAINING_REVISION} | parallel games {cfg['n_env']} × rollout {cfg['rollout_len']} | "
          f"minibatch {cfg['minibatch_size']} | network width {cfg['width']}", flush=True)
    for label, value in _gpu_health_rows(latest):
        print(f'{label}: {value}', flush=True)
    print('Starting training. First JAX compilation can take several minutes; later updates reuse it.', flush=True)
    caught = None
    upgrade_index = max((i for i, row in enumerate(history)
                         if row.get('kind') == 'trainer_upgrade'), default=-1)
    baseline_pending = (upgrade_index >= 0 and not bool(state.bootstrap_complete)
                        and not any(row.get('kind') == 'evaluation'
                                    and row.get('stage') == 'before_teacher_refresh'
                                    for row in history[upgrade_index + 1:]))
    def development_benchmark(stage):
        nonlocal best_score, next_eval
        gpu_admit_workload(cfg, "development benchmark")
        print(f'Running full-match development benchmark ({stage})...', flush=True)
        report = evaluate_policy(state.params, cfg, styles=[7],
            games_per_style=int(run_cfg.get('eval_games', 32)),
            seed=int(run_cfg.get('development_seed', 81073)),
            max_hands=int(run_cfg.get('eval_max_hands', 160)))
        rows = report.to_dict(orient='records')
        # Report unresolved trials explicitly; never reward a capped game as a win.
        row = rows[0]
        selection_score = float(row.get('ci_low', row.get('win_rate', 0.)))
        if (not np.isfinite(selection_score) or row.get('unresolved', 0) != 0
                or row.get('illegal_actions', 0) != 0):
            selection_score = None
        record = dict(kind='evaluation', stage=stage, update=int(state.updates),
                      env_steps=int(state.env_steps), rows=rows,
                      selection_score=selection_score)
        record = _replace_nonfinite(record)
        history.append(record)
        _append_jsonl(folder, record)
        if selection_score is not None and selection_score > best_score and row.get('unresolved', 0) == 0:
            best_score = selection_score
            save_checkpoint(state, cfg, history, folder, cumulative_seconds+time.monotonic()-start,
                            tag='best_development', keep=1)
        display(report)
        next_eval = time.monotonic() + eval_period if eval_period > 0 else float('inf')

    try:
        while time.monotonic() - start < duration:
            if baseline_pending:
                # Preserve a benchmark of the transferred actor before refreshing
                # it. Persist the record so a resumed teacher stage cannot repeat it.
                development_benchmark('before_teacher_refresh')
                baseline_pending = False
                now = time.monotonic()
                last_path = save_checkpoint(state, cfg, history, folder,
                    cumulative_seconds + now - start, keep=run_cfg.get('keep_checkpoints', 3))
                while next_save <= now:
                    next_save += checkpoint_period
                continue
            if not bool(state.bootstrap_complete):
                if int(state.bootstrap_updates) < bootstrap_target:
                    gpu_admit_workload(cfg, "teacher update")
                    candidate, metrics = bootstrap_fn(state)
                    boot = _scalar_metrics(metrics)
                    _check_training_numerics(state, candidate, boot, 'bootstrap', history, folder)
                    state = candidate
                    record = dict(boot, kind='bootstrap')
                    history.append(record)
                    _append_jsonl(folder, record)
                    now = time.monotonic()
                    print(f"Teacher initialization {int(state.bootstrap_updates)}/{bootstrap_target}: "
                          f"loss {boot['bootstrap_loss']:.4f}, accuracy {boot['bootstrap_accuracy']:.1%}", flush=True)
                    if now >= next_save:
                        last_path = save_checkpoint(state, cfg, history, folder,
                            cumulative_seconds + now-start, keep=run_cfg.get('keep_checkpoints', 3))
                        while next_save <= now:
                            next_save += checkpoint_period
                else:
                    gpu_admit_workload(cfg, "finish teacher initialization")
                    state = finish_bootstrap(state, cfg)
                    last_path = save_checkpoint(state, cfg, history, folder,
                        cumulative_seconds + time.monotonic()-start, keep=run_cfg.get('keep_checkpoints', 3))
                    print('Teacher initialization complete; saved checkpoint and starting PPO.', flush=True)
                    if eval_period > 0:
                        next_eval = time.monotonic()
                continue
            gpu_admit_workload(cfg, "PPO update")
            candidate, metrics = step_fn(state)
            scalars = _scalar_metrics(metrics)  # Synchronizes accelerator, so timers measure actual work.
            _check_training_numerics(state, candidate, scalars, 'PPO', history, folder)
            if scalars.get('illegal_rate', 0.) > 0:
                raise RuntimeError('An illegal policy action was detected; stop and inspect the engine.')
            # Validate raw integer work counts and exact counters before commit.
            work = _optimizer_work_record(state, candidate, cfg, metrics)
            proposed_rows = optimizer_rows + [work]
            health = _optimizer_health(proposed_rows)
            state = candidate
            optimizer_rows = proposed_rows
            optimizer_totals = _optimizer_totals(optimizer_rows)
            history.append(work)  # Every accepted PPO rollout, independent of the dashboard timer.
            if health['should_stop']:
                reason = 'Three consecutive KL-stopped PPO rollouts accepted fewer than25% of planned optimizer slots.'
                stopped = dict(kind='optimizer_health_stop', update=int(state.updates),
                    optimizer_health_version=1, config_signature=_optimizer_config_signature(cfg),
                    stop_id=hashlib.sha256(f'{int(state.updates)}:{state.env_steps}:{time.time_ns()}'.encode()).hexdigest(),
                    env_steps=int(state.env_steps), reason=reason, optimizer_health=health,
                    optimizer_totals=optimizer_totals, checkpoint_tag='optimizer_health_stop',
                    automatic_retry=False)
                history.append(stopped)
                # Unique health-tag saves have no retention cleanup. Commit the valid
                # candidate above, force durability, then stop before any benchmark.
                last_path = save_checkpoint(state, cfg, history, folder,
                    cumulative_seconds + time.monotonic()-start, tag='optimizer_health_stop', keep=None)
                stopped['checkpoint_path'] = str(last_path)
                globals()['LAST_OPTIMIZER_HEALTH_STOP'] = stopped
                try:
                    _append_jsonl(folder, work)
                    _append_jsonl(folder, stopped)
                except OSError as log_error:
                    stopped['journal_error'] = f'{type(log_error).__name__}: {log_error}'
                raise OptimizerHealthStop(reason + ' Accepted state saved at ' + str(last_path)
                    + '. Inspect the journal; do not rerun the unchanged trainer.')
            _append_jsonl(folder, work)
            scalars['nil_success_rate'] = scalars.get('nil_successes', 0.) / max(scalars.get('nil_attempts', 0.), 1.)
            scalars['dn_success_rate'] = scalars.get('dn_successes', 0.) / max(scalars.get('dn_attempts', 0.), 1.)
            now = time.monotonic()
            if now >= next_save:
                last_path = save_checkpoint(state, cfg, history, folder,
                    cumulative_seconds + now - start, keep=run_cfg.get('keep_checkpoints', 3))
                print('Checkpoint saved:', last_path.name, flush=True)
                while next_save <= now:
                    next_save += checkpoint_period
            if now >= next_metrics:
                steps = int(state.env_steps)
                latest = dict(scalars, kind='train', update=int(state.updates), env_steps=steps,
                              cumulative_hours=(cumulative_seconds + now - start)/3600,
                              decisions_per_second=(steps-last_metric_steps)/max(now-last_metric_time, 1e-9))
                latest['optimizer_work_totals'] = dict(optimizer_totals)
                latest.update(run_metadata)
                latest.update(_gpu_telemetry())
                history.append(latest)
                _append_jsonl(folder, latest)
                show_training_metrics(history, latest, now-start, duration, last_path)
                last_metric_steps, last_metric_time = steps, now
                next_metrics = now + metric_period
            # Avoid beginning a benchmark at the end of a segment or just before a due save.
            if (now >= next_eval and now + 120 < start + duration and now + 120 < next_save):
                development_benchmark('development')
    except KeyboardInterrupt:
        print('Interrupted. Saving the most recent completed update...', flush=True)
    except Exception as exc:
        caught = exc
        history.append(dict(kind='runtime_stop', update=int(state.updates), env_steps=int(state.env_steps),
            reason=f'{type(exc).__name__}: {exc}', automatic_retry=False))
        print(f'Training stopped: {type(exc).__name__}: {exc}\nSaving last completed update...', flush=True)
    finally:
        elapsed = time.monotonic() - start
        globals()['RECOVERED_TRAINING_STATE'] = state
        globals()['LAST_RUN_RECOVERY'] = (state, history, cumulative_seconds + elapsed)
        try:
            last_path = save_checkpoint(state, cfg, history, folder,
                                       cumulative_seconds + elapsed, keep=run_cfg.get('keep_checkpoints', 3))
        except Exception as save_error:
            # If Drive temporarily fails, retain a downloadable local recovery file.
            emergency_dir = Path(tempfile.gettempdir()) / 'modified_spades_emergency'
            last_path = save_checkpoint(state, cfg, history, emergency_dir,
                                       cumulative_seconds + elapsed, tag='emergency', keep=2)
            print(f'Drive checkpoint failed: {save_error}. Local recovery file: {last_path}. '
                  'Copy/download this file before disconnecting; local storage is temporary.', flush=True)
            if caught is None:
                caught = save_error
        continuation = ('Run this training cell again for another healthy segment.' if caught is None else
                        'Inspect the error and recovery checkpoint before continuing; do not rerun unchanged.')
        print(f'Saved {last_path}\nSegment wall time: {elapsed/3600:.3f} h. ' + continuation, flush=True)
    if caught is not None:
        # Preserve the recovered state for the next cell even if the caller receives an exception.
        globals()['RECOVERED_TRAINING_STATE'] = state
        raise caught
    return state, history, cumulative_seconds + elapsed


def _replace_nonfinite(obj):
    if isinstance(obj, dict):
        return {k: _replace_nonfinite(v) for k, v in obj.items()}
    if isinstance(obj, (tuple, list)):
        return [_replace_nonfinite(x) for x in obj]
    if isinstance(obj, (float, np.floating)) and not np.isfinite(obj):
        return None
    return _json_safe(obj)


# The configuration cell checks the function object, catching a stale runner
# left in a live Colab session after only some new definition cells were run.
train_segment.training_revision = 'strong-teacher-hand-gae-v3'

train_segment.runtime_safety_revision = 'checkpoint-memory-safety-v2'


## 9 · Verification

The built-in checks use separate fixtures. They verify the unchanged rules, real PPO masks/gradients/returns, numerical rollback, teacher information boundaries, play-guidance gradients, terminal and sampling-reset bootstraps, resized decision counters, legacy checkpoint loading, and exact next-update continuation. Extended switches add more compilation and injected-failure tests.

CPU development experiments compare policies on complete paired matches and multiple deal seeds, including the user's development seed 81073. Longer coherent teaching improved small-model strength, and continuing 0.3 play guidance preserved more strength than bidding-only guidance in controlled small-model trials across multiple deal seeds. A separate production-width teacher trial improved from 25/128 to 43/128 adaptive wins as teaching increased from 262,144 to 2,097,152 decisions; every match resolved with no illegal actions. These experiments do not prove expert play or predict the result of upgrading an unseen checkpoint. The notebook's before/after benchmarks measure that on your actual policy.

GPU profile selection has mock tests and real CPU executable-compilation checks. Actual T4 allocation, throughput, and the best batch size are measured in Colab; no measured 14 GiB GPU result is claimed here.

In [ ]:
# Independent rules checks. This cell is self-contained after the engine cells.
def score_hand_numpy(bids, tricks, bags):
    """Independent NumPy reference scorer; accepts (..., 4)/(..., 2) arrays."""
    import numpy as _np
    bids, tricks, bags = map(_np.asarray, (bids, tricks, bags))
    deltas, carries, successes, extras = [], [], [], []
    for team in range(2):
        seats = [team, team + 2]
        b, t = bids[..., seats], tricks[..., seats]
        normal = b > 0
        value = _np.where(b == -1, 200, _np.where(b == 0, 100, 10 * b)).sum(-1)
        both_normal = normal.all(-1)
        made_combined = t.sum(-1) >= b.sum(-1)
        made_individual = _np.where(normal, t >= b, t == 0).all(-1)
        success = _np.where(both_normal, made_combined, made_individual)
        over = _np.where(both_normal, t.sum(-1) - b.sum(-1),
                         _np.where(normal, t - b, 0).sum(-1))
        added = _np.where(success, over, 0)
        bag_total = bags[..., team] + added
        delta = _np.where(success, value + added, -value) - 100 * (bag_total // 10)
        deltas.append(delta)
        carries.append(bag_total % 10)
        successes.append(success)
        extras.append(added)
    return tuple(_np.stack(x, -1) for x in (deltas, carries, successes, extras))


def run_self_tests(quick=True):
    """Test rules, hidden information, JIT/vmap, and complete game transitions.

    quick=True still enumerates every partner bid pair and every feasible partner
    trick allocation. quick=False adds every starting bag count and longer scans.
    Does not train or modify the live training/checkpoint state.
    """
    import time as _time
    import numpy as _np
    _started = _time.monotonic()
    _checks = 0
    _groups = []
    _numeric_cases = 0

    def check(condition, message):
        nonlocal _checks
        _checks += 1
        if not bool(_np.all(_np.asarray(condition))):
            raise AssertionError(message)

    def equal(actual, expected, message):
        nonlocal _checks
        _checks += 1
        _np.testing.assert_array_equal(_np.asarray(actual), _np.asarray(expected), err_msg=message)

    def group(name):
        _groups.append(name)
        print(f"  PASS {name}", flush=True)

    print("Checking modified Spades rules and information boundaries…", flush=True)
    key = jax.random.PRNGKey(123456)
    base = reset(key)
    check(_np.asarray(base.hands).shape == (4, 52), "Hands must use the documented card encoding")
    equal(base.hands.sum(0), _np.ones(52), "Every card belongs to exactly one hand")
    equal(base.hands.sum(1), [13] * 4, "Deal 13 cards per player")
    equal(base.scores, [0, 0], "Initial scores")
    equal(base.bags, [0, 0], "Initial bags")
    check(not _np.asarray(base.eligible).any(), "Double Nil is forbidden at a tied start")
    check(_np.isfinite(_np.asarray(observe(base))).all(), "Actor observation must be finite")
    check(_np.isfinite(_np.asarray(critic_observe(base))).all(), "Critic observation must be finite")
    group("initial deal and observation shapes")

    # Enumerate all 15×15 ordered bid pairs, 105 possible team trick splits,
    # and bag values at rollover boundaries (all ten values in the full suite).
    pairs = _np.array([(a, b) for a in range(-1, 14) for b in range(-1, 14)], _np.int32)
    splits = _np.array([(a, b) for a in range(14) for b in range(14 - a)], _np.int32)
    bag_values = _np.array([0, 8, 9] if quick else list(range(10)), _np.int32)
    pi, ti, bi = _np.meshgrid(_np.arange(len(pairs)), _np.arange(len(splits)),
                            _np.arange(len(bag_values)), indexing="ij")
    chosen_b, chosen_t = pairs[pi.ravel()], splits[ti.ravel()]
    sb = _np.stack((chosen_b[:, 0], chosen_b[:, 1], chosen_b[:, 1], chosen_b[:, 0]), -1)
    remaining = 13 - chosen_t.sum(-1)
    st = _np.stack((chosen_t[:, 0], remaining, chosen_t[:, 1], _np.zeros(len(sb), _np.int32)), -1)
    sg = _np.stack((bag_values[bi.ravel()], 9 - bag_values[bi.ravel()]), -1)
    expected = score_hand_numpy(sb, st, sg)
    actual = jax.jit(jax.vmap(score_hand))(jnp.asarray(sb), jnp.asarray(st), jnp.asarray(sg))
    for a, e, label in zip(actual, expected, ("points", "bags", "contract success", "bags added")):
        equal(a, e, f"Exhaustive independent scorer: {label}")
    _numeric_cases += len(sb) * 2
    # Explicit extreme rollover: 9 carried bags plus 12 newly won bags = 2 penalties.
    explicit_b = _np.array([0, 13, 1, 13], _np.int32)
    explicit_t = _np.array([0, 0, 13, 0], _np.int32)
    explicit = score_hand(explicit_b, explicit_t, _np.array([9, 0], _np.int32))
    equal(explicit[0], [-78, -260], "Two bag thresholds in one successful Nil hand")
    equal(explicit[1], [1, 0], "Bag carry after two thresholds")
    group(f"independent scoring oracle ({len(sb):,} hands; all Nil/Double Nil combinations)")

    resolver_cases = [
        ([499, 499], 0, False, -1, 0), ([500, 499], 0, True, 0, 0),
        ([499, 500], 0, True, 1, 0), ([600, 510], 0, True, 0, 0),
        ([510, 600], 0, True, 1, 0), ([500, 500], 0, False, -1, 1),
        ([-10000, -20000], 0, False, -1, 0), ([400, 399], 1, True, 0, 1),
        ([-500, -499], 1, True, 1, 1), ([0, 0], 1, False, -1, 2),
        ([0, 0], 9, False, -1, 10), ([0, 0], 10, True, 2, 10),
    ]
    rs = jnp.asarray([x[0] for x in resolver_cases], jnp.int32)
    rt = jnp.asarray([x[1] for x in resolver_cases], jnp.int32)
    rd, rw, rn = jax.jit(jax.vmap(resolve_match))(rs, rt)
    equal(rd, [x[2] for x in resolver_cases], "500-point/tiebreak termination")
    equal(rw, [x[3] for x in resolver_cases], "Match winner and ten-extra-hand draw")
    # Once done, the next tiebreak counter is immaterial; check active continuations only.
    ongoing = _np.array([not x[2] for x in resolver_cases])
    equal(_np.asarray(rn)[ongoing], _np.array([x[4] for x in resolver_cases])[ongoing],
          "Tiebreak counter advances exactly once per additional hand")
    group("500-point threshold, negative scores, and ten-hand tiebreak limit")

    threshold_scores = jnp.array([[0, 149], [0, 150], [0, 151], [149, 0],
                                 [150, 0], [-500, -350], [500, 500]], jnp.int32)
    threshold_hands = jax.jit(jax.vmap(lambda scores: deal_next(
        base._replace(scores=scores, bags=jnp.array([8, 9], jnp.int32)), key)))(threshold_scores)
    equal(threshold_hands.eligible,
          [[False, False], [True, False], [True, False], [False, False],
           [False, True], [True, False], [False, False]],
          "Double Nil becomes available at exactly a 150-point deficit")
    equal(threshold_hands.scores, threshold_scores, "Redealing preserves scores including negative scores")
    equal(threshold_hands.bags, _np.tile([8, 9], (7, 1)), "Redealing preserves accumulated bags")
    equal(threshold_hands.dealer, _np.full(7, (int(base.dealer) + 1) % 4), "Redealing rotates dealer")
    equal(threshold_hands.hand_number, _np.full(7, int(base.hand_number) + 1), "Hand index advances")
    group("exact Double Nil eligibility boundary and cumulative redeal state")

    # Actor observations must not encode hidden hands, blind own cards, or sealed selections.
    blind_alt = base._replace(hands=jnp.roll(base.hands, 1, axis=0),
                             pending_pass=jnp.array([1, 2, 3, 4], jnp.int32),
                             passed=jnp.array([5, 6, 7, 8], jnp.int32),
                             received=jnp.array([9, 10, 11, 12], jnp.int32))
    equal(observe(base), observe(blind_alt), "Blind observations reveal no dealt or passing cards")
    revealed = base._replace(turn=jnp.int32(8))
    who = int(actor(revealed))
    other = [(who + 1) % 4, (who + 2) % 4]
    swapped_hidden = revealed.hands.at[other[0]].set(revealed.hands[other[1]]).at[other[1]].set(revealed.hands[other[0]])
    hidden_alt = revealed._replace(hands=swapped_hidden,
                                  pending_pass=jnp.arange(4, dtype=jnp.int32) + 20)
    equal(observe(revealed), observe(hidden_alt), "Actor observations reveal no other player's hand or pending choice")
    # During the bidding phase every non-DN player must be able to bid every 0..13.
    mask = _np.asarray(legal_mask(revealed))
    equal(_np.flatnonzero(mask), _np.arange(52, 66), "Every normal bid including zero and thirteen must be legal")
    # Blind and DN-skip mask checks do not depend on the actual random deal.
    blind_mask = _np.asarray(legal_mask(base))
    equal(_np.flatnonzero(blind_mask), [66], "Ineligible players must decline blind bidding")
    dn_state = revealed._replace(bids=revealed.bids.at[who].set(-1))
    equal(_np.flatnonzero(_np.asarray(legal_mask(dn_state))), [66], "Double Nil players skip normal bidding")
    group("blind information privacy, hidden hands, and bid legality")

    # Use dealer 3 so phase order is seats 0,1,2,3. Seat 0 declares DN.
    exchange = base._replace(dealer=jnp.int32(3), turn=jnp.int32(0),
                             scores=jnp.array([-200, 0], jnp.int32),
                             eligible=jnp.array([True, False]))
    step_jit = jax.jit(step)
    check(bool(legal_mask(exchange)[67]), "An eligible trailing player may declare Double Nil")
    for declaration in (67, 66, 66, 66):
        exchange, _, _, _ = step_jit(exchange, jnp.int32(declaration), key)
    initial_hands = _np.asarray(exchange.hands).copy()
    c0 = int(_np.flatnonzero(initial_hands[0])[0])
    c2 = int(_np.flatnonzero(initial_hands[2])[-1])
    exchange, _, _, _ = step_jit(exchange, jnp.int32(c0), key)
    equal(exchange.hands, initial_hands, "The first selected passing card stays sealed until both selections exist")
    exchange, _, _, _ = step_jit(exchange, jnp.int32(66), key)
    # Seat 2 is about to select. A different sealed seat-0 card is invisible to it.
    alternate_c0 = int(_np.flatnonzero(initial_hands[0])[1])
    changed_sealed = exchange._replace(pending_pass=exchange.pending_pass.at[0].set(alternate_c0))
    equal(observe(exchange), observe(changed_sealed), "The partner cannot inspect the first sealed selection")
    exchange, _, _, _ = step_jit(exchange, jnp.int32(c2), key)
    exchange, _, _, _ = step_jit(exchange, jnp.int32(66), key)
    expected_hands = initial_hands.copy()
    expected_hands[0, c0] = expected_hands[2, c2] = False
    expected_hands[0, c2] = expected_hands[2, c0] = True
    equal(exchange.hands, expected_hands, "Exactly one simultaneous partner swap")
    equal(exchange.hands.sum(1), [13] * 4, "Passing preserves 13 cards in every hand")
    check(int(exchange.turn) == 8, "Normal bidding starts after all four passing polls")
    check(int(exchange.bids[0]) == -1, "Double Nil remains locked after exchange")
    group("simultaneous sealed partner exchange")

    # Both partners may declare Double Nil; they still exchange only once.
    double_exchange = base._replace(dealer=jnp.int32(3), turn=jnp.int32(0),
                                    scores=jnp.array([-200, 0], jnp.int32),
                                    eligible=jnp.array([True, False]))
    for action in (67, 66, 67, 66, c0, 66, c2, 66):
        double_exchange, _, _, info = step_jit(double_exchange, jnp.int32(action), key)
        check(not bool(info["illegal"]), "Both eligible partners can declare and pass for Double Nil")
    equal(double_exchange.hands, expected_hands, "Two Double Nils still exchange exactly one card each")
    equal(double_exchange.bids, [-1, -2, -1, -2], "Both Double Nil bids are retained")
    group("two simultaneous Double Nil declarations")

    # Legal play masks: leading a spade is always allowed, and following is compulsory.
    hand = jnp.zeros((4, 52), jnp.bool_).at[0, jnp.array([0, 12, 13, 39, 51])].set(True)
    play_state = base._replace(turn=jnp.int32(12), current_player=jnp.int32(0),
                              leader=jnp.int32(0), trick_pos=jnp.int32(0), hands=hand,
                              bids=jnp.ones(4, jnp.int32))
    equal(_np.flatnonzero(_np.asarray(legal_mask(play_state))), [0, 12, 13, 39, 51],
          "Unbroken spades may be led")
    following = play_state._replace(turn=jnp.int32(13), trick_pos=jnp.int32(1),
                                   table=jnp.array([5, -1, -1, -1], jnp.int32),
                                   table_players=jnp.array([3, -1, -1, -1], jnp.int32), leader=jnp.int32(3))
    equal(_np.flatnonzero(_np.asarray(legal_mask(following))), [0, 12], "Following suit excludes all off-suit cards")
    void_hand = hand.at[0, jnp.array([0, 12])].set(False)
    equal(_np.flatnonzero(_np.asarray(legal_mask(following._replace(hands=void_hand)))), [13, 39, 51],
          "A void player may discard or trump")
    _, _, _, invalid_info = step_jit(base, jnp.int32(0), key)
    check(bool(invalid_info["illegal"]), "Invalid actions are diagnosed")
    group("follow-suit enforcement, free spade leads, and invalid-action diagnostics")

    # A separate rank/suit oracle checks fourth-card resolution, not engine helpers.
    rng = _np.random.default_rng(8675309)
    n_tricks = 256 if quick else 4096
    cards = _np.stack([rng.choice(52, 4, replace=False) for _ in range(n_tricks)]).astype(_np.int32)
    cards[:4] = [[12, 25, 38, 1], [12, 39, 25, 38], [51, 50, 49, 48], [13, 25, 24, 12]]
    led = cards[:, 0] // 13
    strength = _np.where(cards // 13 == 3, 100 + cards % 13,
                         _np.where(cards // 13 == led[:, None], cards % 13, -1))
    winners = strength.argmax(-1)

    def finish_fixture(c):
        fixture = base._replace(turn=jnp.int32(15), current_player=jnp.int32(3),
                                leader=jnp.int32(0), trick_pos=jnp.int32(3),
                                hands=jnp.zeros((4, 52), jnp.bool_).at[3, c[3]].set(True),
                                table=jnp.array([c[0], c[1], c[2], -1], jnp.int32),
                                table_players=jnp.array([0, 1, 2, -1], jnp.int32),
                                bids=jnp.ones(4, jnp.int32))
        next_state, _, _, info = step(fixture, c[3], key)
        return next_state.current_player, next_state.tricks, next_state.trick_pos, info["illegal"]

    tw, tt, tp, illegal = jax.jit(jax.vmap(finish_fixture))(jnp.asarray(cards))
    equal(tw, winners, "Independent trump/led-suit winner oracle")
    equal(tt, _np.eye(4, dtype=_np.int32)[winners], "Winner gains exactly one trick")
    equal(tp, _np.zeros(n_tricks), "A completed trick clears its position")
    check(~_np.asarray(illegal), "Every fixture's fourth card is legal")
    _numeric_cases += n_tricks
    group(f"independent trick-winner oracle ({n_tricks:,} tricks)")

    # Crossing 500 before a two-bag penalty is applied must NOT end the match.
    # Team 0's Nil+1 contract succeeds with 13 tricks and 9 carried bags:
    # 110 + 12 - 200 = -78 points, so 490 becomes 412.
    final_fixture = base._replace(
        turn=jnp.int32(63), current_player=jnp.int32(3), leader=jnp.int32(0),
        trick_pos=jnp.int32(3), hands=jnp.zeros((4, 52), jnp.bool_).at[3, 26].set(True),
        table=jnp.array([51, 0, 13, -1], jnp.int32),
        table_players=jnp.array([0, 1, 2, -1], jnp.int32),
        bids=jnp.array([1, 1, 0, 1], jnp.int32), tricks=jnp.array([12, 0, 0, 0], jnp.int32),
        scores=jnp.array([490, 0], jnp.int32), bags=jnp.array([9, 0], jnp.int32))
    after_penalty, _, penalty_done, penalty_info = step_jit(final_fixture, jnp.int32(26), key)
    check(not bool(penalty_done), "Apply bag penalties before deciding who reaches 500")
    equal(penalty_info["final_scores"], [412, -20], "Final scoring includes both ten-bag penalties")
    equal(after_penalty.bags, [1, 0], "Remaining bags carry into the next hand")
    group("winning threshold is checked after bag penalties")

    # Scan full random trajectories on-device; verify invariants before every turn.
    # Returning small hand-final evidence allows scoring to be audited off-device.
    n_envs, n_steps = (4, 256) if quick else (16, 1024)
    keys = jax.random.split(jax.random.PRNGKey(9001), n_envs)
    batched = jax.vmap(reset)(keys)

    def trajectory_one(state, random_key):
        def tick(carry, unused):
            s, k = carry
            k, ka, ks = jax.random.split(k, 3)
            legal = legal_mask(s)
            a = jax.random.categorical(ka, jnp.where(legal, 0.0, -jnp.inf)).astype(jnp.int32)
            p = actor(s)
            played_count = jnp.maximum(s.turn - 12, 0)
            invariant = (jnp.all(s.hands.sum(0) + (s.played_by >= 0) == 1)
                         & (s.hands.sum() == 52 - played_count)
                         & (s.tricks.sum() == played_count // 4)
                         & (s.trick_pos == played_count % 4)
                         & jnp.any(legal))
            ns, reward, done, info = step(s, a, ks)
            score_continues = jnp.all(ns.scores == info["final_scores"])
            reset_ok = ((ns.turn == 0) & jnp.all(ns.hands.sum(1) == 13)
                        & jnp.all(ns.bids == -2) & jnp.all(ns.tricks == 0)
                        & jnp.all(ns.played_by == -1) & jnp.all(ns.played_at == -1))
            hand_transition = jnp.where(info["hand_done"],
                reset_ok & jnp.where(done, jnp.all(ns.scores == 0) & jnp.all(ns.bags == 0),
                                    (ns.dealer == (s.dealer + 1) % 4) & score_continues),
                (ns.turn == s.turn + 1) & (reward == 0))
            evidence = (invariant & hand_transition & (~info["illegal"]), info["hand_done"],
                        s.bids, s.tricks, s.bags, s.scores, s.table, s.table_players, p, a,
                        info["score_delta"], info["final_scores"], info["discount"], done)
            return (ns, k), evidence
        return jax.lax.scan(tick, (state, random_key), None, length=n_steps)

    _, evidence = jax.jit(jax.vmap(trajectory_one))(batched, keys)
    evidence = jax.device_get(evidence)
    good, ended, ebids, etricks, ebags, escores, etable, eplayers, ep, ea, edelta, efinal, ediscount, edone = evidence
    check(good, "Random trajectories preserve deck, trick counts, dealer rotation, cumulative scores, and legal actions")
    check(_np.isfinite(ediscount), "All discounts are finite")
    equal(ediscount[~ended], _np.ones_like(ediscount[~ended]), "Non-hand transitions have unit discount")
    equal(ediscount[edone], _np.zeros_like(ediscount[edone]), "Terminal games have zero bootstrap discount")
    final_indices = _np.argwhere(ended)
    for e, t in final_indices:
        tc = _np.asarray(etable[e, t]).copy()
        ps = _np.asarray(eplayers[e, t]).copy()
        tc[3], ps[3] = ea[e, t], ep[e, t]
        power = _np.where(tc // 13 == 3, 100 + tc % 13,
                          _np.where(tc // 13 == tc[0] // 13, tc % 13, -1))
        full_tricks = _np.asarray(etricks[e, t]).copy()
        full_tricks[ps[power.argmax()]] += 1
        oracle_delta = score_hand_numpy(ebids[e, t], full_tricks, ebags[e, t])[0]
        equal(edelta[e, t], oracle_delta, "Whole-hand integration agrees with independent scoring")
        equal(efinal[e, t], escores[e, t] + oracle_delta, "Final scores include all hand points and bag penalties")
    group(f"JIT/vmap random trajectories ({n_envs * n_steps:,} turns, {len(final_indices)} hands)")

    if "batch_step" in globals():
        # The optimized synchronized trainer engine must agree exactly with the
        # general asynchronous engine through every phase and a hand boundary.
        def compare_tick(carry, i):
            fast, slow = carry
            masks = jax.vmap(legal_mask)(fast)
            keys_i = jax.random.split(jax.random.fold_in(key, i), n_envs)
            actions = jax.vmap(lambda k, m: jax.random.categorical(
                k, jnp.where(m, 0.0, -jnp.inf)))(keys_i, masks).astype(jnp.int32)
            fast_result = batch_step(fast, actions, keys_i)
            slow_result = jax.vmap(step)(slow, actions, keys_i)
            matches = jax.tree_util.tree_leaves(jax.tree_util.tree_map(
                lambda a, b: jnp.all(a == b), fast_result, slow_result))
            return (fast_result[0], slow_result[0]), jnp.all(jnp.stack(matches))
        _, equivalent = jax.jit(lambda: jax.lax.scan(
            compare_tick, (batched, batched), jnp.arange(128)))()
        check(equivalent, "Synchronized batch_step agrees with vmap(step) for every state, reward, and info field")
        group("optimized batch engine agrees with the general engine")

    # Complete matches using controlled legitimate deals: the chosen winning seat
    # holds all spades, bids 13; their partner bids Nil. All 13 tricks yield +230.
    def scripted_match(win_team):
        s = reset(jax.random.fold_in(key, win_team))
        def tick(s, i):
            owners = jnp.array([(win_team + 1) % 4, (win_team + 2) % 4,
                                (win_team + 3) % 4, win_team], jnp.int32)
            prescribed = jnp.arange(4)[:, None] == owners[(jnp.arange(52) // 13)[None, :]]
            s = s._replace(hands=jnp.where(s.turn == 0, prescribed, s.hands))
            p = actor(s)
            bid = jnp.where(p == win_team, 13, jnp.where(p == (win_team + 2) % 4, 0, 1))
            legal = legal_mask(s)
            a = jnp.where(s.turn < 8, 66, jnp.where(s.turn < 12, 52 + bid, jnp.argmax(legal)))
            ns, reward, done, info = step(s, a.astype(jnp.int32), jax.random.fold_in(key, i))
            return ns, (info["hand_done"], done, info["winner"], info["final_scores"],
                        info["score_delta"], reward, info["discount"], ns.eligible)
        return jax.lax.scan(tick, s, jnp.arange(192))

    _, match_evidence = jax.jit(jax.vmap(scripted_match))(jnp.arange(2, dtype=jnp.int32))
    mh, md, mw, ms, mdelta, mr, mdiscount, meligible = jax.device_get(match_evidence)
    equal(mh.sum(1), [3, 3], "Three fully played hands per controlled match")
    equal(md.sum(1), [1, 1], "A match terminates once, on crossing 500")
    for winning in range(2):
        equal(ms[winning, [63, 127, 191], winning], [230, 460, 690], "Successful normal-plus-Nil contracts accumulate")
        equal(ms[winning, [63, 127, 191], 1 - winning], [-20, -40, -60], "Failed normal contracts accumulate")
        check(int(mw[winning, 191]) == winning, "The correct partnership wins the complete match")
        check(float(mdiscount[winning, 191]) == 0, "Completed match cannot bootstrap from a fresh game")
        check(bool(meligible[winning, 63, 1 - winning]), "Trailing by 150+ enables Double Nil on the next hand")
        check(not bool(meligible[winning, 63, winning]), "Leading team cannot bid Double Nil")
        check(float(mr[winning, 191]) > 0 if winning == 0 else float(mr[winning, 191]) < 0,
              "Terminal reward uses team-zero perspective")
    group("complete matches for both partnerships, score carry, and next-hand eligibility")

    # Heuristic legality and secrecy, when the optional opponent cell is loaded.
    if "heuristic_action" in globals() and "HEURISTIC_NAMES" in globals():
        n_styles = len(HEURISTIC_NAMES)
        styles = jnp.arange(n_styles, dtype=jnp.int32)
        hs = jax.jit(jax.vmap(lambda style: heuristic_action(revealed, key, style)))(styles)
        check(_np.asarray(legal_mask(revealed))[_np.asarray(hs)], "Every heuristic emits a legal bid")
        hidden_hs = jax.jit(jax.vmap(lambda style: heuristic_action(hidden_alt, key, style)))(styles)
        equal(hs, hidden_hs, "Heuristic bidding does not peek at hidden hands or sealed cards")
        bs = jax.jit(jax.vmap(lambda style: heuristic_action(base, key, style)))(styles)
        alt_bs = jax.jit(jax.vmap(lambda style: heuristic_action(blind_alt, key, style)))(styles)
        equal(bs, alt_bs, "Heuristic blind decisions cannot inspect the deal")
        group(f"opponent legality and information privacy ({n_styles} styles)")

    elapsed = _time.monotonic() - _started
    summary = {"passed": True, "assertions": _checks, "groups": len(_groups),
               "independent_numeric_cases": _numeric_cases, "seconds": round(elapsed, 2),
               "quick": bool(quick)}
    print(f"All {_checks} assertions passed in {elapsed:.1f}s ({len(_groups)} groups).", flush=True)
    return summary


In [ ]:
def run_learning_tests(verbose=True):
    """Test actual PPO internals: privacy, masks, signs, terminal GAE and updates.

    Small CPU/GPU fixtures; does not mutate the notebook's training state.
    Run after the engine, opponents and learning cells.
    """
    assertions = 0

    def check(condition, message):
        nonlocal assertions
        assert bool(condition), message
        assertions += 1

    def close(actual, expected, message, atol=2e-6):
        nonlocal assertions
        np.testing.assert_allclose(np.asarray(actual), np.asarray(expected),
                                   rtol=2e-5, atol=atol, err_msg=message)
        assertions += 1

    cfg = learning_config(dict(n_env=4, rollout_len=1, ppo_epochs=1,
                               minibatch_size=4, width=16, snapshot_size=2, gae_per_hand=False))
    initial = initialize_training(cfg)
    trainer = make_train_step(cfg)
    model = model_for_config(cfg)
    params = jax.tree_util.tree_map(lambda x: x, initial.params)
    kernel = params['critic_network']['output']['kernel']
    params['critic_network']['output']['kernel'] = (
        .1 * jax.random.normal(jax.random.PRNGKey(916), kernel.shape))
    envs = initial.envs._replace(turn=jnp.full(4, 8, jnp.int32))
    players = jax.vmap(actor)(envs)
    state = initial._replace(params=params, opt_state=_optimizer(cfg).init(params),
                             envs=envs, league_kind=jnp.array([0, 1, 2, 2]),
                             learner_team=(players % 2 + jnp.array([0, 1, 1, 0])) % 2,
                             league_style=jnp.full(4, 7, jnp.int32))
    obs = jax.vmap(observe)(envs)
    central = jax.vmap(critic_observe)(envs)
    logits, old_values = model.apply({'params': params}, obs, central)
    altered_logits, _ = model.apply({'params': params}, obs, central * -3. + 7.)
    close(logits, altered_logits, 'Actor must be independent of privileged critic input', atol=0)
    hidden_gradient = jax.grad(lambda c: model.apply({'params': params}, obs, c)[0].sum())(central)
    close(hidden_gradient, jnp.zeros_like(hidden_gradient), 'No actor gradient through critic data', atol=0)

    collected, batch, metrics = trainer.collect(state)
    jax.block_until_ready(batch.returns)
    expected_mask = jnp.array([1., 0., 0., 1.])
    close(batch.policy_mask, expected_mask, 'Historical/scripted opponent decisions excluded')
    for name in ('league_kind', 'league_style', 'league_snapshot', 'learner_team'):
        close(getattr(collected, name), getattr(state, name), 'Opponent assignment stable within match')
    next_values = model.apply({'params': params}, jax.vmap(critic_observe)(collected.envs),
                               method=model.value)
    # One nonterminal turn has zero reward, discount one and a value bootstrap.
    close(batch.returns, next_values, 'Nonterminal truncated GAE bootstraps next value')
    signed = (next_values - old_values) * jnp.where(players % 2 == 0, 1., -1.)
    mean = jnp.sum(signed * expected_mask) / expected_mask.sum()
    var = jnp.sum((signed - mean) ** 2 * expected_mask) / expected_mask.sum()
    expected_adv = (signed - mean) / jnp.sqrt(var + 1e-8)
    close(batch.advantages, expected_adv, 'Team0 GAE sign converted exactly once for acting team', atol=3e-5)
    check(float(metrics['illegal_actions']) == 0, 'No illegal actions during mixed-opponent rollout')

    # Changing only opponent-row policy targets must not affect actor gradients.
    _, gradients = jax.value_and_grad(trainer.loss_fn, has_aux=True)(params, batch)
    changed = batch._replace(advantages=jnp.where(expected_mask > 0, batch.advantages, 1e6))
    _, changed_gradients = jax.value_and_grad(trainer.loss_fn, has_aux=True)(params, changed)
    for a, b in zip(jax.tree_util.tree_leaves(gradients['actor_network']),
                    jax.tree_util.tree_leaves(changed_gradients['actor_network'])):
        close(a, b, 'Opponent policy targets must produce zero actor gradient', atol=0)

    # A critic-only minibatch updates critic parameters while leaving the actor untouched.
    critic_batch = batch._replace(policy_mask=jnp.zeros(4),
                                  returns=batch.old_values + jnp.array([1., -.5, .7, -.3]))
    (_, diagnostics), critic_gradients = jax.value_and_grad(trainer.loss_fn, has_aux=True)(params, critic_batch)
    for g in jax.tree_util.tree_leaves(critic_gradients['actor_network']):
        close(g, jnp.zeros_like(g), 'Critic baseline must not train actor encoder', atol=0)
    check(float(optax.global_norm(critic_gradients['critic_network'])) > 0,
          'Critic-only loss has a meaningful nonzero critic gradient')
    updated, update_metrics = trainer.optimize(state, critic_batch)
    jax.block_until_ready(updated.params)
    check(bool(update_metrics['finite']), 'Actual PPO optimizer update remains finite')
    check(int(updated.updates) == int(state.updates) + 1, 'Optimizer increments update counter')
    check(int(updated.env_steps) == 4, 'Host-derived decision counter agrees with config')
    for a, b in zip(jax.tree_util.tree_leaves(params['actor_network']),
                    jax.tree_util.tree_leaves(updated.params['actor_network'])):
        close(a, b, 'Fresh optimizer does not change actor from critic-only minibatch', atol=0)
    check(float(optax.global_norm(jax.tree_util.tree_map(
        lambda a, b: a - b, updated.params['critic_network'], params['critic_network']))) > 0,
        'Actual optimizer changes critic weights')
    changed_actor, actor_metrics = trainer.optimize(state, batch)
    jax.block_until_ready(changed_actor.params)
    check(bool(actor_metrics['finite']), 'Actual mixed-opponent actor update remains finite')
    check(float(optax.global_norm(jax.tree_util.tree_map(
        lambda a, b: a - b, changed_actor.params['actor_network'], params['actor_network']))) > 0,
        'Learner policy receives a meaningful update')

    # Reach valid last-turn positions and force an unambiguous team0 match win.
    @jax.jit
    def final_turn_positions(start):
        def advance(s, i):
            actions = jnp.argmax(jax.vmap(legal_mask)(s), axis=1)
            keys = jax.random.split(jax.random.fold_in(jax.random.PRNGKey(917), i), 4)
            return batch_step(s, actions, keys)[0], None
        return jax.lax.scan(advance, start, jnp.arange(63))[0]

    final_envs = final_turn_positions(initial.envs)
    final_envs = final_envs._replace(scores=jnp.array([[1000, -1000], [1000, -500],
                                                      [1000, 0], [1000, 500]], jnp.int32))
    terminal_state = state._replace(envs=final_envs, league_kind=jnp.zeros(4, jnp.int32))
    keys = jax.random.split(terminal_state.rng, 5)
    final_actions = jnp.argmax(jax.vmap(legal_mask)(final_envs), axis=1)
    expected_next, expected_rewards, expected_done, expected_info = batch_step(
        final_envs, final_actions, jax.random.split(keys[2], 4))
    after_terminal, terminal_batch, terminal_metrics = trainer.collect(terminal_state)
    expected_score_bonus = cfg['hand_score_reward_coef'] * (
        expected_info['score_delta'][:, 0] - expected_info['score_delta'][:, 1]) / 100.
    close(terminal_batch.returns, expected_rewards + expected_score_bonus,
          'Terminal discount must cut value leakage from fresh reset match')
    check(bool(jnp.all(expected_done)), 'Fixture exercises completed matches')
    check(bool(jnp.all(after_terminal.envs.turn == 0)), 'Terminal matches start fresh hands')
    check(bool(jnp.all(terminal_batch.policy_mask == 0)), 'Forced final cards excluded from actor loss')
    check(int(terminal_metrics['matches']) == 4, 'Completed match metrics recorded')
    # Count remains exact beyond signed 32-bit cumulative decisions.
    far_future = state._replace(updates=jnp.int32(1_000_000),
                                 decisions_per_update=jnp.int32(16_384))
    check(far_future.env_steps == 16_384_000_000, 'Decision count remains exact above 2**31')
    if verbose:
        print(f'Learning checks passed: {assertions} assertions; actor privacy, '
              'opponent/forced masks, team signs, terminal GAE, finite optimizer updates, '
              'and long-running counters.')
    return dict(assertions=assertions, passed=True)


def run_numerical_tests(verbose=True, extended=False):
    """Exercise safety checks, partial PPO epochs, and actual rejected updates.

    Default checks compile only tiny reductions. Set extended=True to also
    exercise actual PPO/bootstrap updates with a separate tiny model; neither
    mode changes the active training state. Runs on the selected Colab device.
    """
    assertions = 0

    def check(condition, message):
        nonlocal assertions
        assert bool(condition), message
        assertions += 1

    def same_tree(actual, expected, message):
        nonlocal assertions
        actual_leaves, actual_def = jax.tree_util.tree_flatten(actual)
        expected_leaves, expected_def = jax.tree_util.tree_flatten(expected)
        assert actual_def == expected_def, message
        for a, b in zip(actual_leaves, expected_leaves):
            np.testing.assert_array_equal(np.asarray(a), np.asarray(b), err_msg=message)
        assertions += 1

    # Three epochs of sixteen minibatches match the default training schedule.
    # Check every possible KL stop, especially the float32 reciprocal edge
    # cases 41 and 47, while also checking that skipped rows are neutral.
    def partial_epoch_metrics(attempted, reject_last):
        slots = jnp.arange(48).reshape(3, 16)
        used = slots < attempted
        failed = used & (slots == attempted - 1) & reject_last
        diagnostics = jnp.where(used[..., None], jnp.array([2., 4.]), 0.)
        counts = jnp.stack((used, used & ~failed), axis=-1).astype(jnp.int32)
        flags = jnp.ones((3, 16, len(_UPDATE_CHECK_NAMES)), jnp.bool_)
        flags = flags.at[..., -1].set(~failed)
        per_epoch = jax.vmap(_reduce_update_rows)((diagnostics, counts, flags))
        return _update_metrics(per_epoch, ('fixture_loss', 'fixture_norm'))

    prefix_metrics = jax.jit(jax.vmap(partial_epoch_metrics, in_axes=(0, None)))
    for rejected in (False, True):
        all_metrics = prefix_metrics(jnp.arange(1, 49, dtype=jnp.int32), rejected)
        for i in range(48):
            m = {name: np.asarray(value)[i] for name, value in all_metrics.items()}
            check(bool(m['finite']) == (not rejected), 'Partial epoch safety flag must be exact')
            check(int(m['attempted_updates']) == i + 1, 'Attempted minibatches must be exact')
            check(int(m['accepted_updates']) == i + 1 - int(rejected), 'Only accepted updates count')
            check(int(m['rejected_updates']) == int(rejected), 'Rejected update count must be exact')
            check(np.isclose(m['fixture_loss'], 2.), 'Skipped rows must not dilute loss metrics')
        check(np.asarray(all_metrics['finite']).dtype == np.bool_, 'Safety flags remain boolean')
        check(np.asarray(all_metrics['attempted_updates']).dtype == np.int32, 'Counts remain integer')

    # Verify that each diagnostic identifies its own failure, independently of
    # numerical reductions in the model or the optimizer.
    finite_tree = {'leaf': jnp.array([1., -2.])}
    arguments = (jnp.array([1., 2.]), finite_tree, jnp.float32(2.), finite_tree, finite_tree)
    probe_checks = jax.jit(_update_checks)
    check(np.asarray(probe_checks(*arguments)).all(), 'Finite update passes every guard')
    for index in range(5):
        broken = list(arguments)
        if index in (1, 3, 4):
            broken[index] = {'leaf': jnp.array([1., jnp.inf])}
        elif index == 2:
            broken[index] = jnp.float32(jnp.inf)
        else:
            broken[index] = jnp.array([1., jnp.nan])
        actual = np.asarray(probe_checks(*broken))
        expected = np.ones(5, dtype=bool)
        expected[index] = False
        check(np.array_equal(actual, expected), 'Failure report identifies the affected component')

    if not extended:
        if verbose:
            print(f'Numerical checks passed: {assertions} assertions; every partial PPO epoch, '
                  'boolean safety flags, integer counts, and named failure checks.')
        return dict(assertions=assertions, passed=True, extended=False)

    cfg = learning_config(dict(n_env=4, rollout_len=1, minibatch_size=4,
                               ppo_epochs=3, width=16, snapshot_size=2,
                               target_kl=10.))
    state = initialize_training(cfg)
    state = state._replace(envs=state.envs._replace(turn=jnp.full(4, 8, jnp.int32)))
    trainer = make_train_step(cfg)
    collected, batch, _ = trainer.collect(state)
    batch = batch._replace(returns=batch.old_values + 1.)
    trained, healthy = trainer.optimize(collected, batch)
    check(bool(healthy['finite']), 'Actual PPO update is accepted')
    check(int(healthy['attempted_updates']) == 3, 'Healthy PPO completes all three epochs')
    check(int(healthy['accepted_updates']) == 3, 'Healthy PPO counts actual accepted updates')
    check(all(np.isfinite(np.asarray(x)).all() for x in jax.tree_util.tree_leaves(trained.opt_state)),
          'Accepted PPO optimizer state is finite')

    # Non-finite training targets exercise the actual differentiated loss and
    # must reject the first minibatch, skip later epochs, and retain weights.
    bad_batch = batch._replace(returns=jnp.full_like(batch.returns, jnp.inf))
    rejected, failure = trainer.optimize(collected, bad_batch)
    check(not bool(failure['finite']), 'Actual non-finite PPO update is rejected')
    check(not bool(failure['diagnostics_finite']), 'Non-finite loss is named in the report')
    check(not bool(failure['gradients_finite']), 'Non-finite gradient is named in the report')
    check(int(failure['attempted_updates']) == 1 and int(failure['accepted_updates']) == 0,
          'Numerical failure stops later minibatches and epochs')
    check(not bool(failure['kl_early_stop']), 'Numerical failure is not mislabeled as KL stopping')
    same_tree(rejected.params, collected.params, 'Rejected PPO update preserves parameters')
    same_tree(rejected.opt_state, collected.opt_state, 'Rejected PPO update preserves optimizer')

    # Each gradient can be finite while squaring them overflows global_norm.
    # A zero-initialized critic output keeps this fixture's forward loss finite.
    huge_features = batch._replace(critic_obs=batch.critic_obs * jnp.float32(1e20))
    rejected, failure = trainer.optimize(collected, huge_features)
    check(bool(failure['diagnostics_finite']) and bool(failure['gradients_finite']),
          'Large-feature fixture has finite loss and individual gradients')
    check(not bool(failure['grad_norm_finite']) and not bool(failure['finite']),
          'Overflowing global gradient norm is rejected and identified')
    check(int(failure['attempted_updates']) == 1 and int(failure['accepted_updates']) == 0,
          'Overflowing gradient norm cannot count as an accepted update')
    same_tree(rejected.params, collected.params, 'Gradient-norm overflow preserves parameters')
    same_tree(rejected.opt_state, collected.opt_state, 'Gradient-norm overflow preserves optimizer')

    # Adam can produce finite weights despite an infinite second moment. This
    # used to pass the guard because only weights and gradients were checked.
    adam, empty = collected.opt_state[1]
    damaged_adam = adam._replace(nu=jax.tree_util.tree_map(lambda x: jnp.full_like(x, jnp.inf), adam.nu))
    damaged = collected._replace(opt_state=(collected.opt_state[0], (damaged_adam, empty)))
    rejected, failure = trainer.optimize(damaged, batch)
    check(not bool(failure['finite']) and not bool(failure['optimizer_finite']),
          'Infinite Adam moments are rejected and identified')
    check(bool(failure['params_finite']) and bool(failure['gradients_finite']),
          'Optimizer corruption is detected even with finite weights and gradients')
    check(int(failure['attempted_updates']) == 1 and int(failure['accepted_updates']) == 0,
          'Corrupt optimizer cannot count as an accepted update')
    same_tree(rejected.params, damaged.params, 'Optimizer-state failure preserves parameters')
    same_tree(rejected.opt_state, damaged.opt_state, 'Optimizer-state failure does not commit candidate')

    bootstrap = make_bootstrap_step(cfg)
    booted, healthy = bootstrap(state)
    expected_teacher_updates = cfg['bootstrap_epochs'] * (cfg['n_env'] * cfg['rollout_len'] //
        min(cfg['bootstrap_minibatch_size'], cfg['n_env'] * cfg['rollout_len']))
    check(bool(healthy['finite']) and int(healthy['accepted_updates']) == expected_teacher_updates,
          'Actual supervised initialization accepts a finite update')
    rejected, failure = bootstrap(damaged)
    check(not bool(failure['finite']) and not bool(failure['optimizer_finite']),
          'Supervised initialization uses the same optimizer-state guard')
    check(int(failure['accepted_updates']) == 0, 'Rejected initialization update is not counted')
    same_tree(rejected.params, damaged.params, 'Rejected initialization preserves parameters')
    same_tree(rejected.opt_state, damaged.opt_state, 'Rejected initialization preserves optimizer')

    if verbose:
        print(f'Numerical checks passed: {assertions} assertions; every partial PPO epoch, '
              'boolean safety flags, integer counts, named failures, and actual PPO/bootstrap rollback.')
    return dict(assertions=assertions, passed=True, extended=True)


def run_strength_tests(extended=False):
    """Check teacher guidance, hand-clock credit, scoring, privacy, and recovery.

    The default uses small fixtures and an independent two-hand game replay.
    Extended checks also execute teacher initialization, PPO, and an exact
    checkpoint/resume roundtrip. Neither mode changes the active run.
    """
    assertions = 0

    def check(condition, message):
        nonlocal assertions
        assert bool(condition), message
        assertions += 1

    def close(actual, expected, message, atol=2e-6):
        nonlocal assertions
        np.testing.assert_allclose(np.asarray(actual), np.asarray(expected),
                                   rtol=2e-5, atol=atol, err_msg=message)
        assertions += 1

    def exact_tree(actual, expected, message):
        nonlocal assertions
        actual_leaves, actual_tree = jax.tree_util.tree_flatten(jax.device_get(actual))
        expected_leaves, expected_tree = jax.tree_util.tree_flatten(jax.device_get(expected))
        assert actual_tree == expected_tree, message + ': tree structure'
        for a, b in zip(actual_leaves, expected_leaves):
            np.testing.assert_array_equal(a, b, err_msg=message)
        assertions += 1

    cfg = learning_config(dict(n_env=4, rollout_len=1, ppo_epochs=1,
                               minibatch_size=4, width=16, snapshot_size=2,
                               hand_score_reward_coef=.25))
    zeros = jnp.zeros(4, jnp.float32)
    zero_info = dict(score_delta=jnp.zeros((4, 2), jnp.int32))
    close(_training_rewards(zeros, zero_info, cfg), zeros,
          'Ordinary turns add no score reward', atol=0)
    deltas = jnp.array([[40, -120], [-28, 42], [0, 0], [-330, -250]], jnp.int32)
    engine = jnp.array([-.2, .3, 1., -1.], jnp.float32)
    # Independently calculated: .25 times the partnership point difference / 100.
    bonuses = np.array([.4, -.175, 0., -.2], np.float32)
    combined = _training_rewards(engine, dict(score_delta=deltas), cfg)
    default_cfg = learning_config()
    check(default_cfg['hand_score_reward_coef'] == 0.,
          'Default training retains the original engine reward objective')
    close(_training_rewards(engine, dict(score_delta=deltas), default_cfg), engine,
          'Default rewards exactly preserve match-win and potential rewards', atol=0)
    close(combined, np.asarray(engine) + bonuses,
          'Both partnerships and cumulative bag penalties affect score feedback')
    close(_training_rewards(-engine, dict(score_delta=deltas[:, ::-1]), cfg), -combined,
          'Swapping partnerships reverses the entire training reward')
    close(_training_rewards(engine, zero_info, cfg), engine,
          'Zero score difference preserves terminal win/loss bonuses', atol=0)
    close(_training_rewards(engine, dict(score_delta=deltas), dict(cfg, hand_score_reward_coef=0)),
          engine, 'Disabling the auxiliary exactly preserves the engine reward', atol=0)
    for name in ('hand_score_reward_coef', 'bidding_anchor_coef'):
        for invalid in (-.1, float('nan'), float('inf')):
            try:
                learning_config({name: invalid})
            except ValueError:
                assertions += 1
            else:
                raise AssertionError('Invalid training auxiliary coefficients must be rejected')

    # Verify the actual engine observation coordinates, including the actor's
    # partnership-relative ordering and the critic's fixed team-0 ordering.
    score_state = reset(jax.random.PRNGKey(780))._replace(
        turn=jnp.int32(12), current_player=jnp.int32(1),
        scores=jnp.array([-1000000, 500], jnp.int32))
    actor_features, critic_features = observe(score_state), critic_observe(score_state)
    close(actor_features[133:135], [1., -2000.], 'Actor score coordinates/order are correct', atol=0)
    close(critic_features[133:135], [-2000., 1.], 'Critic score coordinates/order are correct', atol=0)
    for features, expected in ((actor_features, [1., -(1. + np.log(2000.))]),
                               (critic_features, [-(1. + np.log(2000.)), 1.])):
        encoded = _network_features(features)
        close(encoded[133:135], expected, 'Extreme scores use logarithmic network inputs')
        unchanged = np.ones(features.shape[-1], dtype=bool)
        unchanged[133:135] = False
        exact_tree(np.asarray(encoded)[unchanged], np.asarray(features)[unchanged],
                   'Only the two score features are transformed')
    ordinary = jnp.tile(actor_features[None], (7, 1))
    ordinary_values = jnp.array([-1., -.5, -.01, 0., .01, .5, 1.])
    ordinary = ordinary.at[:, 133].set(ordinary_values).at[:, 134].set(ordinary_values[::-1])
    exact_tree(_network_features(ordinary), ordinary, 'Ordinary score features remain exactly unchanged')
    extremes = jnp.array([-np.finfo(np.float32).max, -1e12, -1000., -2., -1.,
                          0., 1., 2., 1000., 1e12, np.finfo(np.float32).max], jnp.float32)
    extreme_features = jnp.tile(actor_features[None], (len(extremes), 1))
    extreme_features = extreme_features.at[:, 133].set(extremes).at[:, 134].set(-extremes)
    encoded_extremes = _network_features(extreme_features)[:, 133:135]
    check(jnp.all(jnp.isfinite(encoded_extremes)) & jnp.all(jnp.diff(encoded_extremes[:, 0]) > 0),
          'Extreme finite score inputs stay finite and strictly ordered')
    close(encoded_extremes[:, 0], -encoded_extremes[:, 1],
          'Score encoding preserves partnership antisymmetry', atol=0)
    close(score_state.scores, [-1000000, 500], 'Network preprocessing leaves game scores untouched', atol=0)

    initial = initialize_training(cfg)
    # Four locally complete final-trick positions: player 3 must play the ace of
    # spades over clubs 2/3/4. Every card is either played or in that one-card hand.
    hands = jnp.zeros((4, 52), jnp.bool_).at[3, 51].set(True)
    played_by = jnp.full(52, -1, jnp.int32).at[3:51].set(jnp.arange(48) % 4)
    played_by = played_by.at[:3].set(jnp.arange(3))
    played_at = jnp.full(52, -1, jnp.int32).at[3:51].set(jnp.arange(48))
    played_at = played_at.at[:3].set(jnp.arange(48, 51))
    base = reset(jax.random.PRNGKey(781))._replace(
        hands=hands, turn=jnp.int32(63), current_player=jnp.int32(3),
        table=jnp.array([0, 1, 2, -1], jnp.int32),
        table_players=jnp.array([0, 1, 2, -1], jnp.int32),
        trick_pos=jnp.int32(3), leader=jnp.int32(0),
        played_by=played_by, played_at=played_at,
        scores=jnp.array([1000, -1000], jnp.int32),
        eligible=jnp.array([False, True]))
    envs = jax.tree_util.tree_map(lambda x: jnp.repeat(x[None], 4, axis=0), base)
    envs = envs._replace(
        bids=jnp.array([[5, 2, 2, 0], [5, 2, 2, -1],
                        [0, 5, 2, 3], [5, -1, 2, 3]], jnp.int32),
        tricks=jnp.array([[6, 3, 3, 0], [6, 3, 3, 0],
                          [0, 6, 3, 3], [6, 0, 3, 3]], jnp.int32),
        bags=jnp.array([[9, 0], [0, 0], [0, 0], [0, 0]], jnp.int32))
    legal = jax.vmap(legal_mask)(envs)
    check(jnp.all(jnp.sum(legal, axis=1) == 1) & jnp.all(legal[:, 51]),
          'Every final-trick fixture has exactly the intended legal card')
    check(jnp.all(jnp.sum(envs.tricks, axis=1) == 12), 'Fixtures contain twelve completed tricks')
    params = jax.tree_util.tree_map(lambda x: x, initial.params)
    params['critic_network']['output']['bias'] = jnp.full_like(
        params['critic_network']['output']['bias'], 8.)
    terminal = initial._replace(params=params, opt_state=_optimizer(cfg).init(params),
        envs=envs, league_kind=jnp.array([0, 2, 2, 2], jnp.int32),
        learner_team=jnp.array([0, 1, 0, 1], jnp.int32))
    trainer = make_train_step(cfg)
    after, batch, metrics = trainer.collect(terminal)
    # Hand 0: 70 + 2 overtricks - 100 ten-bag penalty = -28; failed Nil = -120.
    # Hand 1: +72 versus failed DN = -220. Hand 2: +121 versus +82.
    # Hand 3: +72 versus successful DN + partner's 30 points + 1 overtrick = +231.
    expected_deltas = np.array([[-28, -120], [72, -220], [121, 82], [72, 231]], np.float32)
    expected_bonus = np.array([.23, .73, .0975, -.3975], np.float32)
    expected_terminal_engine = 1. - .5 * np.tanh(4.)
    close(batch.returns, expected_terminal_engine + expected_bonus,
          'Actual terminal GAE includes explicit score feedback and no reset-critic leakage')
    close(metrics['score_bonus_abs_sum'], np.abs(expected_bonus).sum(),
          'Collected score-feedback diagnostic matches hand-calculated bonuses')
    check(jnp.all(after.envs.turn == 0) & jnp.all(after.envs.scores == 0),
          'Completed terminal matches reset for future play')
    check(float(metrics['matches']) == 4 and float(metrics['illegal_actions']) == 0,
          'All four terminal fixtures complete legally')
    expected_metrics = dict(learner_contracts=5, learner_successful_contracts=3,
        learner_hand_score_sum=-16, opponent_hand_score_sum=226, opponent_contracts=3,
        learner_bid_seats=10, learner_nil_bids=2, learner_dn_bids=2,
        learner_nil_made=1, learner_dn_made=1, learner_normal_bid_sum=16,
        learner_normal_bids=6, nil_attempts=2, nil_successes=1,
        dn_attempts=2, dn_successes=1, successful_contracts=6)
    for name, expected in expected_metrics.items():
        close(metrics[name], expected, 'Final-trick learner metric: ' + name, atol=0)

    # At this score gap tanh has saturated; the same scored hands still provide
    # their full auxiliary signal. No artificial match reset or score floor is used.
    saturated_envs = envs._replace(scores=jnp.tile(jnp.array([-10000, -1000]), (4, 1)))
    _, engine_rewards, done, info = batch_step(saturated_envs, jnp.full(4, 51, jnp.int32),
                                             jax.random.split(jax.random.PRNGKey(782), 4))
    check(~jnp.any(done), 'Deeply negative fixtures continue their real matches')
    close(info['score_delta'], expected_deltas, 'Actual rule scoring includes the bag penalty', atol=0)
    dense = _training_rewards(engine_rewards, info, cfg)
    close(dense - engine_rewards, expected_bonus, 'Dense score feedback survives saturated potential')
    check(float(dense[3]) < -.39, 'An inferior scored hand still receives a meaningful penalty')

    # Refresh only a continuing match at a completed-hand boundary. Construct a
    # critic whose output depends on team-0 score so resetting before valuation
    # would give a detectably different bootstrap.
    refresh_params = jax.tree_util.tree_map(lambda x: x, initial.params)
    critic = jax.tree_util.tree_map(jnp.zeros_like, refresh_params['critic_network'])
    critic['input']['kernel'] = critic['input']['kernel'].at[133, 0].set(1.)
    critic['output']['kernel'] = critic['output']['kernel'].at[0, 0].set(1.)
    refresh_params['critic_network'] = critic
    limit = cfg['max_training_match_hands']
    continuing_envs = initial.envs._replace(
        scores=jnp.tile(jnp.array([-5000, -1000], jnp.int32), (4, 1)),
        hand_number=jnp.array([limit, limit, limit, limit - 1], jnp.int32))
    incoming_rewards = jnp.array([-.3, .2, 1., -.7], jnp.float32)
    incoming_discounts = jnp.array([GAMMA_HAND, 1., 0., GAMMA_HAND], jnp.float32)
    done_flags = jnp.array([False, False, True, False])
    hand_flags = jnp.array([True, False, True, True])
    continuation_values = value_apply(refresh_params, jax.vmap(critic_observe)(continuing_envs), cfg)
    fresh_envs, adjusted_rewards, trace_discounts, truncated = _refresh_training_trajectories(
        refresh_params, continuing_envs, incoming_rewards, incoming_discounts,
        done_flags, hand_flags, jax.random.PRNGKey(784), cfg)
    close(truncated, [1, 0, 0, 0],
          'Refresh requires a completed hand at the limit and excludes true terminals', atol=0)
    expected_adjustment = np.array(incoming_rewards)
    expected_adjustment[0] += float(incoming_discounts[0]) * float(continuation_values[0])
    close(adjusted_rewards, expected_adjustment, 'Refresh adds exactly gamma times the continuing value')
    close(trace_discounts, [0., 1., 0., GAMMA_HAND],
          'Only refreshed trajectories have their GAE trace stopped')
    check(jnp.all(fresh_envs.scores[0] == 0) & (fresh_envs.hand_number[0] == 0),
          'A refreshed sample receives a fresh match')
    exact_tree(jax.tree_util.tree_map(lambda x: x[1:], fresh_envs),
               jax.tree_util.tree_map(lambda x: x[1:], continuing_envs),
               'Non-boundary, terminal, and below-limit samples remain untouched')
    fresh_values = value_apply(refresh_params, jax.vmap(critic_observe)(fresh_envs), cfg)
    check(abs(float(fresh_values[0] - continuation_values[0])) > 1e-4,
          'Fixture distinguishes continuing-game value from reset-game value')
    untouched_envs, untouched_rewards, untouched_discounts, untouched_flags = _refresh_training_trajectories(
        refresh_params, continuing_envs, incoming_rewards, incoming_discounts,
        done_flags, jnp.zeros(4, jnp.bool_), jax.random.PRNGKey(785), cfg)
    exact_tree(untouched_envs, continuing_envs, 'Long matches are never refreshed mid-hand')
    exact_tree(untouched_rewards, incoming_rewards, 'No refresh leaves rewards unchanged')
    exact_tree(untouched_discounts, incoming_discounts, 'No refresh leaves discounts unchanged')
    check(~jnp.any(untouched_flags), 'No completed hands means no trajectory refreshes')

    boundary_envs = saturated_envs._replace(hand_number=jnp.full(4, limit - 1, jnp.int32))
    boundary_state = terminal._replace(envs=boundary_envs)
    boundary_after, boundary_batch, boundary_metrics = trainer.collect(boundary_state)
    close(boundary_batch.returns, np.asarray(dense) + GAMMA_HAND * 8.,
          'Actual GAE includes continuing value exactly once and cannot leak from the fresh match')
    close(boundary_metrics['training_truncations'], 4, 'Every refreshed training sample is counted', atol=0)
    close(metrics['training_truncations'], 0, 'True match endings are not trajectory refreshes', atol=0)
    for name in ('matches', 'team0_wins', 'team1_wins', 'draws', 'opponent_matches', 'learner_wins'):
        close(boundary_metrics[name], 0, 'A trajectory refresh cannot create fake outcomes: ' + name, atol=0)
    check(jnp.all(boundary_after.envs.scores == 0) & jnp.all(boundary_after.envs.hand_number == 0),
          'Actual collector installs fresh matches at training boundaries')
    league_key = jax.random.split(boundary_state.rng, 5)[4]
    expected_assignment = _league_assignments(league_key, cfg, boundary_state.snapshot_count,
                                             boundary_state.snapshot_retiring)
    for name, expected in zip(('league_kind', 'league_style', 'league_snapshot', 'learner_team'),
                              expected_assignment):
        close(getattr(boundary_after, name), expected,
              'Training refresh also draws a new league assignment: ' + name, atol=0)

    # Exercise actual collection masks, then isolate the anchor gradient from
    # PPO/value/entropy losses. Opponents, forced choices, passing, and play must
    # never acquire bidding gradients; card guidance is checked independently below.
    bid_envs = initial.envs._replace(turn=jnp.full(4, 8, jnp.int32))
    bid_players = jax.vmap(actor)(bid_envs)
    bid_envs = bid_envs._replace(bids=bid_envs.bids.at[3, bid_players[3]].set(-1))
    bid_state = initial._replace(envs=bid_envs,
        league_kind=jnp.array([0, 2, 1, 0], jnp.int32),
        learner_team=(bid_players % 2 + jnp.array([0, 1, 0, 0])) % 2)
    _, bid_batch, _ = trainer.collect(bid_state)
    close(bid_batch.teacher_mask, [1, 0, 1, 0],
          'Only learner-controlled, non-forced bidding choices receive anchors', atol=0)
    check(jnp.all(jnp.take_along_axis(bid_batch.legal, bid_batch.teacher_actions[:, None], axis=1)),
          'Collected bidding teacher labels are legal')
    anchor_cfg = dict(cfg, entropy_coef=0., value_coef=0.)
    anchor_trainer = make_train_step(anchor_cfg)
    anchor_batch = bid_batch._replace(advantages=jnp.zeros_like(bid_batch.advantages))
    anchor_grad = jax.grad(lambda p: anchor_trainer.loss_fn(p, anchor_batch)[0])(initial.params)
    check(float(optax.global_norm(anchor_grad['actor_network'])) > 0,
          'Valid bidding examples provide a nonzero anchor gradient')
    close(optax.global_norm(anchor_grad['critic_network']), 0.,
          'The bidding anchor cannot train the privileged critic', atol=0)
    excluded_changed = anchor_batch._replace(teacher_actions=jnp.where(
        anchor_batch.teacher_mask, anchor_batch.teacher_actions,
        (anchor_batch.teacher_actions + 1) % N_ACTIONS))
    changed_grad = jax.grad(lambda p: anchor_trainer.loss_fn(p, excluded_changed)[0])(initial.params)
    exact_tree(changed_grad, anchor_grad, 'Excluded teacher labels cannot change actor gradients')
    no_anchor = make_train_step(dict(anchor_cfg, bidding_anchor_coef=0., play_anchor_coef=0.))
    no_anchor_grad = jax.grad(lambda p: no_anchor.loss_fn(p, anchor_batch)[0])(initial.params)
    close(optax.global_norm(no_anchor_grad), 0., 'Disabling the anchor removes its gradient', atol=0)

    all_learner = initial._replace(league_kind=jnp.zeros(4, jnp.int32))
    _, forced_blind_batch, _ = trainer.collect(all_learner)
    check(~jnp.any(forced_blind_batch.teacher_mask), 'Ineligible forced blind declines receive no anchor')
    blind_players = jax.vmap(actor)(initial.envs)
    eligibility = jnp.zeros((4, 2), jnp.bool_).at[jnp.arange(4), blind_players % 2].set(True)
    blind_state = all_learner._replace(envs=initial.envs._replace(eligible=eligibility))
    _, blind_batch, _ = trainer.collect(blind_state)
    check(jnp.all(blind_batch.teacher_mask), 'Eligible learner blind choices receive anchors')
    changed_blind = blind_state._replace(envs=blind_state.envs._replace(
        hands=jnp.roll(blind_state.envs.hands, 1, axis=1)))
    _, changed_blind_batch, _ = trainer.collect(changed_blind)
    close(changed_blind_batch.teacher_actions, blind_batch.teacher_actions,
          'Blind teacher labels are independent of hidden card ownership', atol=0)
    pass_envs = initial.envs._replace(turn=jnp.full(4, 4, jnp.int32))
    pass_players = jax.vmap(actor)(pass_envs)
    pass_envs = pass_envs._replace(bids=pass_envs.bids.at[jnp.arange(4), pass_players].set(-1))
    _, pass_batch, _ = trainer.collect(all_learner._replace(envs=pass_envs))
    check(jnp.all(pass_batch.policy_mask) & ~jnp.any(pass_batch.teacher_mask),
          'Actionable card exchanges receive no bidding anchor')
    play_envs = initial.envs._replace(turn=jnp.full(4, 12, jnp.int32),
                                      bids=jnp.full((4, 4), 3, jnp.int32))
    _, play_batch, _ = trainer.collect(all_learner._replace(envs=play_envs))
    check(jnp.all(play_batch.policy_mask) & ~jnp.any(play_batch.teacher_mask),
          'Actionable card play receives no bidding anchor')

    # Additional strong-teacher/hand-clock checks are inserted here.
    _strength_additional_checks(check, close, exact_tree, cfg, initial, trainer)

    # Give the old state visibly non-fresh actor, critic, moments, games, and bank.
    legacy_params = jax.tree_util.tree_map(lambda x: x + .03, initial.params)
    legacy = initial._replace(params=legacy_params,
        opt_state=jax.tree_util.tree_map(lambda x: x + 1, initial.opt_state),
        envs=initial.envs._replace(scores=jnp.full((4, 2), -30000, jnp.int32),
                                  hand_number=jnp.full(4, 200, jnp.int32)),
        rng=jax.random.PRNGKey(783), updates=jnp.int32(348),
        snapshot_count=jnp.int32(2), snapshot_retiring=jnp.int32(1),
        bootstrap_updates=jnp.int32(16), bootstrap_complete=jnp.bool_(True))
    original = jax.tree_util.tree_map(lambda x: np.asarray(x).copy(), legacy)
    recovered = recover_legacy_training(legacy, cfg)
    exact_tree(recovered.params['actor_network'], legacy.params['actor_network'],
               'Recovery transfers every actor parameter exactly')
    exact_tree(recovered.params['critic_network'], initial.params['critic_network'],
               'Recovery resets the critic for the new return objective')
    exact_tree(recovered.opt_state, _optimizer(cfg).init(recovered.params),
               'Recovery resets all Adam moments and counters')
    exact_tree(legacy, original, 'Recovery does not mutate the original state')
    check(int(recovered.updates) == 348 and recovered.env_steps == legacy.env_steps,
          'Recovery preserves completed updates and decision counts')
    check(int(recovered.bootstrap_updates) == 0 and not bool(recovered.bootstrap_complete),
          'Recovery schedules a fresh teacher initialization')
    check(jnp.all(recovered.envs.scores == 0) & jnp.all(recovered.envs.hand_number == 0)
          & jnp.all(recovered.envs.turn == 0), 'Recovery clears old deeply negative games')
    check(int(recovered.snapshot_count) == 1 and int(recovered.snapshot_retiring) == -1
          and int(recovered.last_snapshot_update) == 348, 'Recovery clears stale league bookkeeping')
    for slot in range(cfg['snapshot_size']):
        snapshot = jax.tree_util.tree_map(lambda p: p[slot], recovered.snapshot_params)
        exact_tree(snapshot, recovered.params, 'Every recovery snapshot uses the transferred policy')
    check(not np.array_equal(np.asarray(recovered.rng), np.asarray(legacy.rng)),
          'Recovery advances the saved RNG stream')
    for altered in (dict(cfg, width=32), dict(cfg, rollout_len=2)):
        try:
            recover_legacy_training(legacy, altered)
        except ValueError:
            assertions += 1
        else:
            raise AssertionError('Recovery must reject changed architecture or rollout dimensions')

    if extended:
        from tempfile import TemporaryDirectory
        _, final_metrics = trainer(terminal)
        check(bool(final_metrics['finite']), 'Final-trick fixture also survives the real optimizer')
        for name, expected in dict(learner_contract_success_rate=.6,
                learner_hand_score=-3.2, opponent_hand_score=226 / 3,
                mean_normal_bid=16 / 6, nil_bid_rate=.2, dn_bid_rate=.2,
                learner_nil_success_rate=.5, learner_dn_success_rate=.5).items():
            close(final_metrics[name], expected, 'Derived learner metric: ' + name)
        full_cfg = learning_config(dict(cfg, rollout_len=64, minibatch_size=64, bootstrap_minibatch_size=32))
        full_legacy = initialize_training(full_cfg)._replace(
            params=legacy.params, updates=legacy.updates, rng=legacy.rng,
            bootstrap_complete=jnp.bool_(True))
        full_recovery = recover_legacy_training(full_legacy, full_cfg)
        teacher_state, teacher_metrics = make_bootstrap_step(full_cfg)(full_recovery)
        check(bool(teacher_metrics['finite']), 'Recovered teacher initialization stays finite')
        check(int(teacher_metrics['accepted_updates']) == 16,
              'Teacher SGD uses its own minibatch size and two epochs, independent of PPO minibatches')
        ready = finish_bootstrap(teacher_state, full_cfg)
        full_trainer = make_train_step(full_cfg)
        trained, trained_metrics = full_trainer(ready)
        check(bool(trained_metrics['finite']) and float(trained_metrics['illegal_rate']) == 0.,
              'Recovered policy completes a real finite PPO update')
        check(float(trained_metrics['hands']) == 4,
              'Extended PPO check includes complete scored hands')
        check(len(jax.tree_util.tree_leaves(trained)) == 214,
              'Full checkpoint contains all 214 arrays including the exact lifetime counter')
        event = dict(kind='trainer_upgrade', update=348, training_revision=TRAINING_REVISION)
        with TemporaryDirectory(prefix='spades_score_feedback_') as directory:
            path = save_checkpoint(trained, full_cfg, [event], directory, cumulative_seconds=123.)
            restored, stored_cfg, history, metadata = load_checkpoint(path, full_cfg)
            exact_tree(restored, trained, 'Recovered state survives complete checkpoint roundtrip')
            check(stored_cfg == full_cfg and history == [event], 'Checkpoint retains config and migration event')
            check(metadata['training_revision'] == TRAINING_REVISION
                  and metadata['cumulative_seconds'] == 123., 'Checkpoint identifies the new trainer revision')
            continued, continued_metrics = full_trainer(trained)
            resumed, resumed_metrics = full_trainer(restored)
            exact_tree(resumed, continued, 'Recovered checkpoint resumes the next PPO update exactly')
            exact_tree(resumed_metrics, continued_metrics, 'Resumed PPO diagnostics are reproducible')
    result = dict(passed=True, assertions=assertions, extended=bool(extended))
    print(f'Strength-training checks passed: {assertions} assertions; scoring, final-trick metrics, '
          f'hand-clock GAE, score encoding, public teachers, card/bid gradients, trajectory boundaries, and recovery'
          + (' with real PPO/checkpoint resume.' if extended else '.'))
    return result


def _strength_additional_checks(check, close, exact_tree, cfg, initial, trainer):
    # A teacher is permitted to read only the acting seat's cards and public
    # information. Change every hidden ownership assignment, including our own
    # during blind bidding, while keeping public observations and PRNG keys fixed.
    sample = reset(jax.random.PRNGKey(341))._replace(
        eligible=jnp.ones(2, jnp.bool_))
    advice = jax.jit(lambda s: teacher_action(s, jax.random.PRNGKey(342), cfg))
    for phase_turn in (0, 4, 8, 12):
        state = sample._replace(turn=jnp.int32(phase_turn), bids=jnp.full(4, 3, jnp.int32))
        me = int(actor(state))
        if phase_turn == 4:
            state = state._replace(bids=state.bids.at[me].set(-1))
        others = [p for p in range(4) if p != me]
        hidden = (jnp.roll(state.hands, 1, axis=0) if phase_turn == 0 else
                  state.hands.at[jnp.array(others)].set(state.hands[jnp.array(others[1:] + others[:1])]))
        changed = state._replace(hands=hidden)
        exact_tree(observe(state), observe(changed), 'Hidden hand changes preserve public observation')
        check(int(advice(state)) == int(advice(changed)),
              'Teacher does not depend on concealed card ownership at turn ' + str(phase_turn))
        check(bool(legal_mask(state)[advice(state)]), 'Public teacher label is legal in every phase')
        if phase_turn == 0:
            check(int(advice(state)) == NOOP, 'The coherent teacher declines an eligible blind gamble')
            check(bool(legal_mask(state)[67]), 'Guidance does not remove the policy Double Nil action')
    # Validate independent teacher controls; a zero guidance coefficient disables
    # learning from that source without changing the game's action space.
    for name in ('play_anchor_coef',):
        for invalid in (-1., float('nan'), float('inf')):
            try:
                learning_config({name: invalid})
            except ValueError:
                check(True, 'Reject invalid card-play guidance coefficients')
            else:
                raise AssertionError('Invalid guidance coefficient accepted')
    for name in ('gae_per_hand', 'teacher_decline_blind'):
        try:
            learning_config({name: 'yes'})
        except ValueError:
            check(True, 'Boolean trainer controls require actual boolean values')
        else:
            raise AssertionError('String boolean trainer control accepted')
    for invalid in (-1, len(HEURISTIC_NAMES), .5):
        try:
            learning_config({'teacher_style': invalid})
        except ValueError:
            check(True, 'Only existing public-information teachers are accepted')
        else:
            raise AssertionError('Invalid teacher style accepted')

    # Card/pass actions must receive guidance only on actionable learner rows.
    envs = initial.envs._replace(turn=jnp.full(4, 12, jnp.int32), bids=jnp.full((4, 4), 3, jnp.int32))
    players = jax.vmap(actor)(envs)
    state = initial._replace(envs=envs, league_kind=jnp.array([0, 2, 1, 0]),
        learner_team=(players % 2 + jnp.array([0, 1, 1, 0])) % 2)
    _, play, _ = trainer.collect(state)
    close(play.policy_mask, [1., 0., 0., 1.], 'Only learner seats provide card-play policy targets', atol=0)
    check(jnp.all(jnp.take_along_axis(play.legal, play.teacher_actions[:, None], axis=1)),
          'Every collected card-play teacher action is legal')
    pure_cfg = dict(cfg, value_coef=0., entropy_coef=0., bidding_anchor_coef=0., play_anchor_coef=.13)
    pure = make_train_step(pure_cfg)
    play = play._replace(advantages=jnp.zeros_like(play.advantages))
    gradient = jax.grad(lambda p: pure.loss_fn(p, play)[0])(initial.params)
    check(float(optax.global_norm(gradient['actor_network'])) > 0,
          'Card-play guidance provides a nonzero actor gradient')
    close(optax.global_norm(gradient['critic_network']), 0., 'Card teacher does not train privileged critic', atol=0)
    # Independent manual CE checks both the loss and all actor gradient leaves.
    def reference_ce(params):
        logits = policy_logits(params, play.obs, cfg)
        log_probs = jax.nn.log_softmax(jnp.where(play.legal, logits, -1e9), axis=-1)
        selected = log_probs[jnp.arange(4), play.teacher_actions]
        return -.13 * (selected[0] + selected[3]) / 2.
    manual_loss, manual_gradient = jax.value_and_grad(reference_ce)(initial.params)
    close(pure.loss_fn(initial.params, play)[0], manual_loss, 'Actual card CE equals independent two-learner calculation')
    for actual, expected in zip(jax.tree_util.tree_leaves(gradient), jax.tree_util.tree_leaves(manual_gradient)):
        close(actual, expected, 'Card CE gradients agree with independent reference', atol=2e-7)
    excluded = play._replace(teacher_actions=jnp.where(play.policy_mask > 0, play.teacher_actions,
                                                       (play.teacher_actions + 3) % N_ACTIONS))
    exact_tree(jax.grad(lambda p: pure.loss_fn(p, excluded)[0])(initial.params), gradient,
               'Opponent labels cannot affect play guidance')
    forced = play._replace(policy_mask=jnp.zeros_like(play.policy_mask))
    close(optax.global_norm(jax.grad(lambda p: pure.loss_fn(p, forced)[0])(initial.params)), 0.,
          'Forced rows provide no card guidance gradient', atol=0)
    no_guidance = make_train_step(dict(pure_cfg, play_anchor_coef=0.))
    close(optax.global_norm(jax.grad(lambda p: no_guidance.loss_fn(p, play)[0])(initial.params)), 0.,
          'Disabling both guides leaves no imitation gradient', atol=0)
    # On a real collected batch with zero imitation coefficients, the exact
    # scalar loss must be solely the PPO, clipped-value, and entropy objectives.
    ppo_cfg = dict(cfg, bidding_anchor_coef=0., play_anchor_coef=0.)
    ppo = make_train_step(ppo_cfg)
    _, ppo_batch, _ = ppo.collect(state)
    ppo_loss, diag = ppo.loss_fn(initial.params, ppo_batch)
    close(ppo_loss, diag[1] + ppo_cfg['value_coef'] * diag[2] - ppo_cfg['entropy_coef'] * diag[3],
          'Zero teacher coefficients restore the ordinary PPO objective exactly')

    # Independently verify bidding CE as well, including the exact coefficient.
    bid_envs = envs._replace(turn=jnp.full(4, 8, jnp.int32))
    bid_who = jax.vmap(actor)(bid_envs)
    bid_state = state._replace(envs=bid_envs,
        learner_team=(bid_who % 2 + jnp.array([0, 1, 1, 0])) % 2)
    _, bid, _ = trainer.collect(bid_state)
    bid = bid._replace(advantages=jnp.zeros_like(bid.advantages))
    bid_only = make_train_step(dict(pure_cfg, bidding_anchor_coef=.17, play_anchor_coef=0.))
    def reference_bid_ce(params):
        logits = policy_logits(params, bid.obs, cfg)
        log_probs = jax.nn.log_softmax(jnp.where(bid.legal, logits, -1e9), axis=-1)
        selected = log_probs[jnp.arange(4), bid.teacher_actions]
        return -.17 * (selected[0] + selected[3]) / 2.
    manual_bid_loss, manual_bid_grad = jax.value_and_grad(reference_bid_ce)(initial.params)
    actual_bid_loss, actual_bid_grad = jax.value_and_grad(lambda p: bid_only.loss_fn(p, bid)[0])(initial.params)
    close(actual_bid_loss, manual_bid_loss, 'Actual bidding CE equals independent two-learner calculation')
    for actual, expected in zip(jax.tree_util.tree_leaves(actual_bid_grad), jax.tree_util.tree_leaves(manual_bid_grad)):
        close(actual, expected, 'Bidding CE gradients agree with independent reference', atol=2e-7)

    # Independent hand-clock scalar GAE reference, including an actual terminal
    # and a sampling boundary. Check both supported lambda timescales.
    discounts = jnp.array([1., 1., GAMMA_HAND, 1., 0., 1., GAMMA_HAND, 0.], jnp.float32)
    rewards = np.array([0., 0., .4, 0., 1., 0., -.3, -1.], np.float64)
    values = np.array([.1, -.2, .3, .5, -.1, .7, .4, .2], np.float64)
    def scalar_gae(reward, discount, value, last, lam, hand_clock):
        result = np.zeros_like(reward, dtype=np.float64)
        carry = 0.; next_value = float(last)
        for t in reversed(range(len(reward))):
            # Deliberately separate from the implementation helper under test.
            trace = float(discount[t])
            if not hand_clock or float(discount[t]) < 1.:
                trace *= lam
            delta = float(reward[t]) + float(discount[t]) * next_value - float(value[t])
            carry = delta + trace * carry
            result[t] = carry
            next_value = float(value[t])
        return result
    for hand_clock in (False, True):
        mode = dict(cfg, gae_per_hand=hand_clock)
        trace = np.array(discounts, dtype=np.float64)
        trace *= np.where(trace < 1., cfg['gae_lambda'], 1.) if hand_clock else cfg['gae_lambda']
        close(_gae_trace_discount(discounts, mode), trace, 'Trace discount follows selected lambda clock')
        def recurrence(carry, row):
            reward, discount, value = row
            adv, next_value = carry
            adv = reward + discount * next_value - value + _gae_trace_discount(discount, mode) * adv
            return (adv, value), adv
        _, actual = jax.lax.scan(recurrence, (jnp.float32(0.), jnp.float32(9.)),
                                (jnp.asarray(rewards, jnp.float32), discounts, jnp.asarray(values, jnp.float32)), reverse=True)
        expected = scalar_gae(rewards, np.asarray(discounts), values, 9., cfg['gae_lambda'], hand_clock)
        close(actual, expected, 'GAE agrees with an independent scalar reverse recurrence')
        changed_reward = rewards.copy(); changed_reward[5:] += 1000.
        changed_reference = scalar_gae(changed_reward, np.asarray(discounts), values, 8000., cfg['gae_lambda'], hand_clock)
        close(changed_reference[:5], expected[:5], 'Zero discounts stop rewards and values from a later independent match', atol=0)
    horizon = 512
    delayed_rewards = np.zeros(horizon); delayed_rewards[-1] = 1.
    delayed_discounts = np.ones(horizon); delayed_discounts[63::64] = GAMMA_HAND; delayed_discounts[-1] = 0.
    hand = scalar_gae(delayed_rewards, delayed_discounts, np.zeros(horizon), 0., cfg['gae_lambda'], True)
    turn = scalar_gae(delayed_rewards, delayed_discounts, np.zeros(horizon), 0., cfg['gae_lambda'], False)
    close(hand[0], (GAMMA_HAND * cfg['gae_lambda']) ** 7, 'A late result decays once per completed hand')
    check(hand[0] > 10. * turn[0], 'Hand-clock trace retains distant match credit')

    # Replay the actual collector's PRNG/game path independently. This validates
    # real engine hand boundaries, actor signs, normalisation, and bootstrapping
    # through terminal and time-limit resets, not just a duplicate helper.
    game_cfg = learning_config(dict(cfg, n_env=4, rollout_len=132, minibatch_size=4,
                                   bootstrap_minibatch_size=4, max_training_match_hands=1, gae_per_hand=True))
    root_state = initialize_training(game_cfg)
    params = jax.tree_util.tree_map(lambda x: x, root_state.params)
    params['critic_network']['output']['kernel'] = .03 * jax.random.normal(
        jax.random.PRNGKey(344), params['critic_network']['output']['kernel'].shape)
    root_state = root_state._replace(params=params,
        envs=root_state.envs._replace(scores=jnp.array([[1000, -1000], [-1000, -1000],
                                                       [-1000, -1000], [-1000, -1000]], jnp.int32)),
        league_kind=jnp.zeros(4, jnp.int32))
    game_cfg = dict(game_cfg, selfplay_prob=1., snapshot_prob=0.)
    game_trainer = make_train_step(game_cfg)
    after, real_batch, real_metrics = game_trainer.collect(root_state)
    def reference_step(carry, unused):
        envs, key = carry
        key, ka, ke, _, _ = jax.random.split(key, 5)
        legal = jax.vmap(legal_mask)(envs)
        logits = policy_logits(params, jax.vmap(observe)(envs), game_cfg)
        action = jax.random.categorical(ka, jnp.where(legal, logits, -1e9), axis=-1).astype(jnp.int32)
        value = value_apply(params, jax.vmap(critic_observe)(envs), game_cfg)
        who = jax.vmap(actor)(envs)
        next_envs, reward, done, info = batch_step(envs, action, jax.random.split(ke, 4))
        # The optional score bonus is specified independently in the test.
        reward += game_cfg['hand_score_reward_coef'] * (info['score_delta'][:, 0] - info['score_delta'][:, 1]) / 100.
        truncated = info['hand_done'] & ~done & (next_envs.hand_number >= 1)
        continuing_value = value_apply(params, jax.vmap(critic_observe)(next_envs), game_cfg)
        reward += jnp.where(truncated, info['discount'] * continuing_value, 0.)
        discount = jnp.where(truncated, 0., info['discount'])
        fresh = jax.vmap(reset)(jax.random.split(jax.random.fold_in(ke, 1), 4))
        next_envs = jax.tree_util.tree_map(lambda new, old: jnp.where(
            truncated.reshape((4,) + (1,) * (old.ndim - 1)), new, old), fresh, next_envs)
        return (next_envs, key), (reward, discount, value, jnp.where(who % 2 == 0, 1., -1.),
                                 (jnp.sum(legal, axis=-1) > 1), done, truncated)
    (reference_envs, reference_key), rows = jax.jit(lambda envs, key: jax.lax.scan(
        reference_step, (envs, key), None, length=132))(root_state.envs, root_state.rng)
    exact_tree(after.envs, reference_envs, 'Independent game replay reaches identical final states')
    exact_tree(after.rng, reference_key, 'Independent game replay advances the same RNG stream')
    rewards, discounts, values, signs, masks, terminals, truncations = map(np.asarray, rows)
    last = np.asarray(value_apply(params, jax.vmap(critic_observe)(reference_envs), game_cfg))
    expected_adv = np.stack([scalar_gae(rewards[:, e], discounts[:, e], values[:, e], last[e],
                                       game_cfg['gae_lambda'], True) for e in range(4)], axis=1)
    close(real_batch.returns.reshape(132, 4), expected_adv + values,
          'Actual collector returns equal independent game replay and scalar hand-clock GAE', atol=2e-5)
    signed = expected_adv * signs
    mean = (signed * masks).sum() / masks.sum()
    variance = (((signed - mean) ** 2) * masks).sum() / masks.sum()
    close(real_batch.advantages.reshape(132, 4), (signed - mean) / np.sqrt(variance + 1e-8),
          'Actual learner advantages use exactly one partnership sign and valid-row normalization', atol=3e-5)
    check(terminals.sum() >= 1 and truncations.sum() >= 3, 'Real replay exercises both actual wins and sampling resets')
    close(real_metrics['matches'], terminals.sum(), 'Only genuine terminal matches enter strength counters', atol=0)
    close(real_metrics['training_truncations'], truncations.sum(), 'Training sampling resets are counted separately', atol=0)


def run_strength_migration_tests():
    """Exercise both checkpoint schemas and exact counters without a GPU run."""
    from tempfile import TemporaryDirectory
    import copy
    import io
    import json
    import hashlib
    import zipfile

    cfg = learning_config(dict(n_env=4, rollout_len=32, ppo_epochs=1,
                               minibatch_size=64, width=16, snapshot_size=2))
    state = initialize_training(cfg)._replace(updates=jnp.int32(40_000_000))
    expected_legacy_steps = 5_120_000_000
    assert state.env_steps == expected_legacy_steps, 'Counters must remain exact beyond 2**32.'
    checks = 0

    def exact(left, right):
        left_leaves, left_tree = jax.tree_util.tree_flatten(jax.device_get(left))
        right_leaves, right_tree = jax.tree_util.tree_flatten(jax.device_get(right))
        assert left_tree == right_tree and len(left_leaves) == len(right_leaves)
        for a, b in zip(left_leaves, right_leaves):
            np.testing.assert_array_equal(a, b)
            assert np.asarray(a).dtype == np.asarray(b).dtype

    with TemporaryDirectory(prefix='spades_migration_tests_') as tmp:
        folder = Path(tmp)
        path = save_checkpoint(state, cfg, [{'kind': 'test'}], folder, cumulative_seconds=12.5)
        with zipfile.ZipFile(path) as source:
            raw_payload = source.read('state.npz')
            manifest = json.loads(source.read('manifest.json'))
            history = source.read('history.json')
        with np.load(io.BytesIO(raw_payload), allow_pickle=False) as archive:
            source_arrays = {name: archive[name].copy() for name in archive.files}
        n_leaves = len(source_arrays)
        assert manifest['training_state_schema'] == 2
        assert manifest['training_revision'] == TRAINING_REVISION
        restored, saved_cfg, restored_history, restored_meta = load_checkpoint(path, cfg)
        exact(state, restored)
        assert restored.env_steps == expected_legacy_steps == restored_meta['env_steps']
        assert saved_cfg == cfg and restored_history == [{'kind': 'test'}]
        checks += 1

        def archive_case(name, *, legacy=False, schema='absent', revision='absent',
                         mutate_arrays=None, mutate_manifest=None):
            arrays = {key: value.copy() for key, value in source_arrays.items()}
            meta = copy.deepcopy(manifest)
            if legacy:
                for i in range(n_leaves - 3, n_leaves):
                    arrays.pop(f'leaf_{i:05d}')
                meta['leaves'] = meta['leaves'][:-3]
                meta.pop('training_state_schema', None)
                meta.pop('training_revision', None)
            if schema != 'absent':
                meta['training_state_schema'] = schema
            if revision != 'absent':
                meta['training_revision'] = revision
            if mutate_arrays:
                mutate_arrays(arrays, meta)
            payload = io.BytesIO()
            np.savez(payload, **arrays)
            payload = payload.getvalue()
            meta['sha256'] = hashlib.sha256(payload).hexdigest()
            if mutate_manifest:
                mutate_manifest(meta)
            target = folder / (name + '.zip')
            with zipfile.ZipFile(target, 'w') as archive:
                archive.writestr('manifest.json', json.dumps(meta))
                archive.writestr('state.npz', payload)
                archive.writestr('history.json', history)
            return target

        for name, schema, revision in [
            ('original_missing_revision', 'absent', 'absent'),
            ('original_v1', 'absent', 'potential-only-v1'),
            ('recovery_v2', 'absent', 'score-feedback-bidding-anchor-v2'),
            ('explicit_schema1', 1, 'score-feedback-bidding-anchor-v2'),
        ]:
            legacy_path = archive_case(name, legacy=True, schema=schema, revision=revision)
            before = hashlib.sha256(legacy_path.read_bytes()).hexdigest()
            legacy_state, _, _, legacy_meta = load_checkpoint(legacy_path)
            exact(state, legacy_state)
            assert legacy_state.env_steps == expected_legacy_steps
            assert tuple(int(x) for x in legacy_state[-3:]) == (0, 0, 0)
            assert hashlib.sha256(legacy_path.read_bytes()).hexdigest() == before
            # Inspection must not silently rewrite the source's version metadata.
            if schema == 'absent':
                assert 'training_state_schema' not in legacy_meta
            checks += 1

        # A batch change begins a new exact integer segment, preserving old work.
        resized_cfg = learning_config(dict(cfg, n_env=8, minibatch_size=128))
        resized = initialize_training(resized_cfg)._replace(
            updates=state.updates, base_updates=state.updates,
            env_steps_low=jnp.uint32(expected_legacy_steps & 0xffffffff),
            env_steps_high=jnp.uint32(expected_legacy_steps >> 32))
        assert resized.env_steps == expected_legacy_steps
        resized = resized._replace(updates=resized.updates + jnp.int32(3))
        assert resized.env_steps == expected_legacy_steps + 3 * 256
        resized_path = save_checkpoint(resized, resized_cfg, [], folder / 'resized')
        resized_back, _, _, resized_meta = load_checkpoint(resized_path)
        exact(resized, resized_back)
        assert resized_meta['env_steps'] == expected_legacy_steps + 768
        checks += 1

        # Exercise the actual public migration, including the parameter choices.
        changed_params = dict(state.params)
        changed_params['actor_network'] = jax.tree_util.tree_map(
            lambda x: x + jnp.asarray(.125, x.dtype), state.params['actor_network'])
        changed_params['critic_network'] = jax.tree_util.tree_map(
            lambda x: x - jnp.asarray(.25, x.dtype), state.params['critic_network'])
        changed = state._replace(params=changed_params, bootstrap_updates=jnp.int32(9),
                                 bootstrap_complete=jnp.bool_(True))
        source_copy = jax.device_get(changed)
        for reset_critic in (False, True):
            migrated = upgrade_training_workload(changed, resized_cfg, reset_critic=reset_critic)
            exact(changed, source_copy)
            exact(migrated.params['actor_network'], changed.params['actor_network'])
            wanted_critic = (initialize_training(resized_cfg).params['critic_network']
                             if reset_critic else changed.params['critic_network'])
            exact(migrated.params['critic_network'], wanted_critic)
            exact(migrated.opt_state, _optimizer(resized_cfg).init(migrated.params))
            assert migrated.env_steps == expected_legacy_steps
            assert int(migrated.updates) == int(migrated.base_updates) == int(changed.updates)
            assert int(migrated.decisions_per_update) == 256
            assert int(migrated.bootstrap_updates) == 0 and not bool(migrated.bootstrap_complete)
            assert len(migrated.envs.turn) == 8 and np.all(np.asarray(migrated.envs.turn) == 0)
            migrated_path = save_checkpoint(migrated, resized_cfg, [], folder / ('upgrade_' + str(reset_critic)))
            migrated_back, _, _, _ = load_checkpoint(migrated_path)
            exact(migrated, migrated_back)
            checks += 1

        # The uint32 high/low pair preserves offsets far beyond a 32-bit count.
        enormous = (1 << 48) + 123
        huge = state._replace(updates=jnp.int32(13), base_updates=jnp.int32(10),
                              env_steps_low=jnp.uint32(enormous & 0xffffffff),
                              env_steps_high=jnp.uint32(enormous >> 32))
        huge_path = save_checkpoint(huge, cfg, [], folder / 'huge')
        huge_back, _, _, huge_meta = load_checkpoint(huge_path)
        assert huge_back.env_steps == huge_meta['env_steps'] == enormous + 384
        exact(huge, huge_back)
        checks += 1

        def rejected(name, **kwargs):
            nonlocal checks
            bad_path = archive_case(name, **kwargs)
            before = hashlib.sha256(bad_path.read_bytes()).hexdigest()
            try:
                load_checkpoint(bad_path)
            except (ValueError, IOError):
                checks += 1
            else:
                raise AssertionError('Invalid checkpoint accepted: ' + name)
            assert hashlib.sha256(bad_path.read_bytes()).hexdigest() == before

        def pop_leaf(arrays, meta):
            arrays.pop(sorted(arrays)[-1])
            meta['leaves'].pop()

        def append_leaf(arrays, meta):
            arrays[f'leaf_{len(arrays):05d}'] = np.asarray(0, np.int32)
            meta['leaves'].append({'shape': [], 'dtype': 'int32'})

        rejected('legacy_short', legacy=True, mutate_arrays=pop_leaf)
        rejected('legacy_extra', legacy=True, mutate_arrays=append_leaf)
        rejected('current_short', mutate_arrays=pop_leaf)
        rejected('current_extra', mutate_arrays=append_leaf)
        rejected('legacy_claiming_new_schema', legacy=True, schema=2, revision=TRAINING_REVISION)
        rejected('current_claiming_legacy', schema=1, revision='score-feedback-bidding-anchor-v2')
        rejected('unknown_schema', schema=3)
        rejected('schema_boolean', schema=True, revision='score-feedback-bidding-anchor-v2')
        rejected('unknown_revision', revision='future-unknown-trainer')
        rejected('legacy_unknown_revision', legacy=True, revision='future-unknown-trainer')
        rejected('current_missing_revision', mutate_manifest=lambda m: m.pop('training_revision'))
        rejected('negative_manifest_steps', mutate_manifest=lambda m: m.update(env_steps=-1))
        rejected('float_manifest_steps', mutate_manifest=lambda m: m.update(env_steps=5120000000.0))
        rejected('boolean_manifest_updates', mutate_manifest=lambda m: m.update(updates=True))
        rejected('mismatched_manifest_steps', mutate_manifest=lambda m: m.update(env_steps=123))
        rejected('mismatched_manifest_updates', mutate_manifest=lambda m: m.update(updates=123))
        rejected('wrong_checksum', mutate_manifest=lambda m: m.update(sha256='0' * 64))

        def wrong_base(arrays, meta):
            arrays[f'leaf_{n_leaves - 1:05d}'] = np.asarray(40_000_001, np.int32)
        rejected('base_after_updates', mutate_arrays=wrong_base)

        def wrong_counter_dtype(arrays, meta):
            key = f'leaf_{n_leaves - 3:05d}'
            arrays[key] = np.asarray(0, np.int32)
            meta['leaves'][-3]['dtype'] = 'int32'
        rejected('signed_counter_word', mutate_arrays=wrong_counter_dtype)

        def nonfinite_moment(arrays, meta):
            keyed, _ = jax.tree_util.tree_flatten_with_path(state)
            index = next(i for i, (key, value) in enumerate(keyed)
                         if jax.tree_util.keystr(key).startswith('.opt_state')
                         and np.issubdtype(np.asarray(value).dtype, np.inexact))
            arrays[f'leaf_{index:05d}'].flat[0] = np.nan
        rejected('legacy_nonfinite_moment', legacy=True, mutate_arrays=nonfinite_moment)
        rejected('current_nonfinite_moment', mutate_arrays=nonfinite_moment)

    print(f'Checkpoint migration checks passed: {checks}; exact legacy/new restores, '
          f'{n_leaves} new leaves, resized counters above 2**32, corrupt input rejected.')
    return {'passed': True, 'checks': checks, 'state_leaves': n_leaves}


def run_strength_runtime_flow_tests():
    """Check baseline-before-refresh, resume deduplication, and GPU observation cadence."""
    from tempfile import TemporaryDirectory
    from types import SimpleNamespace
    import copy

    cfg = learning_config(dict(n_env=4, rollout_len=32, ppo_epochs=1,
                               minibatch_size=64, width=16, snapshot_size=2))
    state = initialize_training(cfg)._replace(updates=jnp.int32(5))
    namespace = globals()
    absent = object()
    names = ('time', 'evaluate_policy', 'gpu_train_step', 'gpu_memory_metrics',
             'make_bootstrap_step', 'finish_bootstrap', 'show_training_metrics',
             'display', 'save_checkpoint')
    originals = {name: namespace.get(name, absent) for name in names}
    real_save = save_checkpoint
    clock = SimpleNamespace(now=0., ppo_seconds=.8)
    operations, shown, telemetry_calls = [], [], []

    def fake_gpu_train_step(config):
        assert config == cfg
        operations.append(('compile_hook',))
        def step(current):
            operations.append(('ppo', int(current.updates)))
            clock.now += clock.ppo_seconds
            return current._replace(updates=current.updates + 1), dict(
                finite=jnp.float32(1.), illegal_rate=jnp.float32(0.), loss=jnp.float32(.25))
        return step

    def fake_bootstrap_step(config):
        assert config == cfg
        def step(current):
            operations.append(('teacher', int(current.bootstrap_updates)))
            clock.now += .6
            return current._replace(bootstrap_updates=current.bootstrap_updates + 1), dict(
                finite=jnp.float32(1.), bootstrap_loss=jnp.float32(.25), bootstrap_accuracy=jnp.float32(.75))
        return step

    def fake_evaluate(params, config, **kwargs):
        assert config == cfg
        assert kwargs['styles'] == [7] and kwargs['games_per_style'] == 32
        operations.append(('evaluate',))
        clock.now += .25
        first = sum(op[0] == 'evaluate' for op in operations) == 1
        # A worse post-refresh result must not overwrite the transferred best actor.
        return pd.DataFrame([dict(opponent='adaptive', games=32, wins=24 if first else 8,
            losses=8 if first else 24, draws=0, unresolved=0, unresolved_fraction=0.,
            illegal_actions=0, win_rate=.75 if first else .25,
            ci_low=.5 if first else .1, ci_high=.9 if first else .4)])

    def fake_memory():
        telemetry_calls.append(clock.now)
        return dict(jax_live_gib=1., jax_reserved_gib=13.,
                    board_used_gib=14., gpu_target_gib=14.,
                    gpu_optional_sensor=float('nan'))

    def saving(*args, **kwargs):
        operations.append(('save', kwargs.get('tag', 'checkpoint')))
        return real_save(*args, **kwargs)

    try:
        namespace.update(time=SimpleNamespace(monotonic=lambda: clock.now),
            evaluate_policy=fake_evaluate, gpu_train_step=fake_gpu_train_step,
            gpu_memory_metrics=fake_memory, make_bootstrap_step=fake_bootstrap_step,
            finish_bootstrap=lambda current, config: current._replace(bootstrap_complete=jnp.bool_(True)),
            show_training_metrics=lambda history, latest, *a: shown.append(copy.deepcopy(latest)),
            display=lambda *a, **kw: None, save_checkpoint=saving)
        with TemporaryDirectory(prefix='spades_runtime_flow_') as tmp:
            config = dict(checkpoint_dir=str(Path(tmp) / 'upgrade'), duration_hours=1 / 3600,
                          checkpoint_hours=1.5, metrics_every_seconds=30,
                          eval_every_minutes=30, bootstrap_updates=3, keep_checkpoints=20,
                          eval_games=32, development_seed=81073, eval_max_hands=160)
            history = [dict(kind='evaluation', selection_score=.99, rows=[dict(unresolved=0, illegal_actions=0)]),
                       dict(kind='trainer_upgrade', update=5)]
            first, history, elapsed = train_segment(state, cfg, history, config)
            assert int(first.bootstrap_updates) == 2 and not bool(first.bootstrap_complete)
            assert elapsed >= 1 and elapsed < 2
            action_order = [op[0] for op in operations if op[0] in ('evaluate', 'teacher', 'ppo')]
            assert action_order == ['evaluate', 'teacher', 'teacher']
            evaluations = [row for row in history if row.get('stage') == 'before_teacher_refresh']
            assert len(evaluations) == 1 and evaluations[0]['update'] == 5
            assert evaluations[0]['selection_score'] == .5
            assert len(telemetry_calls) == 1, 'Teacher steps must not poll GPU telemetry per minibatch.'
            (restored, _, saved_history, _), _ = find_resume_checkpoint(config['checkpoint_dir'])
            assert len([row for row in saved_history if row.get('stage') == 'before_teacher_refresh']) == 1
            assert int(restored.bootstrap_updates) == 2

            clock.ppo_seconds = 100.
            config['duration_hours'] = 300 / 3600
            before_telemetry = len(telemetry_calls)
            resumed, saved_history, _ = train_segment(restored, cfg, saved_history, config, elapsed)
            assert bool(resumed.bootstrap_complete)
            evaluations = [row for row in saved_history if row.get('kind') == 'evaluation' and row.get('stage')]
            assert [row['stage'] for row in evaluations] == ['before_teacher_refresh', 'development']
            assert sum(op[0] == 'evaluate' for op in operations) == 2
            assert len(shown) == 3
            assert len(telemetry_calls) - before_telemetry == 1 + len(shown)
            assert all(row['training_revision'] == TRAINING_REVISION for row in shown)
            assert all(row['n_env'] == 4 and row['rollout_len'] == 32
                       and row['minibatch_size'] == 64 and row['width'] == 16 for row in shown)
            assert all(row['jax_live_gib'] == 1. and row['gpu_optional_sensor'] is None for row in shown)
            best_paths = list(Path(config['checkpoint_dir']).glob('best_development_*.zip'))
            assert len(best_paths) == 1
            best, _, best_history, _ = load_checkpoint(best_paths[0])
            assert int(best.updates) == 5 and int(best.bootstrap_updates) == 0
            assert best_history[-1]['stage'] == 'before_teacher_refresh'

            # Starting fresh has no incoming trained actor to benchmark first.
            previous_evaluations = sum(op[0] == 'evaluate' for op in operations)
            fresh_config = dict(config, checkpoint_dir=str(Path(tmp) / 'fresh'), duration_hours=.1 / 3600)
            train_segment(initialize_training(cfg), cfg, [], fresh_config)
            assert sum(op[0] == 'evaluate' for op in operations) == previous_evaluations
    finally:
        for name, value in originals.items():
            if value is absent:
                namespace.pop(name, None)
            else:
                namespace[name] = value
    print('Runtime flow checks passed: incoming benchmark before teacher, persisted resume deduplication, '
          'best actor preserved, revision/batch labels, GPU telemetry outside update checks.')
    return {'passed': True, 'checks': 6}


In [ ]:
def run_checkpoint_smoke_test():
    """Verify a mid-hand restore and the next real PPO update using temporary files."""
    from tempfile import TemporaryDirectory
    cfg = learning_config(dict(n_env=4, rollout_len=32, ppo_epochs=1,
                               minibatch_size=64, width=16, snapshot_size=2))
    state = initialize_training(cfg)._replace(bootstrap_complete=jnp.bool_(True))
    train = make_train_step(cfg)
    state, metrics = train(state)
    assert bool(metrics['finite']) and float(metrics['illegal_rate']) == 0.
    with TemporaryDirectory(prefix='spades_checkpoint_check_') as tmp:
        path = save_checkpoint(state, cfg, [{'kind': 'test'}], tmp, cumulative_seconds=12.5)
        restored, saved_cfg, history, metadata = load_checkpoint(path, cfg)
        before, tree = jax.tree_util.tree_flatten(jax.device_get(state))
        after, restored_tree = jax.tree_util.tree_flatten(jax.device_get(restored))
        assert tree == restored_tree and len(before) == len(after)
        for a, b in zip(before, after):
            np.testing.assert_array_equal(a, b)
        assert history == [{'kind': 'test'}] and metadata['cumulative_seconds'] == 12.5
        uninterrupted, m1 = train(state)
        resumed, m2 = train(restored)
        # Same hardware and executable: restoring data and RNG must reproduce the update.
        for a, b in zip(jax.tree_util.tree_leaves(jax.device_get(uninterrupted)),
                        jax.tree_util.tree_leaves(jax.device_get(resumed))):
            np.testing.assert_allclose(a, b, rtol=1e-6, atol=1e-7)
        for k in m1:
            np.testing.assert_allclose(np.asarray(m1[k]), np.asarray(m2[k]), rtol=1e-6, atol=1e-7)
    print(f'Checkpoint check passed: {len(before)} state leaves restored; next PPO update matches.')
    return {'passed': True, 'restored_leaves': len(before)}


In [ ]:
EXTENDED_RULE_TESTS = False
EXTENDED_NUMERICAL_TESTS = False
EXTENDED_STRENGTH_TESTS = False
rules_test_report = run_self_tests(quick=not EXTENDED_RULE_TESTS)
learning_test_report = run_learning_tests()
numerical_test_report = run_numerical_tests(extended=EXTENDED_NUMERICAL_TESTS)
strength_test_report = run_strength_tests(extended=EXTENDED_STRENGTH_TESTS)
migration_test_report = run_strength_migration_tests()
checkpoint_test_report = run_checkpoint_smoke_test()
print('Preflight passed. Next: profile GPU memory, restore weights, and train.')

## 10 · Measured GPU workload and safe automatic migration

The cell below profiles useful JAX training buffers, restores the latest compatible checkpoint, and saves a ready state before training. At most three candidate workloads are compiled. The selected environment count, minibatch size, estimated peak, and requested/safe memory budgets are printed. First-use compilation can take several minutes; an exact v3 resume keeps its saved shapes and skips resizing.

The model width remains the saved width when importing a checkpoint. Additional memory goes into parallel games, rollout data, and larger PPO minibatches. The 128-turn horizon and 8 PPO minibatches per epoch stay fixed; teacher minibatches remain 1,024. A changed batch starts a new exact decision-count segment rather than multiplying old updates by the new batch size.

Migration from recovery-v2 preserves actor and critic parameters; migration from the original potential-only trainer resets its pathological critic. Both reset optimizer, games, league, and teacher progress. Original checkpoints remain in their existing folders. The new run saves in `strong_play_v3_runtime_v1`, and subsequent sessions resume all state. The saved training configuration becomes authoritative on exact resume.

To start an independent experiment, choose a new `RUN_NAME`. Timing/evaluation settings remain editable in `RUN_CFG`. Do not rerun configuration inside a healthy session to continue training; rerun the training cell directly.

In [ ]:
if getattr(train_segment, 'runtime_safety_revision', None) != 'checkpoint-memory-safety-v2':
    raise RuntimeError('Stale runtime safety runner. Restart and run the Runtime Safety V2 definitions before continuing.')
# New workload settings. An existing v3 checkpoint resumes its saved config.
# Older runs transfer learned weights into a separate strong_play_v3 folder.
if TRAINING_REVISION != 'strong-teacher-hand-gae-v3':
    raise RuntimeError('Old trainer definitions are active. Restart and Run all in the Strong Play V3 notebook.')
if getattr(train_segment, 'training_revision', None) != TRAINING_REVISION:
    raise RuntimeError('The segment runner is from another notebook revision. Run the new runtime cell first.')

TRAIN_CFG = learning_config({
    'seed': 42, 'n_env': 256, 'rollout_len': 128, 'width': 256,
    'minibatch_size': 4096, 'ppo_epochs': 3, 'lr': 2e-4,
    'gae_lambda': 0.995, 'gae_per_hand': False,
    'entropy_coef': 0.015, 'bidding_anchor_coef': 0.1, 'play_anchor_coef': 0.3,
    'hand_score_reward_coef': 0.0,
    'teacher_style': 7, 'teacher_decline_blind': True,
    'teacher_warmup_decisions': 6_291_456,
    'bootstrap_minibatch_size': 1024, 'bootstrap_epochs': 2,
    'max_training_match_hands': 128,
})
RUN_CFG = {
    'checkpoint_dir': CHECKPOINT_DIR,
    'duration_hours': 3.0, 'checkpoint_hours': 1.5,
    'metrics_every_seconds': 30.0, 'eval_every_minutes': 30.0,
    'eval_games': 32, 'eval_max_hands': 160, 'development_seed': 81073,
    'keep_checkpoints': 3,
}

if RESUME_FILE:
    loaded, resumed_path = load_checkpoint(RESUME_FILE), Path(RESUME_FILE)
elif RESUME:
    loaded, resumed_path = None, None
    for candidate_folder in (CHECKPOINT_DIR, PRIOR_V3_CHECKPOINT_DIR, RECOVERY_CHECKPOINT_DIR, LEGACY_CHECKPOINT_DIR):
        if loaded is None:
            loaded, resumed_path = find_resume_checkpoint(candidate_folder)
else:
    if list(Path(CHECKPOINT_DIR).glob('checkpoint_*.zip')):
        raise ValueError('This v3 run already has checkpoints. Resume it or choose another RUN_NAME.')
    loaded, resumed_path = None, None

if loaded is None:
    TRAIN_CFG, GPU_WORKLOAD_REPORT = configure_gpu_workload(TRAIN_CFG)
    gpu_admit_workload(TRAIN_CFG, "new training state")
    training_state = initialize_training(TRAIN_CFG)
    training_history, total_seconds = [], 0.0
    print('New STRONG PLAY V3 run:', CHECKPOINT_DIR)
else:
    training_state, saved_cfg, training_history, metadata = loaded
    total_seconds = float(metadata.get('cumulative_seconds', 0.))
    old_revision = metadata.get('training_revision', 'potential-only-v1')
    if old_revision == TRAINING_REVISION:
        TRAIN_CFG, GPU_WORKLOAD_REPORT = configure_gpu_workload(saved_cfg, exact_resume=True)
        print('Exact v3 continuation:', resumed_path)
    elif old_revision in ('potential-only-v1', 'score-feedback-bidding-anchor-v2'):
        if list(Path(CHECKPOINT_DIR).glob('checkpoint_*.zip')):
            raise ValueError('A v3 run already exists. Clear RESUME_FILE to resume it, '
                             'or select another RUN_NAME for a separate experiment.')
        old_decisions = training_state.env_steps
        # Keep the saved neural architecture/weights and its ordinary PPO settings.
        # These new workload/teacher settings are explicit changes for this branch.
        changed = ('n_env', 'rollout_len', 'minibatch_size', 'gae_lambda', 'gae_per_hand',
                   'bidding_anchor_coef', 'play_anchor_coef', 'hand_score_reward_coef',
                   'teacher_style', 'teacher_decline_blind', 'teacher_warmup_decisions',
                   'bootstrap_minibatch_size', 'bootstrap_epochs', 'max_training_match_hands')
        proposed = learning_config({**saved_cfg, **{name: TRAIN_CFG[name] for name in changed}})
        TRAIN_CFG, GPU_WORKLOAD_REPORT = configure_gpu_workload(proposed)
        reset_critic = old_revision == 'potential-only-v1'
        gpu_admit_workload(TRAIN_CFG, "explicit legacy workload migration")
        training_state = upgrade_training_workload(training_state, TRAIN_CFG, reset_critic=reset_critic)
        assert training_state.env_steps == old_decisions
        training_history = list(training_history) + [dict(
            kind='trainer_upgrade', update=int(training_state.updates),
            env_steps=training_state.env_steps, source_checkpoint=str(resumed_path),
            from_revision=old_revision, to_revision=TRAINING_REVISION,
            actor_transferred=True, critic_transferred=not reset_critic,
            reset=['optimizer', 'games', 'league', 'teacher_progress'] + (['critic'] if reset_critic else []),
            previous_config=saved_cfg, new_config=TRAIN_CFG)]
        print('Transferred learned weights from:', resumed_path)
        print('Lifetime decisions preserved:', f'{old_decisions:,}')
        print('Optimizer, games, league and teacher progress reset for the larger workload.')
        print('Original and v2 checkpoints remain untouched.')
    else:
        raise ValueError(f'Unsupported trainer revision {old_revision!r}.')

RUN_CFG['bootstrap_updates'] = math.ceil(TRAIN_CFG['teacher_warmup_decisions'] /
                                       (TRAIN_CFG['n_env'] * TRAIN_CFG['rollout_len']))
training_history = list(training_history) + [dict(
    kind='gpu_workload', update=int(training_state.updates), report=GPU_WORKLOAD_REPORT)]
configuration_path = save_checkpoint(training_state, TRAIN_CFG, training_history,
    CHECKPOINT_DIR, cumulative_seconds=total_seconds, keep=RUN_CFG['keep_checkpoints'])
parameter_count = sum(x.size for x in jax.tree_util.tree_leaves(training_state.params))
print(f'READY — {TRAINING_REVISION} | {parameter_count:,} parameters')
print(f'Workload: {TRAIN_CFG["n_env"]:,} games × {TRAIN_CFG["rollout_len"]} turns; '
      f'PPO minibatch {TRAIN_CFG["minibatch_size"]:,}; teacher minibatch {TRAIN_CFG["bootstrap_minibatch_size"]:,}')
print(f'PPO update {int(training_state.updates):,} | lifetime PPO decisions {training_state.env_steps:,}')
print(f'Teacher budget: {TRAIN_CFG["teacher_warmup_decisions"]:,} decisions; '
      f'{RUN_CFG["bootstrap_updates"]} batches × {TRAIN_CFG["bootstrap_epochs"]} epochs, once per upgrade.')
print('Saved ready state:', configuration_path)
print('Next: run the training cell. Live plots include fixed-opponent wins and GPU memory.')


## 11 · Optional short smoke segment

Leave this disabled for the requested three-hour segment. A short smoke segment can stop during teacher initialization; its saved progress continues on the next training-cell run. It does not replace the GPU profiler or establish full-match strength.

In [ ]:
SHORT_SMOKE_SEGMENT = False
RUN_CFG['duration_hours'] = 0.02 if SHORT_SMOKE_SEGMENT else 3.0
RUN_CFG['checkpoint_hours'] = 0.01 if SHORT_SMOKE_SEGMENT else 1.5
print(f"Next segment: {RUN_CFG['duration_hours']} h; checkpoints every {RUN_CFG['checkpoint_hours']} h.")

## 12 · Train for the next three-hour segment

**Rerun this cell for each additional segment.** On the first upgrade, it benchmarks the transferred actor, performs the teacher refresh, and enters PPO. Warmup progress is saved and does not restart on later segments. Live reports show actual fixed-opponent wins and GPU memory.

Stop normally and wait for **Saved** before switching notebooks or restarting the runtime. Every 90 minutes and at segment end, the last accepted state is checkpointed. A forced VM deletion can lose work since the last durable save. The first compilation or benchmark can take longer than a regular update.
Rerun only after a healthy completed segment. For an error or optimizer health stop, inspect the printed recovery ZIP and journal first; do not rerun unchanged. The stop checkpoint preserves accepted state. After a reviewed fix, use the exact stop-ID/update/config acknowledgement shown at the top to reset only its rolling health window.


In [ ]:
if getattr(train_segment, 'runtime_safety_revision', None) != 'checkpoint-memory-safety-v2':
    raise RuntimeError('Stale runtime safety runner. Restart and run the Runtime Safety V2 definitions before continuing.')
# Rerun this cell for each additional three-hour segment.
if TRAINING_REVISION != 'strong-teacher-hand-gae-v3' or getattr(
        train_segment, 'training_revision', None) != TRAINING_REVISION:
    raise RuntimeError('Stale trainer definitions. Restart and Run all in Strong Play V3.')
# Recovery belongs to this invocation. Preflight failures must never adopt state
# left by another run or an optional test in this Colab session.
LAST_RUN_RECOVERY = (training_state, training_history, total_seconds)
try:
    training_state, training_history, total_seconds = train_segment(
        training_state, TRAIN_CFG, training_history, RUN_CFG, total_seconds)
except Exception:
    if 'LAST_RUN_RECOVERY' in globals():
        training_state, training_history, total_seconds = LAST_RUN_RECOVERY
    raise


## 13 · Optional held-out benchmark

Turn this on after a training segment or after loading a selected checkpoint. It plays every scripted style and the stored historical policies on two fresh seeds, with 128 matches per opponent per seed. Keep the evaluated policy fixed. Reusing these same seeds to choose the next model turns them into development data; choose additional new seeds for the next final assessment.

A useful target is consistently strong performance against `adaptive`, `card_counter`, Nil attackers, bag-focused opponents, and earlier neural policies, with narrow intervals and few unresolved games. Human play supplies additional evidence that bot benchmarks miss. This notebook does not assign an Elo rating from a small bot pool.

In [ ]:
# Optional held-out assessment. Turn on after training; keep these seeds separate
# from the fixed development seed used to choose best_development checkpoints.
RUN_FULL_BENCHMARK = False
if RUN_FULL_BENCHMARK:
    frozen_params = training_state.params
    snapshots = {
        f'league_{i}': jax.tree_util.tree_map(lambda x, i=i: x[i], training_state.snapshot_params)
        for i in range(int(training_state.snapshot_count))
    }
    reports = []
    for heldout_seed in (99173, 132701):
        reports.append(evaluate_policy(
            frozen_params, TRAIN_CFG, styles=None, games_per_style=128,
            seed=heldout_seed, max_hands=300, snapshot_opponents=snapshots))
    holdout_report = pd.concat(reports, ignore_index=True)
    display(holdout_report[['opponent', 'seed', 'wins', 'losses', 'draws', 'unresolved',
                            'win_rate', 'ci_low', 'ci_high', 'paired_ci_low', 'paired_ci_high',
                            'contract_success_rate', 'mean_hands', 'illegal_actions']])
    report_path = Path(CHECKPOINT_DIR) / f'holdout_update_{int(training_state.updates):09d}.csv'
    holdout_report.to_csv(report_path, index=False)
    print('Report saved:', report_path)
else:
    print('Set RUN_FULL_BENCHMARK=True when ready for the optional final benchmark.')


## 14 · Play against the AI, or with an AI partner

Enable the next cell to play a complete match with the AI as your partner. Change the opponent names to any entries in `HEURISTIC_NAMES`, or use `('ai', 'ai')` for two neural opponents. Enter cards as `AS`, `10H`, `2C`, etc., normal bids as 0–13, and `q` to quit. The table shows only your own hand and public information. For optional search, set `USE_SEARCH=True`; first-use compilation may be noticeable.

In [ ]:
# Optional: human seat0 + AI seat2 versus two configurable opponents.
PLAY_A_MATCH = False
USE_SEARCH = False  # First search call compiles; start with late-hand analysis if slow.
if PLAY_A_MATCH:
    SEARCH_CFG = dict(TRAIN_CFG, search_temperature=0.6, search_heuristic_fraction=0.0)
    human_result = human_play_match(
        training_state.params, SEARCH_CFG, opponents=('adaptive', 'card_counter'),
        seed=20261014, human_seat=0, search=USE_SEARCH, determinizations=16)
else:
    print('Opponent styles:', ', '.join(HEURISTIC_NAMES))
    print('Set PLAY_A_MATCH=True to play. Enter q at a prompt to stop.')


## 15 · Inspect progress and select a saved model

The latest regular checkpoint is the exact continuation point. `best_development_*.zip` is the strongest checkpoint under the fixed development selection rule, which may differ from the latest. To evaluate that selected model, use the cell below. This does not replace the active training state or its optimizer.

In [ ]:
INSPECT_BEST_MODEL = False
if INSPECT_BEST_MODEL:
    paths = sorted(Path(CHECKPOINT_DIR).glob('best_development_*.zip'),
                   key=_checkpoint_creation_key)
    if not paths:
        # Exact V3 resumes retain the earlier best score. Until it is beaten,
        # that immutable checkpoint remains the supported selection.
        paths = sorted(Path(PRIOR_V3_CHECKPOINT_DIR).glob('best_development_*.zip'),
                       key=_checkpoint_creation_key)
    if not paths:
        print('No best-development checkpoint yet; let a periodic benchmark finish with all games resolved.')
    else:
        selected_state, selected_cfg, selected_history, selected_metadata = load_checkpoint(paths[-1])
        print('Selected (read only):', paths[-1])
        display(evaluate_policy(selected_state.params, selected_cfg,
                                styles=['adaptive', 'card_counter', 'nil_hunter', 'bag_averse'],
                                games_per_style=128, seed=707123, max_hands=300))

## Practical training notes

- Use fixed-opponent wins, intervals, unresolved counts, and fresh held-out deals to judge improvement. The reported 3/32 benchmark was weak despite healthy 75% training contracts.
- The model does not become stronger just because more GPU memory is allocated. This notebook uses larger real batches and reports their actual throughput and memory.
- A 14 GiB target leaves safety headroom for CUDA, evaluation, and compilation. Compiler estimates and measured usage can differ. If no profiled candidate fits, the notebook stops before changing the live training state.
- Training initialization and PPO share the three-hour segment budget. Setup and GPU profiling precede that budget. Checkpoints resume teacher progress as well as PPO.
- Use the best-development checkpoint for assessment when the latest policy regresses. Keep final evaluation seeds separate from model selection.
- Everything required to run is embedded. Drive checkpoint ZIPs, logs, and optional CSV reports are generated outputs, not additional uploads.
- Use `RESUME_FILE` for an exact native ZIP continuation. Keep the printed health/emergency ZIP and export local recovery files before runtime disposal. Health-stop history is diagnostic and never an automatic license to retry or promote a benchmark.


## Original supplied rules

The following is preserved as the reference specification for this notebook.

<details><summary>Expand the full rules</summary>

Players
4 players in two partnerships.
Deck
Standard 52-card deck. Ace is high; 2 is low. For card comparison, the rank order from highest to lowest is Ace, King, Queen, Jack, 10, 9, 8, 7, 6, 5, 4, 3, 2.
Goal
To be the first partnership to reach 500 points.
Setup
Partners sit across the table from each other. Both partnerships begin the game with 0 points and 0 bags. The engine randomly selects the first dealer, who will shuffle the deck and deal each player 13 cards, one at a time, face-down, in a clockwise direction, starting with the player to the dealer's left. Players must leave their dealt cards face-down on the table and cannot look at them until all players have had the opportunity to declare a Double Nil. At the start of every hand, the point scores, cumulative bag counts, and dealer position from the previous hand are carried forward, while the deck, hands, bids, trick counts, and played-card information are reset for the new hand.
Bidding
The player to the dealer's left bids first. Each player looks at his cards and bids, indicating a number of tricks. Partners' bids are added; the sum is the number of tricks that partnership must win to score points. (Exception: If a player bids Nil or Double Nil, the bids are not added together; the Nil or Double Nil bidder must win exactly zero tricks, while their partner must independently win at least their specifically bid number of tricks.) Any number from 0 ("Nil") to 13 is a legal normal bid for each player. Players are not allowed to pass. Bids do not have to increase with each player. There's only one round of bidding. A team may legally have a combined normal bid greater than 13; such a bid cannot be successfully completed because only 13 tricks exist in a hand. Example: Alex bids 4. Then Beth bids 3, Charlie bids 2 and David bids 2. Alex and Charlie need to win at least 6 tricks; Beth and David need to win at least 5.
Bidding Nil
A player who bids Nil (zero) is claiming that they won't win any tricks during the hand. If the player is successful, their partnership earns a 100-point bonus, which is added to the points earned by their partner successfully making their own bid (e.g., if Joe bids zero and takes zero, and Sally bids three and takes exactly three, the team earns 130 points). However, if the player wins one or more tricks, the Nil bid fails and the entire team fails. Their partnership receives a penalty of 100 points plus the point value of the partner's bid. For scoring purposes, the point value of a normal bid is 10 times the number bid. Ex. Joe and Sally are partners. If Joe bids zero and Sally bids three, but Joe wins a trick, the Nil bid fails and they lose. Even if Sally won five tricks (two more than her bid), the team would lose exactly 130 points. That team would not have bags added to their bag count if they lose. If a player bids Nil and their partner bids a number, then the partner must still try to win at least that number of tricks. If the Nil is successful but the partner fails to meet their bid, the entire team fails and receives a penalty of 100 points plus the point value of the partner's bid (e.g., if Joe bids zero and takes zero, but Sally bids three and takes two, the team loses exactly 130 points). Example: Alex bids 4. Charlie, her partner, bids Nil. Charlie will play the hand trying not to win any tricks. However, Alex needs to win at least four tricks. Note: It's legal for partners to both bid Nil. If both are successful, the partnership earns a 200-point bonus. However, if both partners fail, the partnership receives a 200-point penalty. If one partner succeeds and the other fails, the entire team fails and receives the full combined penalty of 200 points.
Double Nil
Before any bids are made, the engine checks the two partnerships' current scores. A player may bid Double Nil, also known as Blind Nil, before looking at their cards, but only if their partnership is trailing the opposing partnership by 150 points or more at the start of the hand. If the partnership is trailing by less than 150 points, is tied, or is ahead, its players are not allowed to bid Double Nil. This eligibility check occurs before any player is polled and before any other bid, including any other Double Nil declaration, is made. Only the eligible trailing partnership may have one or both players bid Double Nil. It is legal for both partners on the eligible team to bid Double Nil.
To determine Double Nil bids, all four players are polled in a distinct phase in clockwise order, starting from the player to the dealer's left. Players must keep their cards face-down during this phase. Players on an ineligible team may not declare Double Nil. After all four players have been polled, players may pick up and look at their hands. If a Double Nil was bid, both players on that team then swap one card face-down at the same time. Each player on a Double Nil team selects exactly one card from their own hand to pass. The engine must collect both selected cards before completing the exchange so that neither player can gain information from the other player's selection. They may not communicate what card to pass or the strategy they are going to do. After all swaps are completed, regular bidding begins with the player to the dealer's left and proceeds clockwise, skipping any Double Nil bidders. Double Nil bidders do not make a second bid.
A player bidding Double Nil is making a blind commitment to win exactly zero tricks. If they are successful, their partnership earns a 200-point bonus. However, if they fail, the entire team fails. Their partnership receives a penalty of 200 points plus the point value of the partner's normal bid. If the Double Nil is successful but the partner fails to meet their bid, the entire team fails and receives a penalty of 200 points plus the point value of the partner's bid (e.g., if Sam bids Double Nil and takes zero tricks, but Cal bids four tricks and takes three, the team loses exactly 240 points). If the Double Nil player takes one or more tricks, the Double Nil fails immediately for scoring purposes and the entire team fails; the team still receives the same fixed penalty based on the Double Nil and the partner's bid, regardless of the number of tricks either player won.
NOTE: It's legal for partners to both bid Double Nil. If both are successful, the partnership earns a 400-point bonus. If either partner fails, the entire team fails and receives the full combined penalty of 400 points. Ex. If Sam and Cal are partners, and Sam takes a trick after going blind Nil, the team fails. If Cal bid 4 tricks, then they would lose 240 points. It is also legal for one partner to bid Nil and the other to bid Double Nil. If both succeed, the partnership earns a 300-point bonus, but if one or both fail, the entire team fails and receives a 300-point penalty. In every combination involving Nil and/or Double Nil, all special bids must succeed and every normal-bid partner must individually meet at least their own normal bid for the team to score a successful contract.
Gameplay
The player to the dealer's left plays first ("leads"). A trick consists of exactly four cards, one played by each player in clockwise order. They may lead spades. Spades do not have to be broken to be led. Play continues clockwise. Each player must follow suit (i.e. play the same suit that was led) if possible. The engine must enforce this rule by making every card outside the led suit illegal whenever the player has at least one card of the led suit. No illegal off-suit action should be available to a player in that situation, so the probability of selecting an illegal card is zero. If a player cannot follow suit, they may play any card in their hand, including a spade. Generally, each trick is won by the player who played the highest rank of the suit led. Any card played out-of-suit (that is not a spade) cannot win the trick, regardless of its rank. However, if one or more players played spades, the trick is won by the player who played the highest rank of spades. Equivalently, the trick winner is the highest-ranked spade if at least one spade was played; otherwise, the winner is the highest-ranked card of the suit led. When a trick is won, the winning player sets the trick in front of themselves so that it's easy to tell how many tricks each player has won. The player who won the trick then leads the next trick.
Scoring
At the end of the hand, each partnership is scored from its two individual bids and the number of tricks won by each individual player. All scoring is based on the individual contracts described below; a Nil or Double Nil player does not combine their tricks with their partner's tricks to satisfy the special bid.
If both players on a partnership made normal bids (1 through 13), add their bids together to determine the team's contract. If the partnership wins at least that many tricks total, each trick in the combined bid counts for 10 points, and every trick won above the combined bid is worth 1 point as a bag. If the partnership does not meet its combined bid, it scores -10 points for each trick it bid, and the tricks they did win are completely worthless. No bags are added when the team fails. Example: Beth and David bid 5 tricks and they won 7 tricks. They score 52 points (50 for the tricks bid, plus 2 for the extras, which are known as "bags").
If one player bids Nil and the other player makes a normal bid, the Nil player must win exactly 0 tricks, and the normal-bid player must win at least their own bid. If both requirements are met, the team scores 100 points for the successful Nil, plus 10 points for each trick in the normal player's bid, plus 1 point for every trick that the normal-bid player won above their bid. Those excess tricks are bags. If either requirement fails, the entire team fails and scores exactly -(100 + 10 × the normal bid), with no bags added, regardless of how many tricks either player won.
If one player bids Double Nil and the other player makes a normal bid, the Double Nil player must win exactly 0 tricks, and the normal-bid player must win at least their own bid. If both requirements are met, the team scores 200 points for the successful Double Nil, plus 10 points for each trick in the normal player's bid, plus 1 point for every trick that the normal-bid player won above their bid. Those excess tricks are bags. If either requirement fails, the entire team fails and scores exactly -(200 + 10 × the normal bid), with no bags added, regardless of how many tricks either player won.
If both partners bid Nil, both must win exactly 0 tricks for the team to succeed. If both succeed, the partnership earns 200 points. If either or both fail, the partnership receives a 200-point penalty and no bags are added. If both partners bid Double Nil, both must win exactly 0 tricks for the team to succeed. If both succeed, the partnership earns 400 points. If either or both fail, the partnership receives a 400-point penalty and no bags are added. If one partner bids Nil and the other bids Double Nil, both must win exactly 0 tricks for the team to succeed. If both succeed, the partnership earns a 300-point bonus. If either or both fail, the partnership receives a 300-point penalty and no bags are added.
If a Nil or Double Nil is successful and the normal-bid partner takes more tricks than they bid, those extra tricks count as normal bags (+1 point each, adding to the team's cumulative bag count). Once any Nil or Double Nil on a team fails, the entire team fails, meaning the normal partner's bid is automatically lost and no bags are added to the partnership's count. After they fail, they still lose the same fixed amount of points regardless of how many tricks they have after that point.
Sandbagging
A partnership should avoid winning too many tricks above its bid. Each time a partnership wins 10 bags (cumulative through a game), that partnership receives a 100-point penalty. The point values of the bags are not erased entirely; they still count toward the score (+1 point each), but the 100-point penalty is applied. Any bags beyond each group of 10 carry over toward the next penalty. Example: Alex and Charlie bid 4 tricks and win 7, then they bid 3 and win 6, then they bid 4 and win 9. They now have 11 bags (3+3+5) and receive a 100-point penalty. The point value of all 11 bags is still added to the score, so the net hand-and-bag effect includes +11 points and a -100-point bag penalty. The 1 additional bag carries over. If Alex and Charlie win 9 more bags, they receive another penalty.
Continuing Play
At the end of every hand, the player to the dealer's left becomes the new dealer, including during tie-breaker hands. For each new hand, a fresh standard 52-card deck is shuffled and dealt, and all hand-specific bids, hands, trick counts, and played-card information are cleared.
Winning
The first partnership to reach 500 points is the winner. At the end of a hand, all point calculations and cumulative bag penalty deductions must be applied before determining if a team has reached 500 points. There is no minimum negative score that ends the game; if a team falls into negative numbers, they must simply keep playing until someone wins. If only one partnership has a score of 500 points or more after all scoring and bag penalties for the hand have been applied, that partnership wins. If both partnerships have 500 points or more after the same hand and their final scores are different, the partnership with the higher final score wins. If both partnerships have exactly the same final score, play one additional hand. During a tie-breaker hand, the 500-point target is ignored: at the end of that hand, the partnership with the higher final score wins, even if both scores are below 500 points. If the scores are still tied, continue with another tie-breaker hand. The dealer continues to rotate normally during tie-breakers, and all accumulated bags and scores carry forward. If the teams are still tied after 10 additional tie-breaker hands, stop and declare a tie.
Table Talk
Table talk involving strategy is not permitted once a hand has been dealt. There is no penalty for violating this rule, as it is expected that table talk should simply not happen.

</details>